# Mounting

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import geopandas as gpd
from shapely.geometry import Polygon
from shapely.geometry import Point
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.cm as cm
from sklearn import preprocessing
from sklearn.cluster import Birch
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from scipy.special import rel_entr, kl_div
from scipy.stats import entropy
import math
import time
import random
from sklearn.metrics import r2_score

import warnings
warnings.filterwarnings('ignore')

path_seoul_data = '/content/drive/MyDrive/Colab Notebooks/datasets/Seoul/'
path_lta_data = '/content/drive/MyDrive/Colab Notebooks/data_fusion_fcl/lta_data/'

path_data = '/content/drive/MyDrive/Colab Notebooks/Workspace/tour_generation/data_tour/'
path_figure = '/content/drive/MyDrive/Colab Notebooks/Workspace/tour_generation/figure_tour/'
path_result = '/content/drive/MyDrive/Colab Notebooks/Workspace/tour_generation/result_tour/'

def reduce_mem_usage(props):
    start_mem_usg = props.memory_usage().sum() / 1024**2
    #print("Memory usage of properties dataframe is :",start_mem_usg," MB")
    NAlist = [] # Keeps track of columns that have missing values filled in.
    for col in props.columns:
        if (props[col].dtype != object) & (props[col].dtype != 'category'):  # Exclude strings

            # make variables for Int, max and min
            IsInt = False
            mx = props[col].max()
            mn = props[col].min()

            # Integer does not support NA, therefore, NA needs to be filled
            if not np.isfinite(props[col]).all():
                NAlist.append(col)
                props[col].fillna(mn-1,inplace=True)

            # test if column can be converted to an integer
            asint = props[col].fillna(0).astype(np.int64)
            result = (props[col] - asint)
            result = result.sum()
            if result > -0.01 and result < 0.01:
                IsInt = True

            if IsInt:
                if mn >= 0:
                    if mx < 255:
                        props[col] = props[col].astype(np.uint8)
                    elif mx < 65535:
                        props[col] = props[col].astype(np.uint16)
                    elif mx < 4294967295:
                        props[col] = props[col].astype(np.uint32)
                    else:
                        props[col] = props[col].astype(np.uint64)
                else:
                    if mn > np.iinfo(np.int8).min and mx < np.iinfo(np.int8).max:
                        props[col] = props[col].astype(np.int8)
                    elif mn > np.iinfo(np.int16).min and mx < np.iinfo(np.int16).max:
                        props[col] = props[col].astype(np.int16)
                    elif mn > np.iinfo(np.int32).min and mx < np.iinfo(np.int32).max:
                        props[col] = props[col].astype(np.int32)
                    elif mn > np.iinfo(np.int64).min and mx < np.iinfo(np.int64).max:
                        props[col] = props[col].astype(np.int64)
            else:
                props[col] = props[col].astype(np.float32)

    mem_usg = props.memory_usage().sum() / 1024**2
    return props

Mounted at /content/drive


# ***

# Validation: Sampling

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

def rename_columns(df, i):
    df.rename(columns = {'TRIP_PURPOSE':'TRIP_PURPOSE_' + str(i)}, inplace = True)
    df.rename(columns = {'TRAVEL_MODE':'TRAVEL_MODE_' + str(i)}, inplace = True)
    df.rename(columns = {'ORIGIN_DISTANCE':'ORIGIN_DISTANCE_' + str(i)}, inplace = True)
    df.rename(columns = {'DESTINATION_DISTANCE':'DESTINATION_DISTANCE_' + str(i)}, inplace = True)
    df.rename(columns = {'ORIGIN_SUBZONE':'ORIGIN_SUBZONE_' + str(i)}, inplace = True)
    df.rename(columns = {'DESTINATION_SUBZONE':'DESTINATION_SUBZONE_' + str(i)}, inplace = True)
    df.rename(columns = {'TRIP_STARTTIME':'TRIP_STARTTIME_' + str(i)}, inplace = True)
    df.rename(columns = {'TRIP_ENDTIME':'TRIP_ENDTIME_' + str(i)}, inplace = True)
    df.rename(columns = {'ORIGIN_SUBZONE_X':'ORIGIN_SUBZONE_X_' + str(i)}, inplace = True)
    df.rename(columns = {'ORIGIN_SUBZONE_Y':'ORIGIN_SUBZONE_Y_' + str(i)}, inplace = True)
    df.rename(columns = {'DESTINATION_SUBZONE_X':'DESTINATION_SUBZONE_X_' + str(i)}, inplace = True)
    df.rename(columns = {'DESTINATION_SUBZONE_Y':'DESTINATION_SUBZONE_Y_' + str(i)}, inplace = True)

def generate_tour_hts(df_hts):
    df_final = pd.DataFrame()
    df_hts = df_hts[att]
    for z in range(2, TRIP_MAX_ + 1):
        df_hts_ = df_hts[(df_hts['TRIP_MAX'] == z)]
        df_result = df_hts_[df_hts_['TRIP_CNT'] == 1]
        df_result = df_result.drop(columns=['TRIP_CNT'])
        rename_columns(df_result, 1)
        for i in range(2, z+1):
            df_hts_i = df_hts_[df_hts_['TRIP_CNT'] == i]
            df_hts_i = df_hts_i.drop(columns=['TRIP_CNT'])
            rename_columns(df_hts_i, i)
            df_result = pd.merge(df_result, df_hts_i, on=['ID','AGE','GENDER', 'INCOME', 'TRIP_MAX'])
            df_result = df_result[df_result['TRIP_STARTTIME_' + str(i)] >= df_result['TRIP_ENDTIME_' + str(i-1)]]
        if z == 2:
            df_final = df_result
        else:
            df_final = pd.concat([df_final, df_result])
    return df_final

def distance(df, att):
    lon1 = np.radians(df[att[0]])
    lon2 = np.radians(df[att[1]])
    lat1 = np.radians(df[att[2]])
    lat2 = np.radians(df[att[3]])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    return 6371*2 * np.arcsin(np.sqrt(a))

def find_non_closed_tours(df_long: pd.DataFrame) -> pd.DataFrame:
    """
    Return IDs where origin of the first trip != destination of the last trip.

    Assumes df_long contains:
      ID, TRIP_COUNT, ORIGIN_SUBZONE, DESTINATION_SUBZONE
    """
    required = {"ID", "TRIP_CNT", "ORIGIN_SUBZONE", "DESTINATION_SUBZONE"}
    missing = required - set(df_long.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    # Get first trip origin and last trip destination per ID
    g = df_long.sort_values(["ID", "TRIP_CNT"], kind="mergesort").groupby("ID", sort=False)

    first_origin = g["ORIGIN_SUBZONE"].first().rename("FIRST_ORIGIN")
    last_dest = g["DESTINATION_SUBZONE"].last().rename("LAST_DEST")

    out = pd.concat([first_origin, last_dest], axis=1).reset_index()

    # Flag non-closed tours
    out["IS_CLOSED"] = out["FIRST_ORIGIN"] == out["LAST_DEST"]
    non_closed = out.loc[~out["IS_CLOSED"]].copy()

    return non_closed



#####################################################################

att = ['ID', 'AGE', 'GENDER', 'INCOME', 'TRIP_CNT', 'TRIP_MAX', 'TRIP_PURPOSE', 'TRAVEL_MODE', 'TRIP_STARTTIME', 'TRIP_ENDTIME',
        'ORIGIN_SUBZONE', 'ORIGIN_SUBZONE_X', 'ORIGIN_SUBZONE_Y', 'DESTINATION_SUBZONE', 'DESTINATION_SUBZONE_X', 'DESTINATION_SUBZONE_Y']

TRIP_MAX_ = 7

df_hts = pd.read_csv(path_data + 'data_sgp_hts_trip.csv')
non_closed_ids = find_non_closed_tours(df_hts)
non_closed_ids = non_closed_ids['ID'].drop_duplicates()
df_hts = df_hts[~df_hts['ID'].isin(non_closed_ids)]
df_hts_tour = generate_tour_hts(df_hts)


df_hts.to_csv(path_data + 'val_data_true_trip.csv', index = False)
df_hts_tour.to_csv(path_data + 'val_data_true_tour.csv', index = False)

df_id = df_hts[['ID']].drop_duplicates()

ff = 0.5
df_hts_ = df_id.sample(frac = ff)
df_hts__ = pd.merge(df_hts_, df_hts)
df_hts_tour__ = generate_tour_hts(df_hts__)
df_hts__.to_csv(path_data + f'val_data_hts_trip_{ff}.csv', index = False)
df_hts_tour__.to_csv(path_data + f'val_data_hts_tour_{ff}.csv', index = False)

print(df_hts__.info())

df_pcm = df_hts[['TRIP_STARTTIME', 'TRIP_ENDTIME', 'ORIGIN_SUBZONE', 'DESTINATION_SUBZONE',
                 'ORIGIN_SUBZONE_X', 'ORIGIN_SUBZONE_Y', 'DESTINATION_SUBZONE_X', 'DESTINATION_SUBZONE_Y']]

df_pcm = reduce_mem_usage(df_pcm)
#df_pcm.to_csv(path_data + 'val_data_pcm_trip.csv', index = False)

print('Done!')

NameError: name 'path_data' is not defined

#============================

# Validation: Stage 1 Model Embeding

In [ ]:
# ============================================================
# Discrete-latent encoder-only fusion (non-param decoder)
#   + Embedding encoder (Φ = [D,S,E,C] integer indices)
#   + Sparse-per-Φ support for q(H|Φ)
# ============================================================

import os, math, json, re, random
from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional, Callable

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset

from sklearn.cluster import KMeans

# ------------------------------------------------------------
# 0) Utilities: feature builders & small helpers
# ------------------------------------------------------------

def reduce_mem_usage(df: pd.DataFrame) -> pd.DataFrame:
    """Downcast numerics to reduce RAM usage (simple & safe)."""
    start_mem = df.memory_usage().sum() / 1024**2
    for col in df.columns:
        col_type = df[col].dtype
        if pd.api.types.is_numeric_dtype(col_type):
            c_min = df[col].min()
            c_max = df[col].max()
            if pd.api.types.is_float_dtype(col_type):
                df[col] = pd.to_numeric(df[col], downcast='float')
            else:
                df[col] = pd.to_numeric(df[col], downcast='integer')
    end_mem = df.memory_usage().sum() / 1024**2
    # print(f"[reduce_mem] {start_mem:.2f} -> {end_mem:.2f} MB")
    return df

def distance(df, att):
    """Great-circle distance (km). att = [lon_o, lon_d, lat_o, lat_d] (deg)."""
    lon1 = np.radians(df[att[0]]); lon2 = np.radians(df[att[1]])
    lat1 = np.radians(df[att[2]]); lat2 = np.radians(df[att[3]])
    dlon = lon2 - lon1; dlat = lat2 - lat1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 6371*2*np.arcsin(np.sqrt(a))


def land_use(df_hts_, df_pcm_):
    """DESTINATION_SUBZONE -> LU_0..LU_5. Fills missing PCM zones with HTS global averages."""
    # 1. Calculate probabilities from HTS
    df = df_hts_[['DESTINATION_SUBZONE','TRIP_PURPOSE']].copy()
    df['Numbers'] = 1.0
    df = df.groupby(['DESTINATION_SUBZONE','TRIP_PURPOSE'])['Numbers'].sum().reset_index()

    # Normalize per zone
    df['Zone_Total'] = df.groupby('DESTINATION_SUBZONE')['Numbers'].transform('sum')
    df['Prob'] = df['Numbers'] / df['Zone_Total']

    # Pivot to LU_0...LU_5
    df_lu = df.pivot(index='DESTINATION_SUBZONE', columns='TRIP_PURPOSE', values='Prob').reset_index()
    df_lu.columns.name = None
    df_lu = df_lu.rename(columns={i: f'LU_{i}' for i in range(6)})
    df_lu = df_lu.fillna(0.0)

    # 2. Handle missing zones using PCM universe
    pcm_zones = pd.DataFrame({'DESTINATION_SUBZONE': df_pcm_['DESTINATION_SUBZONE'].unique()})
    df_final = pd.merge(pcm_zones, df_lu, on='DESTINATION_SUBZONE', how='left')

    # 3. Fill NaNs with the average of existing zones
    lu_cols = [f'LU_{i}' for i in range(6)]
    avg_values = df_lu[lu_cols].mean()
    df_final[lu_cols] = df_final[lu_cols].fillna(avg_values)

    return df_final

def mode_share(df_hts_, df_pcm_):
    """DESTINATION_SUBZONE -> TRANSIT_RATIO. Fills missing PCM zones with global HTS mean."""
    # 1. Calculate transit ratio per zone from HTS
    df = df_hts_[['DESTINATION_SUBZONE','TRAVEL_MODE']].copy()
    df['is_transit'] = (df['TRAVEL_MODE'] == 1).astype(float)

    # Group by zone and get mean (which is the probability P(mode=1|zone))
    df_ms = df.groupby('DESTINATION_SUBZONE')['is_transit'].mean().reset_index()
    df_ms.rename(columns={'is_transit': 'TRANSIT_RATIO'}, inplace=True)

    # 2. Handle missing zones using PCM universe
    pcm_zones = pd.DataFrame({'DESTINATION_SUBZONE': df_pcm_['DESTINATION_SUBZONE'].unique()})
    df_final = pd.merge(pcm_zones, df_ms, on='DESTINATION_SUBZONE', how='left')

    # 3. Fill NaNs with the global average transit ratio from known zones
    global_avg = df_ms['TRANSIT_RATIO'].mean()
    df_final['TRANSIT_RATIO'] = df_final['TRANSIT_RATIO'].fillna(global_avg)

    return df_final

# ------------------------------------------------------------
# 0.1) Discrete-Φ helpers (bins + clusters → indices)
# ------------------------------------------------------------

@dataclass
class DiscreteSpec:
    n_dist: int = 6
    n_start: int = 8
    n_end: int = 8
    n_lu_mode_clusters: int = 16
    dist_binning: str = "quantile"  # or "uniform"
    time_binning: str = "quantile"  # or "uniform"

def _fit_bins(x: np.ndarray, n: int, kind: str) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    if kind == "uniform":
        lo, hi = np.nanmin(x), np.nanmax(x)
        edges = np.linspace(lo, hi, n + 1)
    else:
        qs = np.linspace(0, 100, n + 1)
        edges = np.percentile(x, qs)
        edges[0]  = min(edges[0],  x.min()) - 1e-9
        edges[-1] = max(edges[-1], x.max()) + 1e-9
    return edges

def _digitize(x: np.ndarray, edges: np.ndarray) -> np.ndarray:
    return np.clip(np.digitize(x, edges, right=False) - 1, 0, len(edges) - 2)

def fit_semantic_spec_on_hts(
    df_hts: pd.DataFrame,
    spec: DiscreteSpec,
    distance_col="TRIP_DISTANCE",
    start_col="TRIP_STARTTIME",
    end_col="TRIP_ENDTIME",
    lu_cols=("LU_0","LU_1","LU_2","LU_3","LU_4","LU_5"),
    tr_col="TRANSIT_RATIO",
    km_random_state=42
) -> Dict:
    dist_edges  = _fit_bins(df_hts[distance_col].values, spec.n_dist,  spec.dist_binning)
    start_edges = _fit_bins(df_hts[start_col].values,   spec.n_start, spec.time_binning)
    end_edges   = _fit_bins(df_hts[end_col].values,     spec.n_end,   spec.time_binning)

    feat_cols = list(lu_cols) + [tr_col]
    X = df_hts[feat_cols].to_numpy(dtype=float)

    eps = 1e-8
    mu = np.nanmean(X, axis=0)
    sd = np.nanstd(X, axis=0)
    sd = np.where(sd < eps, 1.0, sd)
    Xz = (X - mu) / sd

    kmeans = KMeans(n_clusters=spec.n_lu_mode_clusters, n_init="auto", random_state=km_random_state)
    kmeans.fit(Xz)

    return {
        "dist_edges": dist_edges.tolist(),
        "start_edges": start_edges.tolist(),
        "end_edges": end_edges.tolist(),
        "lu_cols": list(lu_cols),
        "tr_col": tr_col,
        "lu_norm": {"mean": mu.tolist(), "std": sd.tolist(), "eps": eps, "feat_order": feat_cols},
        "kmeans_centers": kmeans.cluster_centers_.tolist(),
        "phi_segments": {
            "n_dist": spec.n_dist,
            "n_start": spec.n_start,
            "n_end": spec.n_end,
            "n_cluster": spec.n_lu_mode_clusters
        }
    }

def apply_semantic_spec(
    df: pd.DataFrame,
    fitted: Dict,
    distance_col="TRIP_DISTANCE",
    start_col="TRIP_STARTTIME",
    end_col="TRIP_ENDTIME"
) -> pd.DataFrame:
    df = df.copy()
    dist_edges  = np.array(fitted["dist_edges"], dtype=float)
    start_edges = np.array(fitted["start_edges"], dtype=float)
    end_edges   = np.array(fitted["end_edges"], dtype=float)

    d_bin = _digitize(df[distance_col].to_numpy(dtype=float), dist_edges)
    s_bin = _digitize(df[start_col].to_numpy(dtype=float),    start_edges)
    e_bin = _digitize(df[end_col].to_numpy(dtype=float),      end_edges)

    lu_meta = fitted["lu_norm"]
    feat_cols = lu_meta["feat_order"]
    mu = np.array(lu_meta["mean"], dtype=float)
    sd = np.array(lu_meta["std"], dtype=float)
    eps = float(lu_meta.get("eps", 1e-8))

    X = df[feat_cols].to_numpy(dtype=float)
    sd_safe = np.where(sd < eps, 1.0, sd)
    Xz = (X - mu) / sd_safe

    centers = np.array(fitted["kmeans_centers"], dtype=float)
    d2 = ((Xz[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2)
    c_id = d2.argmin(axis=1)

    df["D_BIN"] = d_bin.astype(int)
    df["S_BIN"] = s_bin.astype(int)
    df["E_BIN"] = e_bin.astype(int)
    df["C_ID"]  = c_id.astype(int)
    return df

def build_phi_indices_from_bins(
    df: pd.DataFrame,
    cols=("D_BIN","S_BIN","E_BIN","C_ID")
) -> Tuple[pd.DataFrame, int]:
    """Store Φ as integer indices [D,S,E,C]."""
    df = df.copy()
    D = df[cols[0]].astype(int).to_numpy()
    S = df[cols[1]].astype(int).to_numpy()
    E = df[cols[2]].astype(int).to_numpy()
    C = df[cols[3]].astype(int).to_numpy()
    df["PHI_IDX"] = [[int(D[i]), int(S[i]), int(E[i]), int(C[i])] for i in range(len(df))]
    return df, 4  # four categorical fields

# ------------------------------------------------------------
# 1) Data containers (index-based Φ)
# ------------------------------------------------------------

class HTSDataset(Dataset):
    """One HTS row = (phi_idx[4], xy_id)."""
    def __init__(self, df_hts: pd.DataFrame, xy_ids: np.ndarray):
        self.phi_idx = np.vstack(df_hts["PHI_IDX"].to_numpy()).astype(np.int64)  # [N,4]
        self.xy = xy_ids.astype(np.int64)
    def __len__(self): return self.phi_idx.shape[0]
    def __getitem__(self, i):
        return self.phi_idx[i], self.xy[i]

class PCMDataset(Dataset):
    """One PCM row = (phi_idx[4], count)."""
    def __init__(self, df_pcm: pd.DataFrame):
        self.phi_idx = np.vstack(df_pcm["PHI_IDX"].to_numpy()).astype(np.int64)  # [N,4]
        self.count = df_pcm["COUNT"].to_numpy(dtype=np.float32)
    def __len__(self): return self.phi_idx.shape[0]
    def __getitem__(self, i):
        return self.phi_idx[i], self.count[i]

# ------------------------------------------------------------
# 2) Embedding Encoder (Φ indices -> embeddings -> MLP -> logits[K])
# ------------------------------------------------------------

class EncoderEmbed(nn.Module):
    """
    Embedding encoder: [D,S,E,C] indices -> concat(emb_D, emb_S, emb_E, emb_C) -> MLP -> logits[K].
    """
    def __init__(
        self,
        cardinals: Tuple[int, int, int, int],  # (n_dist, n_start, n_end, n_cluster)
        K: int,
        emb_dims: Tuple[int, int, int, int] = (16, 16, 16, 16),
        hidden: int = 256,
        num_layers: int = 2,
        dropout: float = 0.0
    ):
        super().__init__()
        nD, nS, nE, nC = map(int, cardinals)
        eD, eS, eE, eC = emb_dims

        self.emb_D = nn.Embedding(nD, eD)
        self.emb_S = nn.Embedding(nS, eS)
        self.emb_E = nn.Embedding(nE, eE)
        self.emb_C = nn.Embedding(nC, eC)

        in_dim = eD + eS + eE + eC
        layers: List[nn.Module] = []
        dims = [in_dim] + [hidden]*(num_layers-1) + [K]
        for i in range(len(dims)-2):
            layers += [nn.Linear(dims[i], dims[i+1]), nn.ReLU(inplace=True)]
            if dropout > 0:
                layers += [nn.Dropout(dropout)]
        layers += [nn.Linear(dims[-2], dims[-1])]
        self.net = nn.Sequential(*layers)

    def forward(self, phi_idx: torch.Tensor) -> torch.Tensor:
        """
        phi_idx: LongTensor [B,4] with columns [D,S,E,C].
        returns logits [B,K]
        """
        D = phi_idx[:, 0]; S = phi_idx[:, 1]; E = phi_idx[:, 2]; C = phi_idx[:, 3]
        z = torch.cat([self.emb_D(D), self.emb_S(S), self.emb_E(E), self.emb_C(C)], dim=-1)
        return self.net(z)

# ------------------------------------------------------------
# 3) Numerics, schedules, divergences, masked softmax
# ------------------------------------------------------------

def safe_log(x: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return torch.log(x.clamp_min(eps))

def entropy_categorical(probs: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    return -(probs * safe_log(probs, eps)).sum(dim=-1)

def js_divergence(p: torch.Tensor, q: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    p = p / p.sum().clamp_min(eps)
    q = q / q.sum().clamp_min(eps)
    m = 0.5 * (p + q)
    kl_pm = (p * (safe_log(p, eps) - safe_log(m, eps))).sum()
    kl_qm = (q * (safe_log(q, eps) - safe_log(m, eps))).sum()
    return 0.5 * (kl_pm + kl_qm)

def softmax_tau(logits: torch.Tensor, tau: float) -> torch.Tensor:
    return F.softmax(logits / max(tau, 1e-6), dim=-1)

def get_tau(ep: int, total_ep: int, tau_start: float, tau_end: float, schedule: str = "linear") -> float:
    if total_ep <= 1: return tau_end
    t = (ep - 1) / (total_ep - 1)  # 0..1
    if schedule == "linear":
        s = t
    elif schedule == "cosine":
        s = 0.5 * (1 - math.cos(math.pi * t))
    elif schedule == "exp":
        s = 1 - math.exp(-5 * t)
    else:
        s = t
    return (1 - s) * tau_start + s * tau_end

# ------------------------------------------------------------
# 3.1) Sparse support policies & masked softmax
# ------------------------------------------------------------

SupportPolicy = Callable[[torch.Tensor, Optional[torch.Tensor]], torch.Tensor]
# signature: mask = policy(logits, phi_idx_b) -> bool tensor [B,K], True = allowed

def masked_softmax(logits: torch.Tensor,
                   mask: torch.Tensor,
                   tau: float = 1.0,
                   eps: float = 1e-12) -> torch.Tensor:
    B, K = logits.shape
    scaled = logits / max(tau, 1e-6)
    scaled = scaled.masked_fill(~mask, -1e9)
    q = F.softmax(scaled, dim=-1)
    q = q * mask.float()
    z = q.sum(dim=-1, keepdim=True).clamp_min(eps)
    return q / z

def topk_support_policy(k: int) -> SupportPolicy:
    def policy(logits: torch.Tensor, phi_idx_b: Optional[torch.Tensor] = None) -> torch.Tensor:
        B, K = logits.shape
        k_eff = min(max(1, k), K)
        idx = torch.topk(logits, k=k_eff, dim=-1).indices
        mask = torch.zeros_like(logits, dtype=torch.bool)
        mask.scatter_(1, idx, True)
        return mask
    return policy

def prob_threshold_policy(thr: float) -> SupportPolicy:
    def policy(logits: torch.Tensor, phi_idx_b: Optional[torch.Tensor] = None) -> torch.Tensor:
        p = F.softmax(logits, dim=-1)
        return p >= thr
    return policy

# ---- Bucketed support mining (index-based) ----

from collections import defaultdict

def build_phi_bucket_candidates_embed(
    encoder: nn.Module,
    hts_loader: DataLoader,
    K: int,
    tau: float,
    topk: int = 8,
    device: Optional[str] = None
) -> Dict[Tuple[int, int, int, int], np.ndarray]:
    """Aggregate q(H|Φ) over HTS and keep top-k H per Φ bucket. Φ is [D,S,E,C] indices."""
    enc_device = next(encoder.parameters()).device if device is None else torch.device(device)
    encoder.eval()
    acc = defaultdict(lambda: np.zeros(K, dtype=np.float64))

    with torch.no_grad():
        for phi_idx_b, _ in hts_loader:
            phi_idx_b = phi_idx_b.to(enc_device)              # [B,4] long
            logits = encoder(phi_idx_b)                       # [B,K]
            q = F.softmax(logits / max(tau, 1e-6), dim=-1).cpu().numpy()
            key_rows = phi_idx_b.cpu().numpy()                # [[D,S,E,C], ...]
            for i in range(key_rows.shape[0]):
                key = tuple(int(x) for x in key_rows[i])
                acc[key] += q[i]

    out = {}
    for key, vec in acc.items():
        k_eff = min(max(1, topk), K)
        keep = np.argpartition(-vec, k_eff-1)[:k_eff]
        keep = keep[np.argsort(-vec[keep])]
        out[key] = keep
    return out

def bucket_support_policy_embed(
    candidates: Dict[Tuple[int,int,int,int], np.ndarray]
) -> SupportPolicy:
    def policy(logits: torch.Tensor, phi_idx_b: Optional[torch.Tensor]) -> torch.Tensor:
        device = logits.device
        B, K = logits.shape
        mask = torch.zeros((B, K), dtype=torch.bool, device=device)
        keys = phi_idx_b.tolist()  # list of [D,S,E,C]
        for i in range(B):
            key = tuple(int(x) for x in keys[i])
            keep = candidates.get(key)
            if keep is None or len(keep) == 0:
                j = int(torch.argmax(logits[i]))
                mask[i, j] = True
            else:
                idx = torch.as_tensor(keep, dtype=torch.long, device=device)
                mask[i, idx] = True
        return mask
    return policy

# ------------------------------------------------------------
# 4) Non-param decoder  \hat{P}(XY|H)  (masked)
# ------------------------------------------------------------

@torch.no_grad()
def build_nonparam_decoder_xy_given_h(
    encoder: nn.Module,
    hts_loader: DataLoader,
    n_xy: int,
    K: int,
    device: str = "cpu",
    dtype: torch.dtype = torch.float32,
    progress: bool = True,
    tau: float = 1.0,
    support_policy: Optional[SupportPolicy] = None
) -> torch.Tensor:
    """
    \hat P(XY=j | H=h) = E[ 1{XY=j} q(h|Phi) ] / E[ q(h|Phi) ], with q masked per Φ (indices).
    Returns CPU tensor [n_xy, K] with columns summing to 1 (over kept H).
    """
    encoder.eval()
    num = torch.zeros((n_xy, K), dtype=dtype, device="cpu")
    den = torch.zeros((K,), dtype=dtype, device="cpu")

    for bi, (phi_idx_b, xy_b) in enumerate(hts_loader):
        if progress and bi % 50 == 0:
            print(f"[decoder] pass chunk {bi}")
        phi_idx_b = phi_idx_b.to(device=device, dtype=torch.long)
        logits = encoder(phi_idx_b)
        if support_policy is None:
            q = softmax_tau(logits, tau)
        else:
            mask = support_policy(logits, phi_idx_b)
            q = masked_softmax(logits, mask, tau)
        q = q.to("cpu")
        xy_b = xy_b.to("cpu")

        num.index_add_(0, xy_b, q)
        den += q.sum(dim=0)

    P = torch.zeros_like(num)
    mask = den > 0
    if mask.any():
        P[:, mask] = num[:, mask] / den[mask]
    P = P / P.sum(dim=0, keepdim=True).clamp_min(1e-12)
    return P

# ------------------------------------------------------------
# 5) Loss components (masked q everywhere)
# ------------------------------------------------------------

def compute_hts_nll(
    encoder: nn.Module,
    hts_loader: DataLoader,
    P_xy_given_h: torch.Tensor,
    log_norm: float,
    device: str,
    dtype: torch.dtype,
    progress: bool = True,
    tau: float = 1.0,
    support_policy: Optional[SupportPolicy] = None,
):
    total_nll = torch.zeros((), device=device, dtype=dtype)
    totalN = 0
    P_xy_given_h = P_xy_given_h.to(device=device, dtype=dtype)

    for bi, (phi_idx_b, xy_b) in enumerate(hts_loader):
        if progress and bi % 50 == 0:
            print(f"[hts-nll] pass chunk {bi}")
        B = phi_idx_b.shape[0]
        phi_idx_b = phi_idx_b.to(device=device, dtype=torch.long)
        xy_b  = xy_b.to(device=device, dtype=torch.long)

        logits = encoder(phi_idx_b)
        if support_policy is None:
            q = softmax_tau(logits, tau)
        else:
            mask = support_policy(logits, phi_idx_b)
            q = masked_softmax(logits, mask, tau)

        P_rows = P_xy_given_h.index_select(0, xy_b)    # [B, K]
        mix = (q * P_rows).sum(dim=-1).clamp_min(1e-12)
        nll_sum = -torch.log(mix).sum()

        total_nll = total_nll + nll_sum
        totalN += B

    avg_nll = total_nll / max(totalN, 1)
    if log_norm > 0:
        avg_nll = avg_nll / log_norm
    return avg_nll, totalN

def compute_latent_marginals(
    encoder: nn.Module,
    loader: DataLoader,
    K: int,
    device: str,
    dtype: torch.dtype,
    progress: bool = True,
    tau: float = 1.0,
    support_policy: Optional[SupportPolicy] = None,
):
    acc = torch.zeros(K, device=device, dtype=dtype)
    wsum = torch.zeros((), device=device, dtype=dtype)

    for bi, batch in enumerate(loader):
        if progress and bi % 50 == 0:
            print(f"[latent-marg] pass chunk {bi}")
        if len(batch) == 2:
            phi_idx_b, w_b = batch
            # HTS loader provides xy_b here; PCM loader provides COUNT; we handle both
            if w_b.dtype == torch.long or w_b.dtype == torch.int64:
                w_b = None
            else:
                w_b = w_b.to(device=device, dtype=dtype)
        else:
            phi_idx_b = batch[0]
            w_b = None

        phi_idx_b = phi_idx_b.to(device=device, dtype=torch.long)
        logits = encoder(phi_idx_b)
        if support_policy is None:
            q = softmax_tau(logits, tau)
        else:
            mask = support_policy(logits, phi_idx_b)
            q = masked_softmax(logits, mask, tau)

        if w_b is None:
            acc  = acc  + q.sum(dim=0)
            wsum = wsum + torch.tensor(q.shape[0], device=device, dtype=dtype)
        else:
            acc  = acc  + (q * w_b.unsqueeze(-1)).sum(dim=0)
            wsum = wsum + w_b.sum()

    p = acc / wsum.clamp_min(1e-12)
    p = p / p.sum().clamp_min(1e-12)
    return p

def compute_fusion_js(
    encoder: nn.Module,
    hts_loader: DataLoader,
    pcm_loader: DataLoader,
    K: int,
    device: str,
    dtype: torch.dtype,
    use_normalized: bool = True,
    tau: float = 1.0,
    support_policy: Optional[SupportPolicy] = None,
):
    p_hts = compute_latent_marginals(encoder, hts_loader, K, device, dtype, progress=False, tau=tau, support_policy=support_policy)
    p_pcm = compute_latent_marginals(encoder, pcm_loader, K, device, dtype, progress=False, tau=tau, support_policy=support_policy)
    m = 0.5 * (p_hts + p_pcm)
    js = 0.5 * (
        (p_hts * (torch.log(p_hts.clamp_min(1e-12)) - torch.log(m.clamp_min(1e-12)))).sum()
      + (p_pcm * (torch.log(p_pcm.clamp_min(1e-12)) - torch.log(m.clamp_min(1e-12)))).sum()
    )
    return js / math.log(2.0) if use_normalized else js

# ------------------------------------------------------------
# 6) Config
# ------------------------------------------------------------

@dataclass
class TrainConfig:
    K: int = 64
    enc_hidden: int = 256
    enc_layers: int = 2
    enc_dropout: float = 0.0
    epochs: int = 10
    batch_size_hts: int = 8192
    batch_size_pcm: int = 16384
    lr: float = 2e-3
    wd: float = 1e-4
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    use_normalized_objective: bool = True
    lambda_fus: float = 1.0
    dtype: torch.dtype = torch.float32
    rebuild_every: int = 1
    progress: bool = True

    # softmax temperature annealing
    tau_start: float = 1.0
    tau_end:   float = 0.3
    tau_schedule: str = "cosine"

    # support policy controls
    topk_h: int = 8                   # per-Φ top-k for q(H|Φ)
    use_bucket_candidates: bool = False
    bucket_topk: int = 8
    refresh_bucket_every: int = 2     # epochs; 0 disables refresh

# ------------------------------------------------------------
# 7) Training loops (masked)
# ------------------------------------------------------------

def _prepare_xy_and_loaders(df_hts: pd.DataFrame,
                            df_pcm: pd.DataFrame,
                            cfg: TrainConfig):
    att_xy_series = (df_hts["ATT_X"].astype(str) + "|" + df_hts["ATT_Y"].astype(str))
    xy_vals = att_xy_series.to_numpy()
    xy_unique = pd.unique(xy_vals)
    xy_map = {k: i for i, k in enumerate(xy_unique)}
    xy_ids = np.vectorize(xy_map.__getitem__)(xy_vals)
    xy_classes = list(xy_unique)
    n_xy = len(xy_classes)

    hts_ds = HTSDataset(df_hts, xy_ids)
    pcm_ds = PCMDataset(df_pcm)
    hts_loader_full = DataLoader(hts_ds, batch_size=cfg.batch_size_hts, shuffle=False, num_workers=0, pin_memory=False)
    pcm_loader_full = DataLoader(pcm_ds, batch_size=cfg.batch_size_pcm, shuffle=False, num_workers=0, pin_memory=False)
    phi_dim = 4  # indices length, not used by embedding encoder
    return n_xy, xy_map, xy_classes, hts_loader_full, pcm_loader_full, phi_dim

def _make_support_policy(enc: nn.Module,
                         hts_loader_full: DataLoader,
                         tau: float,
                         cfg: TrainConfig) -> SupportPolicy:
    if cfg.use_bucket_candidates:
        cands = build_phi_bucket_candidates_embed(
            enc, hts_loader_full, cfg.K, tau,
            topk=cfg.bucket_topk, device=cfg.device
        )
        return bucket_support_policy_embed(cands)
    else:
        return topk_support_policy(cfg.topk_h)

def train_encoder_only_fusion(
    df_hts: pd.DataFrame,
    df_pcm: pd.DataFrame,                 # kept for signature parity
    cfg: TrainConfig,
    phi_segments_meta: Dict[str,int]      # pass fitted["phi_segments"]
) -> Dict:
    device = cfg.device
    dtype = cfg.dtype

    n_xy, xy_map, xy_classes, hts_loader_full, _pcm_loader_full, phi_dim = _prepare_xy_and_loaders(df_hts, df_pcm, cfg)

    # Embedding encoder
    cardinals = (phi_segments_meta["n_dist"], phi_segments_meta["n_start"],
                 phi_segments_meta["n_end"],  phi_segments_meta["n_cluster"])
    enc = EncoderEmbed(
        cardinals=cardinals, K=cfg.K,
        emb_dims=(16,16,16,16),
        hidden=cfg.enc_hidden, num_layers=cfg.enc_layers, dropout=cfg.enc_dropout
    ).to(device=device, dtype=torch.float32)
    opt = torch.optim.AdamW(enc.parameters(), lr=cfg.lr, weight_decay=cfg.wd)

    logXY = math.log(max(n_xy, 2))

    # Initial tau and support policy
    tau = get_tau(1, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
    support_policy = _make_support_policy(enc, hts_loader_full, tau, cfg)

    # Initial non-param decoder
    P_xy_h = build_nonparam_decoder_xy_given_h(
        encoder=enc, hts_loader=hts_loader_full, n_xy=n_xy, K=cfg.K,
        device=device, dtype=dtype, progress=cfg.progress, tau=tau,
        support_policy=support_policy
    )  # CPU [n_xy, K]

    history = []
    for ep in range(1, cfg.epochs + 1):
        tau = get_tau(ep, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
        enc.train(); opt.zero_grad()

        # Optionally refresh bucket candidates (EM-style)
        if cfg.use_bucket_candidates and cfg.refresh_bucket_every > 0 and (ep == 1 or (ep % cfg.refresh_bucket_every) == 0):
            support_policy = _make_support_policy(enc, hts_loader_full, tau, cfg)

        # (A) HTS NLL only
        L_hts, _ = compute_hts_nll(
            encoder=enc, hts_loader=hts_loader_full,
            P_xy_given_h=P_xy_h,
            log_norm=(logXY if cfg.use_normalized_objective else 1.0),
            device=device, dtype=dtype, progress=cfg.progress, tau=tau,
            support_policy=support_policy
        )

        total = L_hts
        total.backward()
        opt.step()

        # (B) Rebuild decoder periodically with current tau + support
        if (ep % cfg.rebuild_every) == 0:
            P_xy_h = build_nonparam_decoder_xy_given_h(
                encoder=enc, hts_loader=hts_loader_full, n_xy=n_xy, K=cfg.K,
                device=device, dtype=dtype, progress=cfg.progress, tau=tau,
                support_policy=support_policy
            )

        metrics = {
            "epoch": ep,
            "tau": float(tau),
            "L_hts": float(L_hts.detach().cpu().item()),
            "J_total": float(total.detach().cpu().item()),
        }
        history.append(metrics)
        print(f"[epoch {ep:03d}] tau={tau:.4f}  L_hts={metrics['L_hts']:.6f}  total={metrics['J_total']:.6f}")

    # ===== Final evaluation with final tau + fresh support =====
    tau_final = get_tau(cfg.epochs, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
    support_policy = _make_support_policy(enc, hts_loader_full, tau_final, cfg)

    P_xy_h_final = build_nonparam_decoder_xy_given_h(
        encoder=enc, hts_loader=hts_loader_full, n_xy=n_xy, K=cfg.K,
        device=device, dtype=dtype, progress=cfg.progress, tau=tau_final,
        support_policy=support_policy
    )

    L_hts_final, _ = compute_hts_nll(
        encoder=enc, hts_loader=hts_loader_full, P_xy_given_h=P_xy_h_final,
        log_norm=(logXY if cfg.use_normalized_objective else 1.0),
        device=device, dtype=dtype, progress=False, tau=tau_final,
        support_policy=support_policy
    )

    J_final = float(L_hts_final.detach().cpu().item())
    print(f"[FINAL] K={cfg.K}  tau={tau_final:.4f}  L_hts={J_final:.6f}")

    pack = {
        "encoder_state_dict": enc.state_dict(),
        "encoder_cfg": {
            "cardinals": list(cardinals), "K": cfg.K, "hidden": cfg.enc_hidden,
            "layers": cfg.enc_layers, "dropout": cfg.enc_dropout, "dtype": str(dtype)
        },
        "xy_map": xy_map,
        "xy_classes": xy_classes,
        "P_xy_given_h": P_xy_h_final,         # CPU tensor [n_xy, K]
        "history": history,
        "config": cfg.__dict__,
        "tau_final": float(tau_final),
        "J_final": J_final,
        "L_hts_final": J_final,
        "phi_segments": dict(phi_segments_meta),
    }
    return {"model": enc, "pack": pack}

def train_encoder_only_fusion_finding_K(
    df_hts: pd.DataFrame,
    df_pcm: pd.DataFrame,
    cfg: TrainConfig,
    phi_segments_meta: Dict[str,int]
) -> Dict:
    device = cfg.device
    dtype = cfg.dtype

    # Prepare data/loaders
    n_xy, xy_map, xy_classes, hts_loader_full, pcm_loader_full, phi_dim = _prepare_xy_and_loaders(df_hts, df_pcm, cfg)

    # Build embedding encoder
    cardinals = (phi_segments_meta["n_dist"], phi_segments_meta["n_start"],
                 phi_segments_meta["n_end"],  phi_segments_meta["n_cluster"])
    enc = EncoderEmbed(
        cardinals=cardinals, K=cfg.K,
        emb_dims=(16,16,16,16),
        hidden=cfg.enc_hidden, num_layers=cfg.enc_layers, dropout=cfg.enc_dropout
    ).to(device=device, dtype=torch.float32)
    opt = torch.optim.AdamW(enc.parameters(), lr=cfg.lr, weight_decay=cfg.wd)

    logXY = math.log(max(n_xy, 2))

    # Init temperature + support policy
    tau = get_tau(1, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
    support_policy = _make_support_policy(enc, hts_loader_full, tau, cfg)

    # Initial non-parametric decoder P(XY|H)
    P_xy_h = build_nonparam_decoder_xy_given_h(
        encoder=enc, hts_loader=hts_loader_full, n_xy=n_xy, K=cfg.K,
        device=device, dtype=dtype, progress=cfg.progress, tau=tau,
        support_policy=support_policy
    )

    history = []
    for ep in range(1, cfg.epochs + 1):
        tau = get_tau(ep, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
        enc.train(); opt.zero_grad()

        # Optional refresh of Φ-bucket candidates
        if cfg.use_bucket_candidates and cfg.refresh_bucket_every > 0 and (ep == 1 or (ep % cfg.refresh_bucket_every) == 0):
            support_policy = _make_support_policy(enc, hts_loader_full, tau, cfg)

        # ---- Likelihood-only objective ----
        L_hts, _ = compute_hts_nll(
            encoder=enc, hts_loader=hts_loader_full,
            P_xy_given_h=P_xy_h,
            log_norm=(logXY if cfg.use_normalized_objective else 1.0),
            device=device, dtype=dtype, progress=cfg.progress, tau=tau,
            support_policy=support_policy
        )

        total = L_hts  # <-- only likelihood drives training
        total.backward()
        opt.step()

        # Periodically rebuild decoder with current encoder + tau + support
        if (ep % cfg.rebuild_every) == 0:
            P_xy_h = build_nonparam_decoder_xy_given_h(
                encoder=enc, hts_loader=hts_loader_full, n_xy=n_xy, K=cfg.K,
                device=device, dtype=dtype, progress=cfg.progress, tau=tau,
                support_policy=support_policy
            )

        metrics = {
            "epoch": ep,
            "tau": float(tau),
            "L_hts": float(L_hts.detach().cpu().item()),
            "J_total": float(total.detach().cpu().item()),  # equals L_hts
        }
        history.append(metrics)
        print(f"[epoch {ep:03d}] tau={tau:.4f}  L_hts={metrics['L_hts']:.6f}")

    # ===== Final evaluation =====
    tau_final = get_tau(cfg.epochs, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
    support_policy = _make_support_policy(enc, hts_loader_full, tau_final, cfg)

    # Final decoder
    P_xy_h_final = build_nonparam_decoder_xy_given_h(
        encoder=enc, hts_loader=hts_loader_full, n_xy=n_xy, K=cfg.K,
        device=device, dtype=dtype, progress=cfg.progress, tau=tau_final,
        support_policy=support_policy
    )

    # Final likelihood (objective)
    L_hts_final, _ = compute_hts_nll(
        encoder=enc, hts_loader=hts_loader_full, P_xy_given_h=P_xy_h_final,
        log_norm=(math.log(max(n_xy,2)) if cfg.use_normalized_objective else 1.0),
        device=device, dtype=dtype, progress=False, tau=tau_final,
        support_policy=support_policy
    )

    # Fusion consistency (post-hoc) = JS(p_HTS(H), p_PCM(H))
    L_fus_final = compute_fusion_js(
        encoder=enc, hts_loader=hts_loader_full, pcm_loader=pcm_loader_full,
        K=cfg.K, device=device, dtype=dtype,
        use_normalized=cfg.use_normalized_objective, tau=tau_final,
        support_policy=support_policy
    )

    # Print requested summaries
    print(f"[FINAL] K={cfg.K}  tau={tau_final:.4f}")
    print(f"        Final likelihood (HTS NLL): {float(L_hts_final):.6f}")
    print(f"        Fusion consistency (JS):    {float(L_fus_final):.6f}")

    pack = {
        "encoder_state_dict": enc.state_dict(),
        "encoder_cfg": {"cardinals": list(cardinals), "K": cfg.K, "hidden": cfg.enc_hidden,
                        "layers": cfg.enc_layers, "dropout": cfg.enc_dropout, "dtype": str(dtype)},
        "xy_map": xy_map,
        "xy_classes": xy_classes,
        "P_xy_given_h": P_xy_h_final,
        "history": history,
        "config": cfg.__dict__,
        "tau_final": float(tau_final),
        # store both metrics explicitly
        "L_hts_final": float(L_hts_final.detach().cpu().item()),
        "fusion_consistency_js": float(L_fus_final.detach().cpu().item()),
        "phi_segments": dict(phi_segments_meta),
    }
    return {"model": enc, "pack": pack}


# ------------------------------------------------------------
# 8) Decoder table -> DataFrame
# ------------------------------------------------------------

def pack_to_dataframe(pack: dict) -> pd.DataFrame:
    P_xy_h = pack["P_xy_given_h"]         # [n_xy, K] CPU torch.Tensor
    xy_classes = pack["xy_classes"]       # list of "ATT_X|ATT_Y"
    K = int(pack["encoder_cfg"]["K"])

    P = P_xy_h.detach().cpu().numpy()     # (n_xy, K)
    df = pd.DataFrame({"ATT_XY": xy_classes})
    h_cols = [f"H_{h}" for h in range(K)]
    df_probs = pd.DataFrame(P, columns=h_cols)
    df = pd.concat([df, df_probs], axis=1)
    df = df.melt(id_vars=["ATT_XY"], value_vars=h_cols,
                 var_name="H", value_name="PROB")
    df["H"] = df["H"].apply(lambda s: int(re.sub(r"^H_", "", s)))

    def split_att(x):
        parts = str(x).split("|", 1)
        if len(parts) == 2: return parts[0], parts[1]
        else: return parts[0], ""
    att = df["ATT_XY"].apply(split_att)
    df["ATT_X"] = att.apply(lambda t: t[0])
    df["ATT_Y"] = att.apply(lambda t: t[1])

    return df[["ATT_X", "ATT_Y", "H", "PROB"]].reset_index(drop=True)

# ------------------------------------------------------------
# 9) Latent diagnostics from q (supports sparsity via policies)
# ------------------------------------------------------------

def _latent_diagnostics_from_q(
    q_latent: np.ndarray,
    weights: Optional[np.ndarray],
    eps: float = 1e-12
) -> Dict[str, float]:
    N, K = q_latent.shape
    w = np.ones(N, dtype=np.float64) if weights is None else np.asarray(weights, dtype=np.float64)
    w = np.clip(w, 0.0, None)
    wsum = w.sum() + eps

    H = -(q_latent * np.log(np.clip(q_latent, eps, 1.0))).sum(axis=1)
    H_mean = float((H * w).sum() / wsum)
    H_norm = H_mean / math.log(K)

    p = (q_latent * w[:, None]).sum(axis=0) / wsum
    p = p / p.sum()
    H_marg = float(-(p * np.log(np.clip(p, eps, 1.0))).sum())
    I_est  = float(max(0.0, H_marg - H_mean))

    qmax = q_latent.max(axis=1)
    q2   = np.partition(q_latent, -2, axis=1)[:, -2]
    gap  = qmax - q2
    qmax_mean = float((qmax * w).sum() / wsum)
    gap_mean  = float((gap  * w).sum() / wsum)

    KL_qU_mean = float(math.log(K) - H_mean)
    KL_qU_norm = KL_qU_mean / math.log(K)

    winners = q_latent.argmax(axis=1)
    win_counts = np.bincount(winners, minlength=K).astype(np.float64)
    win_ratio_max = float(win_counts.max() / max(N, 1))

    return {
        "E[H(q)]": H_mean,
        "E[H(q)]/logK": H_norm,
        "H_marg": H_marg,
        "I_est = H(P) - E[H(q)]": I_est,
        "E[max q]": qmax_mean,
        "E[max-min2 gap]": gap_mean,
        "E[KL(q||U)]/logK": KL_qU_norm,
        "argmax_dominance_ratio": win_ratio_max,
        "K": float(K),
        "N": float(N),
    }

@torch.no_grad()
def _encode_q_from_df(
    df: 'pd.DataFrame',
    encoder: 'nn.Module',
    K: int,
    device: str,
    batch_size: int = 8192,
    tau: float = 1.0,
    weights_col: Optional[str] = None,
    support_policy: Optional[SupportPolicy] = None
) -> Tuple[np.ndarray, Optional[np.ndarray]]:
    weights = None
    if (weights_col is not None) and (weights_col in df.columns):
        weights = df[weights_col].to_numpy(dtype=np.float64)

    phi_np = np.vstack(df["PHI_IDX"].to_numpy()).astype(np.int64)
    N = phi_np.shape[0]
    q_blocks = []

    encoder.eval()
    for start in range(0, N, batch_size):
        end = min(N, start + batch_size)
        phi_b = torch.from_numpy(phi_np[start:end]).to(device=device, dtype=torch.long)
        logits = encoder(phi_b)
        if support_policy is None:
            q = softmax_tau(logits, tau)
        else:
            mask = support_policy(logits, phi_b)
            q = masked_softmax(logits, mask, tau)
        q_blocks.append(q.cpu().numpy())

    q_latent = np.vstack(q_blocks)
    return q_latent, weights

def uniformity_report_from_df(
    df_hts: 'pd.DataFrame',
    df_pcm: 'pd.DataFrame',
    encoder: 'nn.Module',
    pack: Dict,
    device: str,
    batch_size: int = 8192,
    tau: Optional[float] = None,
    support_policy: Optional[SupportPolicy] = None,
) -> Dict[str, Dict[str, float]]:
    if tau is None:
        tau = float(pack.get("tau_final", 1.0))
    K = int(pack["encoder_cfg"]["K"])

    q_hts, _ = _encode_q_from_df(df_hts, encoder, K, device, batch_size, tau, weights_col=None, support_policy=support_policy)
    rep_hts = _latent_diagnostics_from_q(q_hts, weights=None)

    wcol = "COUNT" if "COUNT" in df_pcm.columns else None
    q_pcm, w_pcm = _encode_q_from_df(df_pcm, encoder, K, device, batch_size, tau, weights_col=wcol, support_policy=support_policy)
    rep_pcm = _latent_diagnostics_from_q(q_pcm, weights=w_pcm)

    return {"HTS": rep_hts, "PCM": rep_pcm}

# ------------------------------------------------------------
# 10) Merge-based latent encoding for PCM (masked export)
# ------------------------------------------------------------

@torch.no_grad()
def encode_pcm_to_latent_df(
    df_pcm: pd.DataFrame,
    model: nn.Module,
    pack: Dict,
    device: str = "cpu",
    batch_size: int = 8192,
    tau: Optional[float] = None,
    topk_h: Optional[int] = None,
    h_prob_threshold: Optional[float] = None,
    support_policy: Optional[SupportPolicy] = None
) -> Tuple[pd.DataFrame, np.ndarray]:
    """
    Returns q_df columns: ["PCM_ROW","H","PROB"] where PROB = masked q_phi(H|Phi_row).
    If topk_h or threshold provided, additionally sparsifies the exported rows.
    """
    if tau is None:
        tau = float(pack.get("tau_final", 1.0))
    K = int(pack["encoder_cfg"]["K"])

    all_rows = []
    q_blocks = []

    N = len(df_pcm)
    model.eval()

    for start in range(0, N, batch_size):
        end = min(N, start + batch_size)
        phi_b = np.vstack(df_pcm["PHI_IDX"].iloc[start:end].to_numpy()).astype(np.int64)
        phi_b = torch.from_numpy(phi_b).to(device=device, dtype=torch.long)

        logits = model(phi_b)
        if support_policy is None:
            q = softmax_tau(logits, tau)
        else:
            mask = support_policy(logits, phi_b)
            q = masked_softmax(logits, mask, tau)

        q_cpu = q.to("cpu").numpy()
        q_blocks.append(q_cpu)

        B = q_cpu.shape[0]
        if (topk_h is None) and (h_prob_threshold is None):
            pcm_idx = np.repeat(np.arange(start, end), K)
            h_idx   = np.tile(np.arange(K), B)
            prob    = q_cpu.reshape(-1)
            block = np.column_stack([pcm_idx, h_idx, prob])
            all_rows.append(block)
        else:
            for i in range(B):
                probs = q_cpu[i]
                if h_prob_threshold is not None:
                    keep = np.where(probs >= h_prob_threshold)[0]
                    vals = probs[keep]
                else:
                    k = min(int(topk_h), K)
                    keep = np.argpartition(-probs, k-1)[:k]
                    keep = keep[np.argsort(-probs[keep])]
                    vals = probs[keep]
                pcm_row = start + i
                if len(keep) > 0:
                    block = np.column_stack([np.full_like(keep, pcm_row), keep, vals])
                    all_rows.append(block)

    q_latent = np.vstack(q_blocks) if len(q_blocks) > 0 else np.empty((0, K))
    if len(all_rows) == 0:
        q_df = pd.DataFrame(columns=["PCM_ROW", "H", "PROB"])
    else:
        arr = np.vstack(all_rows).astype(np.float64)
        q_df = pd.DataFrame(arr, columns=["PCM_ROW", "H", "PROB"]).astype(
            {"PCM_ROW": int, "H": int, "PROB": float}
        )
    return q_df, q_latent

# ------------------------------------------------------------
# 11) Optional: conditional filtering helper
# ------------------------------------------------------------

def filter_by_conditioned_probability(
    df: pd.DataFrame,
    att_cols: List[str],
    prob_col: str = "Prob_XYZ_fus",
    *,
    abs_threshold: float = 1e-4,
    min_group_sum: float = 1e-15,
    add_column_name: str = "P_cond",
    return_splits: bool = True,
    preserve: str = "group_then_global"  # {"group_then_global", "global_only", "none"}
) -> Tuple[pd.DataFrame, Optional[pd.DataFrame]]:
    """
    Filter by per-group conditional probability with an absolute threshold.

    Steps:
      1) Compute group sums and conditional prob: P_cond = prob / group_sum
      2) Keep rows with P_cond >= abs_threshold
      3) Renormalize P_cond within each group to sum to 1
      4) Preserve probability mass:
         - "group_then_global": restore original group mass, then global normalize
         - "global_only": only global normalize
         - "none": leave raw group mass as-is (after step 3)
    Returns:
      kept_df, dropped_df (or None if return_splits=False)
    """
    if preserve not in {"group_then_global", "global_only", "none"}:
        raise ValueError("preserve must be one of {'group_then_global','global_only','none'}")

    df = df.copy()

    # 1) Group totals and conditional probability
    group_sum_orig = (
        df.groupby(att_cols, dropna=False)[prob_col]
          .transform("sum")
          .clip(lower=min_group_sum)
    )
    df[add_column_name] = df[prob_col] / group_sum_orig

    # 2) Keep rows above absolute conditional threshold
    keep_mask = df[add_column_name] >= float(abs_threshold)
    kept = df[keep_mask].reset_index(drop=True)
    dropped = df[~keep_mask].reset_index(drop=True) if return_splits else None

    if kept.empty:
        return kept, dropped

    # 3) Renormalize conditional probabilities within each group
    group_sum_kept_cond = (
        kept.groupby(att_cols, dropna=False)[add_column_name]
            .transform("sum")
            .clip(lower=min_group_sum)
    )
    kept[add_column_name] = kept[add_column_name] / group_sum_kept_cond

    # 4) Preserve probability mass as requested
    if preserve == "group_then_global":
        # Restore each group's original mass
        group_mass = (
            df[att_cols + [prob_col]]
            .groupby(att_cols, dropna=False)[prob_col]
            .sum()
            .rename("_GROUP_MASS_ORIG")
            .reset_index()
        )
        kept = kept.merge(group_mass, on=att_cols, how="left")
        kept["_GROUP_MASS_ORIG"] = kept["_GROUP_MASS_ORIG"].fillna(0.0)
        kept[prob_col] = kept[add_column_name] * kept["_GROUP_MASS_ORIG"]
        kept.drop(columns=["_GROUP_MASS_ORIG"], inplace=True)

        # Global normalization (optional but common in probability tables)
        total = float(kept[prob_col].sum())
        if total > 0:
            kept[prob_col] = kept[prob_col] / total

    elif preserve == "global_only":
        total = float(kept[prob_col].sum())
        if total > 0:
            kept[prob_col] = kept[prob_col] / total

    else:
        # "none": leave as-is; prob_col unchanged except for earlier steps
        pass

    return kept, dropped

def filter_by_conditioned_probability2(
    df: pd.DataFrame,
    att_cols: List[str],
    prob_col: str = "Prob_XYZ_fus",
    *,
    abs_threshold: Optional[float] = 1e-4,
    quantile_q: Optional[float] = None,
    topk_per_group: Optional[int] = None,
    min_group_sum: float = 1e-15,
    add_column_name: str = "P_cond",
    return_splits: bool = True,
    preserve: str = "group_then_global"   # {"group_then_global", "global_only", "none"}
) -> Tuple[pd.DataFrame, Optional[pd.DataFrame]]:
    modes = [abs_threshold is not None, quantile_q is not None, topk_per_group is not None]
    if sum(modes) != 1:
        raise ValueError("Pick exactly ONE of abs_threshold, quantile_q, or topk_per_group.")
    if preserve not in {"group_then_global", "global_only", "none"}:
        raise ValueError("preserve must be one of {'group_then_global','global_only','none'}")

    df = df.copy()
    group_sum_orig = df.groupby(att_cols, dropna=False)[prob_col].transform("sum").clip(lower=min_group_sum)
    df[add_column_name] = df[prob_col] / group_sum_orig

    if abs_threshold is not None:
        keep_mask = df[add_column_name] >= float(abs_threshold)
    elif quantile_q is not None:
        if not (0.0 <= quantile_q <= 1.0):
            raise ValueError("quantile_q must be in [0, 1].")
        q_thr = df.groupby(att_cols, dropna=False)[add_column_name].transform(lambda s: s.quantile(quantile_q))
        keep_mask = df[add_column_name] >= q_thr
    else:
        if topk_per_group <= 0:
            raise ValueError("topk_per_group must be > 0.")
        df["_rank_in_group"] = (
            df.groupby(att_cols, dropna=False)[add_column_name]
              .rank(method="first", ascending=False)
        )
        keep_mask = df["_rank_in_group"] <= float(topk_per_group)
        df.drop(columns=["_rank_in_group"], inplace=True)

    kept    = df[keep_mask].reset_index(drop=True)
    dropped = df[~keep_mask].reset_index(drop=True) if return_splits else None

    if kept.empty:
        return kept, dropped

    group_sum_kept_cond = kept.groupby(att_cols, dropna=False)[add_column_name].transform("sum").clip(lower=min_group_sum)
    kept[add_column_name] = kept[add_column_name] / group_sum_kept_cond

    if preserve == "group_then_global":
        kept = kept.merge(
            df[att_cols + [prob_col]].groupby(att_cols, dropna=False).sum().rename(columns={prob_col: "_GROUP_MASS_ORIG"}).reset_index(),
            on=att_cols, how="left"
        )
        kept["_GROUP_MASS_ORIG"] = kept["_GROUP_MASS_ORIG"].fillna(0.0)
        kept[prob_col] = kept[add_column_name] * kept["_GROUP_MASS_ORIG"]
        kept.drop(columns=["_GROUP_MASS_ORIG"], inplace=True)
        total = float(kept[prob_col].sum())
        if total > 0:
            kept[prob_col] = kept[prob_col] / total
    elif preserve == "global_only":
        total = float(kept[prob_col].sum())
        if total > 0:
            kept[prob_col] = kept[prob_col] / total
    else:
        pass

    return kept, dropped




# Validation: Stage 1 Data fusion

In [ ]:
# ------------------------------------------------------------
# __main__: Example end-to-end run (adjust paths)
# ------------------------------------------------------------

import pandas as pd
from typing import List, Optional, Tuple

def filter_by_conditioned_probability(
    df: pd.DataFrame,
    att_cols: List[str],
    prob_col: str = "Prob_XYZ_fus",
    *,
    abs_threshold: float = 1e-4,
    add_column_name: str = "P_cond",
    return_splits: bool = True,
    preserve: str = "group_then_global"  # {"group_then_global", "global_only", "none"}
) -> Tuple[pd.DataFrame, Optional[pd.DataFrame]]:
    """
    Filter by per-group conditional probability with an absolute threshold.

    Steps:
      1) Compute group sums and conditional prob: P_cond = prob / group_sum
      2) Keep rows with P_cond >= abs_threshold
      3) Renormalize P_cond within each group to sum to 1
      4) Preserve probability mass:
         - "group_then_global": restore original group mass, then global normalize
         - "global_only": only global normalize
         - "none": leave raw group mass as-is (after step 3)
    Returns:
      kept_df, dropped_df (or None if return_splits=False)
    """
    if preserve not in {"group_then_global", "global_only", "none"}:
        raise ValueError("preserve must be one of {'group_then_global','global_only','none'}")

    df = df.copy()

    # 1) Group totals and conditional probability
    group_sum_orig = (
        df.groupby(att_cols, dropna=False)[prob_col]
          .transform("sum")
          .clip(lower=1e-15)
    )
    df[add_column_name] = df[prob_col] / group_sum_orig

    # 2) Keep rows above absolute conditional threshold
    keep_mask = df[add_column_name] >= float(abs_threshold)
    kept = df[keep_mask].reset_index(drop=True)
    dropped = df[~keep_mask].reset_index(drop=True) if return_splits else None

    if kept.empty:
        return kept, dropped

    # 3) Renormalize conditional probabilities within each group
    group_sum_kept_cond = (
        kept.groupby(att_cols, dropna=False)[add_column_name]
            .transform("sum")
            .clip(lower=1e-15)
    )
    kept[add_column_name] = kept[add_column_name] / group_sum_kept_cond

    # 4) Preserve probability mass as requested
    if preserve == "group_then_global":
        # Restore each group's original mass
        group_mass = (
            df[att_cols + [prob_col]]
            .groupby(att_cols, dropna=False)[prob_col]
            .sum()
            .rename("_GROUP_MASS_ORIG")
            .reset_index()
        )
        kept = kept.merge(group_mass, on=att_cols, how="left")
        kept["_GROUP_MASS_ORIG"] = kept["_GROUP_MASS_ORIG"].fillna(0.0)
        kept[prob_col] = kept[add_column_name] * kept["_GROUP_MASS_ORIG"]
        kept.drop(columns=["_GROUP_MASS_ORIG"], inplace=True)

        # Global normalization (optional but common in probability tables)
        total = float(kept[prob_col].sum())
        if total > 0:
            kept[prob_col] = kept[prob_col] / total

    elif preserve == "global_only":
        total = float(kept[prob_col].sum())
        if total > 0:
            kept[prob_col] = kept[prob_col] / total

    else:
        # "none": leave as-is; prob_col unchanged except for earlier steps
        pass

    return kept, dropped



# ------------------------------------------------------------
# 11) Validation: Marginal Distribution Comparison
# ------------------------------------------------------------

def validate_z_marginals(df_fused: pd.DataFrame, df_pcm_orig: pd.DataFrame, att_z_cols: list):
    print("\n" + "="*50)
    print("VALIDATION: Marginal Distribution of Z (PCM)")
    print("="*50)

    # 1. Prepare Ground Truth from original PCM
    # We use the 'COUNT' column to represent the true distribution
    gt_z = df_pcm_orig.groupby(att_z_cols)['COUNT'].sum().reset_index()
    gt_z['P_Z_true'] = gt_z['COUNT'] / gt_z['COUNT'].sum()

    # 2. Prepare Fused Marginal
    fus_z = df_fused.groupby(att_z_cols)['Prob_XYZ_fus'].sum().reset_index()
    fus_z.rename(columns={'Prob_XYZ_fus': 'P_Z_fused'}, inplace=True)

    # 3. Merge for comparison
    comparison = pd.merge(gt_z, fus_z, on=att_z_cols, how='outer').fillna(0)

    # 4. Calculate Metrics
    # Total Variation Distance: 0.5 * sum|p - q|
    tvd = 0.5 * np.abs(comparison['P_Z_true'] - comparison['P_Z_fused']).sum()

    # Pearson Correlation
    corr = np.corrcoef(comparison['P_Z_true'], comparison['P_Z_fused'])[0, 1]

    # Print Results
    print(f"Total Unique Z-bins (Trips): {len(comparison)}")
    print(f"Total Variation Distance (TVD): {tvd:.6f}  (Ideal: 0.0)")
    print(f"Pearson Correlation:           {corr:.6f}  (Ideal: 1.0)")

    # Summary of Drift
    max_drift = (comparison['P_Z_true'] - comparison['P_Z_fused']).abs().max()
    print(f"Max Probability Drift:         {max_drift:.6e}")

    if tvd < 0.01:
        print("RESULT: SUCCESS - P(att_Z) is highly preserved.")
    else:
        print("RESULT: WARNING - Significant drift detected in Z-marginal.")



if __name__ == "__main__":

    ff = 0.5

    # --- Load data ---
    df_hts = pd.read_csv(os.path.join(path_data, f"val_data_hts_trip_{ff}.csv"))
    df_pcm = pd.read_csv(os.path.join(path_data, "val_data_pcm_trip.csv"))

    # 1) Aggregate PCM counts per (O,D,start,end)
    if "COUNT" not in df_pcm.columns:
        df_pcm["COUNT"] = 1.0
        df_pcm["COUNT"] = df_pcm.groupby(
            ["ORIGIN_SUBZONE","DESTINATION_SUBZONE","TRIP_STARTTIME","TRIP_ENDTIME"]
        )["COUNT"].transform("sum")
        df_pcm = df_pcm.drop_duplicates()

    # 2) Base features for discretization
    df_hts["TRIP_DISTANCE"] = distance(df_hts, ['ORIGIN_SUBZONE_X','DESTINATION_SUBZONE_X','ORIGIN_SUBZONE_Y','DESTINATION_SUBZONE_Y'])
    df_pcm["TRIP_DISTANCE"] = distance(df_pcm, ['ORIGIN_SUBZONE_X','DESTINATION_SUBZONE_X','ORIGIN_SUBZONE_Y','DESTINATION_SUBZONE_Y'])
    df_land_use = land_use(df_hts, df_pcm) #check
    df_mode = mode_share(df_hts, df_pcm) #check
    df_hts = df_hts.merge(df_land_use, on='DESTINATION_SUBZONE').merge(df_mode, on='DESTINATION_SUBZONE')
    df_pcm = df_pcm.merge(df_land_use, on='DESTINATION_SUBZONE').merge(df_mode, on='DESTINATION_SUBZONE')
    # Uniform
    #df_pcm.fillna(0.1, inplace=True)

    # 3) Φ (DISCRETE): fit on HTS and apply to HTS/PCM, then store indices
    nb = 10
    spec = DiscreteSpec(n_dist=nb, n_start=nb, n_end=nb, n_lu_mode_clusters=nb,
                        dist_binning="uniform", time_binning="uniform")

    fitted = fit_semantic_spec_on_hts(df_hts, spec)
    df_hts = apply_semantic_spec(df_hts, fitted)
    df_pcm = apply_semantic_spec(df_pcm, fitted)

    # Use the stored segment sizes so bucket policies work later
    phi_segments_meta = fitted["phi_segments"]
    n_d = phi_segments_meta["n_dist"]
    n_s = phi_segments_meta["n_start"]
    n_e = phi_segments_meta["n_end"]
    n_c = phi_segments_meta["n_cluster"]

    df_hts, phi_dim = build_phi_indices_from_bins(df_hts)
    df_pcm, _       = build_phi_indices_from_bins(df_pcm)

    # 4) Attributes (X,Y,Z) and stringified ATT_X, ATT_Y (for XY classes)
    att_X = ['AGE','GENDER','INCOME']  #
    att_Y = ['TRIP_CNT','TRIP_MAX','TRIP_PURPOSE','TRAVEL_MODE']
    att_Z = ['ORIGIN_SUBZONE','DESTINATION_SUBZONE','TRIP_STARTTIME','TRIP_ENDTIME',
             'ORIGIN_SUBZONE_X','DESTINATION_SUBZONE_X','ORIGIN_SUBZONE_Y','DESTINATION_SUBZONE_Y']

    df_hts = df_hts[att_X + att_Y + att_Z + ['PHI_IDX']]
    df_pcm = df_pcm[att_Z + ['PHI_IDX','COUNT']]

    df_hts['ATT_X'] = df_hts[att_X].astype(str).apply('_'.join, axis=1)
    df_hts['ATT_Y'] = df_hts[att_Y].astype(str).apply('_'.join, axis=1)
    df_hts['ATT_Z'] = df_hts[att_Z].astype(str).apply('_'.join, axis=1)
    df_pcm['ATT_Z'] = df_pcm[att_Z].astype(str).apply('_'.join, axis=1)

    df_hts = reduce_mem_usage(df_hts)
    df_pcm = reduce_mem_usage(df_pcm)

    # 5) Train with sparse per-Φ support (embedding encoder)
    cfg = TrainConfig(
        K=2500,                       # large K is fine with embeddings 2000
        enc_hidden=256,
        enc_layers=2,
        enc_dropout=0.0,
        epochs=5,
        batch_size_hts=8192,
        batch_size_pcm=16384,
        lr=2e-3,
        wd=1e-4,
        device=("cuda" if torch.cuda.is_available() else "cpu"),
        use_normalized_objective=True,
        rebuild_every=1,
        progress=True,
        tau_start=0.01,
        tau_end=0.01,
        tau_schedule="cosine",
        topk_h=10,
        use_bucket_candidates=False,
        # bucket_topk=8,
        # refresh_bucket_every=2
    )

    out = train_encoder_only_fusion(df_hts, df_pcm, cfg, phi_segments_meta)
    model, pack = out["model"], out["pack"]

    # 6) Build the decoder table once after training
    decoder_df = pack_to_dataframe(pack).copy()   # ["ATT_X","ATT_Y","H","PROB"]
    # Threshold + renormalize within latent state H
    decoder_df = decoder_df[decoder_df['PROB'] > 1e-4] #1e-3
    sum_h = decoder_df.groupby('H', observed=True)['PROB'].transform('sum')
    decoder_df = decoder_df[sum_h > 0]
    decoder_df['PROB'] = decoder_df['PROB'] / sum_h[sum_h > 0]
    decoder_df.rename(columns={'PROB': 'PROB_XY|H'}, inplace=True)

    # 7) Prepare the SAME support policy for inference/merging
    tau_eval = float(pack.get("tau_final", 1.0))

    if cfg.use_bucket_candidates:
        phi_hts = torch.from_numpy(np.vstack(df_hts["PHI_IDX"].to_numpy()).astype(np.int64))
        dummy_y = torch.zeros((phi_hts.shape[0],), dtype=torch.long)
        hts_phi_loader = DataLoader(TensorDataset(phi_hts, dummy_y),
                                    batch_size=cfg.batch_size_hts, shuffle=False, num_workers=0)
        cand_map = build_phi_bucket_candidates_embed(
            model, hts_phi_loader, cfg.K, tau_eval,
            topk=cfg.bucket_topk, device=cfg.device
        )
        support_policy_infer = bucket_support_policy_embed(cand_map)
    else:
        support_policy_infer = topk_support_policy(cfg.topk_h)

    # 8) Encode PCM to latent with the same support policy
    q_df, q_latent = encode_pcm_to_latent_df(
        df_pcm=df_pcm, model=model, pack=pack, device=cfg.device, batch_size=8192,
        tau=tau_eval,
        topk_h=2,                      # optional export sparsification
        h_prob_threshold=None,
        support_policy=support_policy_infer
    )
    q_df.rename(columns={'PROB': 'PROB_H'}, inplace=True)

    # 9) Merge for fused probabilities

    # 1. Prepare PCM and HTS data components
    df_fus_pcm = df_pcm.copy()
    df_fus_pcm['P_Z'] = df_fus_pcm['COUNT'] / df_fus_pcm['COUNT'].sum()
    df_fus_pcm['PCM_ROW'] = df_fus_pcm.index
    df_fus_pcm = df_fus_pcm.merge(q_df, on='PCM_ROW') # [ATT_Z, P_Z, H, PROB_H]

    df_hts_attrs = df_hts[att_X + att_Y + ['ATT_X', 'ATT_Y']].drop_duplicates()
    df_fus_hts = decoder_df.merge(df_hts_attrs, on=['ATT_X', 'ATT_Y']) # [ATT_X, ATT_Y, H, PROB_XY|H]

    # 2. Identify Cohorts
    att_cols = ['AGE', 'GENDER', 'INCOME', 'TRIP_MAX', 'TRIP_CNT']
    df_cohorts = df_fus_hts[att_cols].drop_duplicates().sort_values(by=att_cols)

    fus_parts = []

    # 3. Cohort Iterator Loop
    for age, gender, income, trip_max, trip_cnt in df_cohorts.itertuples(index=False):
        # Filter Person-side
        mask = (
            (df_fus_hts['AGE'] == age) & (df_fus_hts['GENDER'] == gender) & (df_fus_hts['INCOME'] == income) &
            (df_fus_hts['TRIP_MAX'] == trip_max) & (df_fus_hts['TRIP_CNT'] == trip_cnt)
        )
        df_hts_part = df_fus_hts.loc[mask].copy()
        if df_hts_part.empty: continue

        # Merge with PCM-side via Bridge H
        # This result is P(XY, Z, H)
        df_part = df_hts_part.merge(df_fus_pcm, on='H')

        # Raw Probability: P(XY|H) * P(H|Z) * P(Z)
        df_part['Prob_XYZ_fus'] = df_part['PROB_XY|H'] * df_part['PROB_H'] * df_part['P_Z']

        # Immediate Aggregation to (X, Y, Z) to save RAM
        # We MUST keep att_Z here to perform the global correction later
        df_part = df_part.groupby(att_X + att_Y + att_Z)['Prob_XYZ_fus'].sum().reset_index()

        # Light Filtering (Conditional threshold)
        # Note: We can't fully re-normalize yet because we don't have the other cohorts
        df_part = df_part[df_part['Prob_XYZ_fus'] > 1e-9]

        fus_parts.append(df_part)
        print(f"AGE={age}, GENDER={gender}, INCOME={income}, TRIP_MAX={trip_max}, TRIP_CNT={trip_cnt}")

        # Cleanup
        del df_hts_part, df_part
        # gc.collect() # Uncomment if RAM is extremely tight

    # 4. Global Z-Preservation Correction
    print("Merging cohorts and applying Z-preservation...")
    df_fusion = pd.concat(fus_parts, ignore_index=True)
    del fus_parts

    # Collapse any duplicates that appeared in different cohorts
    df_fusion = df_fusion.groupby(att_X + att_Y + att_Z)['Prob_XYZ_fus'].sum().reset_index()

    # --- THE Z-PRESERVATION KEY ---
    # Calculate the current marginal mass for each trip type Z
    z_mass_current = df_fusion.groupby(att_Z)['Prob_XYZ_fus'].transform('sum')

    # Calculate the target marginal mass from original PCM
    # We use the original df_pcm to get the 'COUNT' distribution
    df_pcm_targets = df_pcm.groupby(att_Z)['COUNT'].sum().reset_index()
    df_pcm_targets['target_P_Z'] = df_pcm_targets['COUNT'] / df_pcm_targets['COUNT'].sum()

    # Merge target mass onto our fused data
    df_fusion = df_fusion.merge(df_pcm_targets[att_Z + ['target_P_Z']], on=att_Z, how='left')

    # Scale probabilities so that sum(Prob) for each Z equals target_P_Z
    df_fusion['Prob_XYZ_fus'] *= (df_fusion['target_P_Z'] / z_mass_current.replace(0, 1))

    # Final clean up
    df_fusion.drop(columns=['target_P_Z'], inplace=True)
    df_fusion['Prob_XYZ_fus'] /= df_fusion['Prob_XYZ_fus'].sum()

    '''df_fusion = df_pcm.copy()
    df_fusion.drop(columns=['PHI_IDX'], inplace=True)
    df_fusion['PROB'] = df_fusion['COUNT'] / df_fusion['COUNT'].sum()
    df_fusion['PCM_ROW'] = df_fusion.index

    df_fusion = df_fusion.merge(q_df, on='PCM_ROW')
    df_fusion.drop(columns=['PCM_ROW', 'COUNT'], inplace=True)

    df_hts_ = df_hts[att_X + att_Y + ['ATT_X', 'ATT_Y']].drop_duplicates()
    decoder_df = decoder_df.merge(df_hts_, on=['ATT_X', 'ATT_Y'])

    print(df_fusion.info())
    print(decoder_df.info())

    #################################

    df_fus_pcm = df_fusion.copy()
    df_fus_hts = decoder_df.copy()
    del df_fusion, decoder_df

    # -----------------------
    # 3) Cohort iterator
    # -----------------------
    att_cols = ['AGE', 'GENDER', 'INCOME', 'TRIP_MAX', 'TRIP_CNT']
    df_indivudal_structure = (
        df_fus_hts[att_cols]
        .drop_duplicates()
        .sort_values(by=att_cols)
        .reset_index(drop=True)
    )

    total_length = 0
    fus_parts = []  # collect all df_tours_seed parts here

    for age, gender, income, trip_max, trip_cnt in df_indivudal_structure[att_cols].itertuples(index=False):
        age = int(age); gender = int(gender); trip_max = int(trip_max); trip_cnt = int(trip_cnt)

        # Filter HTS fusion rows for the cohort, respecting TRIP_CNT <= trip_cnt
        mask = (
            (df_fus_hts['TRIP_CNT'] == trip_cnt) &
            (df_fus_hts['AGE'] == age) &
            (df_fus_hts['INCOME'] == income) &
            (df_fus_hts['GENDER'] == gender) &
            (df_fus_hts['INCOME'] == income) &
            (df_fus_hts['TRIP_MAX'] == trip_max)
        )

        df_fus_part = df_fus_hts.loc[mask].copy()
        df_fus_part = df_fus_part.merge(df_fus_pcm, on='H')
        df_fus_part['Prob_XYZ_fus'] = df_fus_part['PROB_XY|H'] * df_fus_part['PROB_H'] * df_fus_part['PROB']
        df_fus_part.drop(columns=['PROB_XY|H', 'PROB_H', 'H', 'PROB'], inplace=True)
        df_fus_part['Prob_XYZ_fus'] = df_fus_part.groupby(['ATT_X', 'ATT_Y', 'ATT_Z'])['Prob_XYZ_fus'].transform('sum')
        df_fus_part = df_fus_part[att_X + att_Y + att_Z + ['Prob_XYZ_fus']]
        df_fus_part = df_fus_part.drop_duplicates()

        # Filtering
        att_col = att_X + att_Y
        df_fus_part['Prob_group'] = df_fus_part.groupby(att_col)['Prob_XYZ_fus'].transform('sum')
        df_fus_part['Prob_cond'] = df_fus_part['Prob_XYZ_fus'] / df_fus_part['Prob_group']
        df_fus_part = df_fus_part[df_fus_part['Prob_cond'] > 1e-4]
        df_fus_part['Prob_group_filtered'] = df_fus_part.groupby(att_col)['Prob_XYZ_fus'].transform('sum')
        df_fus_part['Prob_XYZ_fus'] = df_fus_part['Prob_XYZ_fus'] * df_fus_part['Prob_group'] / df_fus_part['Prob_group_filtered']

        # Accumulate
        fus_parts.append(df_fus_part)
        total_length += len(df_fus_part)
        print(f"AGE={age}, GENDER={gender}, INCOME={income}, TRIP_MAX={trip_max}, TRIP_CNT={trip_cnt}, LENGTH={total_length}")



    df_fusion = pd.concat(fus_parts, ignore_index=True)
    del fus_parts
    df_fusion['Prob_XYZ_fus'] = df_fusion['Prob_XYZ_fus'] / df_fusion['Prob_XYZ_fus'].sum()
    df_fusion = df_fusion.drop_duplicates()'''

    print(df_fusion['Prob_XYZ_fus'].sum())
    print(df_fusion.info())
    df_fusion.to_csv(os.path.join(path_result, f'val_sim_trip_proposed_{ff}.csv'), index=False)

    # Run the check
    validate_z_marginals(df_fusion, df_pcm, att_Z)



[decoder] pass chunk 0
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 001] tau=0.0100  L_hts=0.614379  total=0.614379
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 002] tau=0.0100  L_hts=0.558169  total=0.558169
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 003] tau=0.0100  L_hts=0.539227  total=0.539227
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 004] tau=0.0100  L_hts=0.524448  total=0.524448
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 005] tau=0.0100  L_hts=0.513572  total=0.513572
[decoder] pass chunk 0
[FINAL] K=2500  tau=0.0100  L_hts=0.510006
AGE=0, GENDER=0, INCOME=2, TRIP_MAX=2, TRIP_CNT=1
AGE=0, GENDER=0, INCOME=2, TRIP_MAX=2, TRIP_CNT=2
AGE=0, GENDER=0, INCOME=3, TRIP_MAX=2, TRIP_CNT=1
AGE=0, GENDER=0, INCOME=3, TRIP_MAX=2, TRIP_CNT=2
AGE=0, GENDER=0, INCOME=3, TRIP_MAX=3, TRIP_CNT=1
AGE=0, GENDER=0, INCOME=3, TRIP_MAX=3, TRIP_CNT=2
AGE=0, GENDER=0, INCOME=3, TRIP_MAX=3, TRIP_CNT=3
AGE=0, GENDER=0, INCOME=3, TRIP_MAX=5, TRIP_CNT=1
AGE=0,

#Validation: Stage 2 Tour generation

In [ ]:
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.cluster import Birch
from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from scipy.special import rel_entr, kl_div
import math
import time
import warnings
warnings.filterwarnings('ignore')

TRIP_MAX_ = 7

# ============================================================
# Memory helpers
# ============================================================

def _downcast_int(s: pd.Series):
    if s.isnull().any():
        return s
    i32 = s.astype(np.int64)
    if i32.min() >= np.iinfo(np.int8).min and i32.max() <= np.iinfo(np.int8).max:
        return i32.astype(np.int8)
    if i32.min() >= np.iinfo(np.int16).min and i32.max() <= np.iinfo(np.int16).max:
        return i32.astype(np.int16)
    if i32.min() >= np.iinfo(np.int32).min and i32.max() <= np.iinfo(np.int32).max:
        return i32.astype(np.int32)
    return i32

def _downcast_float(s: pd.Series):
    return s.astype(np.float32)

def reduce_mem_usage(df: pd.DataFrame, use_cats: bool = True):
    for col in df.columns:
        if pd.api.types.is_integer_dtype(df[col]):
            df[col] = _downcast_int(df[col])
        elif pd.api.types.is_float_dtype(df[col]):
            df[col] = _downcast_float(df[col])
        elif use_cats and (df[col].nunique(dropna=False) / max(1, len(df[col])) < 0.5):
            df[col] = df[col].astype('category')
    return df

def as_codes(df: pd.DataFrame, cols):
    for c in cols:
        if not pd.api.types.is_categorical_dtype(df[c]):
            df[c] = df[c].astype('category')
        df[c] = df[c].cat.codes.astype(np.int32)
    return df

# ============================================================
# Math helpers
# ============================================================

def js_div(p, q):
    h = 0.5 * np.add(p, q)
    return 0.5 * rel_entr(p, h) + 0.5 * rel_entr(q, h)

# ============================================================
# Column ops
# ============================================================

def _renmap(i: int):
    return {
        'TRIP_PURPOSE':          f'TRIP_PURPOSE_{i}',
        'TRAVEL_MODE':           f'TRAVEL_MODE_{i}',
        'ORIGIN_SUBZONE':        f'ORIGIN_SUBZONE_{i}',
        'DESTINATION_SUBZONE':   f'DESTINATION_SUBZONE_{i}',
        'TRIP_STARTTIME':        f'TRIP_STARTTIME_{i}',
        'TRIP_ENDTIME':          f'TRIP_ENDTIME_{i}',
        'ORIGIN_SUBZONE_X':      f'ORIGIN_SUBZONE_X_{i}',
        'ORIGIN_SUBZONE_Y':      f'ORIGIN_SUBZONE_Y_{i}',
        'DESTINATION_SUBZONE_X': f'DESTINATION_SUBZONE_X_{i}',
        'DESTINATION_SUBZONE_Y': f'DESTINATION_SUBZONE_Y_{i}',
        'Prob_XYZ_fus':          f'Prob_{i}',
    }

def rename_columns(df: pd.DataFrame, i: int):
    df.rename(columns=_renmap(i), inplace=True)

# ============================================================
# Sampling utils
# ============================================================

def _prob_sample_indices(prob: np.ndarray, N: int) -> np.ndarray:
    prob = np.asarray(prob, dtype=np.float64)
    prob_sum = prob.sum()
    if prob_sum <= 0:
        return np.array([], dtype=np.int64)
    prob = prob / prob_sum
    return np.random.choice(len(prob), size=N, replace=True, p=prob)

# ============================================================
# Core pipeline
# ============================================================

def simulate_trips(df_fus: pd.DataFrame, i: int, N: int) -> pd.DataFrame:
    att_X = ['AGE', 'GENDER', 'INCOME', 'TRIP_MAX']
    att_Z = [f'TRIP_PURPOSE_{i}', f'TRAVEL_MODE_{i}',
             f'ORIGIN_SUBZONE_{i}', f'ORIGIN_SUBZONE_X_{i}', f'ORIGIN_SUBZONE_Y_{i}',
             f'DESTINATION_SUBZONE_{i}', f'DESTINATION_SUBZONE_X_{i}', f'DESTINATION_SUBZONE_Y_{i}',
             f'TRIP_STARTTIME_{i}', f'TRIP_ENDTIME_{i}']

    df_sim = df_fus[df_fus['TRIP_CNT'] == i].drop(columns=['TRIP_CNT'])
    rename_columns(df_sim, i)
    df_sim.rename(columns={f'Prob_{i}': 'Prob'}, inplace=True)
    df_sim = df_sim[att_X + att_Z + ['Prob']].copy()
    reduce_mem_usage(df_sim)

    idx = _prob_sample_indices(df_sim['Prob'].to_numpy(), N)
    if len(idx) == 0:
        return df_sim.iloc[0:0][att_X + att_Z]

    out = df_sim.iloc[idx][att_X + att_Z].reset_index(drop=True)
    return reduce_mem_usage(out)


def simulate_tours(df_tours: pd.DataFrame, order: int, N: int) -> pd.DataFrame:
    att_X = ['ID', 'AGE', 'GENDER', 'INCOME', 'TRIP_MAX']
    att_Z = []
    for i in range(1, order + 1):
        att_Z.extend([f'TRIP_PURPOSE_{i}', f'TRAVEL_MODE_{i}',
                      f'ORIGIN_SUBZONE_{i}', f'ORIGIN_SUBZONE_X_{i}', f'ORIGIN_SUBZONE_Y_{i}',
                      f'DESTINATION_SUBZONE_{i}', f'DESTINATION_SUBZONE_X_{i}', f'DESTINATION_SUBZONE_Y_{i}',
                      f'TRIP_STARTTIME_{i}', f'TRIP_ENDTIME_{i}'])

    probs = df_tours['Prob'].to_numpy(dtype=np.float64)
    probs = probs / probs.sum() if probs.sum() > 0 else probs
    idx = _prob_sample_indices(probs, N)
    if len(idx) == 0:
        return df_tours.iloc[0:0][att_X + att_Z]

    out = df_tours.iloc[idx][att_X + att_Z].reset_index(drop=True)
    return reduce_mem_usage(out)


def next_start_times(df_hts_: pd.DataFrame) -> pd.DataFrame:
    """
    Compute P(TRIP_STARTTIME | TRIP_ENDTIME, TRIP_PURPOSE) from HTS activity records.
    """
    t = df_hts_[['ACTIVITY_STARTTIME', 'TRIP_PURPOSE', 'ACTIVITY_DURATION']].copy()
    t['TRIP_STARTTIME'] = t['ACTIVITY_STARTTIME'] + t['ACTIVITY_DURATION']
    t = t.rename(columns={'ACTIVITY_STARTTIME': 'TRIP_ENDTIME'}).drop(columns=['ACTIVITY_DURATION'])

    t['Numbers'] = 1.0
    grp_keys = ['TRIP_ENDTIME', 'TRIP_PURPOSE', 'TRIP_STARTTIME']
    agg = t.groupby(grp_keys, observed=True)['Numbers'].sum().astype(np.float64).reset_index()
    agg['Prob'] = agg['Numbers'] / agg['Numbers'].sum()
    agg.drop(columns=['Numbers'], inplace=True)

    base_keys = ['TRIP_ENDTIME', 'TRIP_PURPOSE']
    agg['Prob_base'] = agg.groupby(base_keys, observed=True)['Prob'].transform('sum')
    agg['Prob_cond'] = (agg['Prob'] / agg['Prob_base']).astype(np.float32)
    agg = agg.drop(columns=['Prob_base', 'Prob'])

    reduce_mem_usage(agg)
    return agg.dropna()


def _prep_ith_trips(df_fus: pd.DataFrame, i: int) -> pd.DataFrame:
    """
    Compute P(dest, endtime, mode, purpose | age, gender, income, trip_max, origin, starttime)
    from the fused OD matrix. Pre-building this table once per trip index avoids
    recomputing it inside the chunk loop.
    """
    cols_keep = ['AGE', 'GENDER', 'INCOME', 'TRIP_MAX', 'TRIP_CNT',
                 'TRIP_PURPOSE', 'TRAVEL_MODE',
                 'ORIGIN_SUBZONE', 'ORIGIN_SUBZONE_X', 'ORIGIN_SUBZONE_Y',
                 'DESTINATION_SUBZONE', 'DESTINATION_SUBZONE_X', 'DESTINATION_SUBZONE_Y',
                 'TRIP_STARTTIME', 'TRIP_ENDTIME', 'Prob_XYZ_fus']
    ith = df_fus.loc[df_fus['TRIP_CNT'] == i, cols_keep].copy()
    ith.drop(columns=['TRIP_CNT'], inplace=True)
    rename_columns(ith, i)

    den_keys = ['AGE', 'GENDER', 'INCOME', 'TRIP_MAX',
                f'ORIGIN_SUBZONE_{i}', f'TRIP_STARTTIME_{i}']
    ith['Prob_den'] = ith.groupby(den_keys, observed=True)[f'Prob_{i}'].transform('sum')
    # Guard against zero denominator
    ith[f'Prob_{i}'] = np.where(
        ith['Prob_den'] > 0,
        (ith[f'Prob_{i}'] / ith['Prob_den']).astype(np.float32),
        0.0
    )
    ith = ith[ith[f'Prob_{i}'] > 0].copy()
    ith.drop(columns=['Prob_den'], inplace=True)

    return reduce_mem_usage(ith)


# ============================================================
# Fallback: OD-preserving draw when the strict join finds no match
# ============================================================

def _fallback_trip_sample(df_unmatched: pd.DataFrame,
                           df_fus: pd.DataFrame,
                           i: int) -> pd.DataFrame:
    """
    For individuals that could not be matched via the strict
    (AGE, GENDER, INCOME, TRIP_MAX, ORIGIN, STARTTIME) join, draw
    trip i from the OD matrix using a cascading relaxation strategy
    that preserves the OD distribution at each level.

    Relaxation ladder
    -----------------
    Level 1  Full demographic + spatial match, time relaxed
             Condition on (AGE, GENDER, INCOME, TRIP_MAX, ORIGIN_SUBZONE)
    Level 2  Spatial match only, demographics relaxed
             Condition on (ORIGIN_SUBZONE) only
    Level 3  Unconditional draw from full OD matrix for this trip index
             Last resort — preserves global OD marginals

    In all levels the draw is weighted by Prob_XYZ_fus so the OD
    pair distribution from the source matrix is respected.

    Parameters
    ----------
    df_unmatched : rows from df_tours_ that found no match in concat_trips
    df_fus       : original fused OD table (pre-rename, with Prob_XYZ_fus)
    i            : current trip index

    Returns
    -------
    DataFrame with the same schema as df_unmatched plus trip-i columns appended
    """
    trip_cols = [
        f'TRIP_PURPOSE_{i}', f'TRAVEL_MODE_{i}',
        f'ORIGIN_SUBZONE_{i}', f'ORIGIN_SUBZONE_X_{i}', f'ORIGIN_SUBZONE_Y_{i}',
        f'DESTINATION_SUBZONE_{i}', f'DESTINATION_SUBZONE_X_{i}', f'DESTINATION_SUBZONE_Y_{i}',
        f'TRIP_STARTTIME_{i}', f'TRIP_ENDTIME_{i}',
    ]

    # Prepare a renamed view of df_fus for trip i (do NOT rename in-place)
    fus_i = df_fus.loc[df_fus['TRIP_CNT'] == i].copy()
    fus_i.drop(columns=['TRIP_CNT'], inplace=True)
    rename_columns(fus_i, i)
    fus_i.rename(columns={f'Prob_{i}': '_prob'}, inplace=True)
    fus_i['_prob'] = fus_i['_prob'].astype(np.float64)

    origin_col = f'ORIGIN_SUBZONE_{i}'
    prev_dest  = f'DESTINATION_SUBZONE_{i - 1}'
    prev_age   = 'AGE'
    prev_gnd   = 'GENDER'
    prev_inc   = 'INCOME'
    prev_tmax  = 'TRIP_MAX'

    rows_out = []
    fallback_levels = {1: 0, 2: 0, 3: 0}

    for _, row in df_unmatched.iterrows():
        prev_zone = row[prev_dest]
        age       = row[prev_age]
        gender    = row[prev_gnd]
        income    = row[prev_inc]
        tmax      = row[prev_tmax]

        # --- Level 1: full demographics + origin, relax time ---------------
        pool = fus_i[
            (fus_i[origin_col]  == prev_zone) &
            (fus_i[prev_age]    == age)        &
            (fus_i[prev_gnd]    == gender)     &
            (fus_i[prev_inc]    == income)     &
            (fus_i[prev_tmax]   == tmax)
        ]

        if len(pool) == 0:
            # --- Level 2: spatial only, relax demographics -----------------
            pool = fus_i[fus_i[origin_col] == prev_zone]

        if len(pool) == 0:
            # --- Level 3: unconditional draw from full OD matrix -----------
            pool = fus_i
            fallback_levels[3] += 1
        elif pool is fus_i:
            fallback_levels[2] += 1
        else:
            fallback_levels[1] += 1

        pool = pool.copy()
        pool['_prob'] = pool['_prob'] / pool['_prob'].sum()
        idx = np.random.choice(len(pool), p=pool['_prob'].to_numpy())
        drawn = pool.iloc[idx][trip_cols].to_dict()

        combined = row.to_dict()
        combined.update(drawn)
        rows_out.append(combined)

    if rows_out:
        df_filled = pd.DataFrame(rows_out)
    else:
        df_filled = df_unmatched.copy()
        for c in trip_cols:
            df_filled[c] = np.nan

    return df_filled, fallback_levels


# ============================================================
# concat_trips — strict join + OD-preserving fallback
# ============================================================

def concat_trips(df_tours: pd.DataFrame,
                 df_ith_trips: pd.DataFrame,
                 df_next_start_times: pd.DataFrame,
                 i: int,
                 df_fus: pd.DataFrame = None):
    """
    Attach trip i to partial tours.

    Primary path  : strict join on (AGE, GENDER, INCOME, TRIP_MAX, ORIGIN, STARTTIME)
                    after conditioning on P(next_start | prev_end, prev_purpose).
    Fallback path : OD-preserving draw via _fallback_trip_sample() for any
                    individual that found no match in the primary join.

    Returns
    -------
    df_tours_complete   : rows where TRIP_MAX == i
    df_tours_incomplete : rows where TRIP_MAX >  i
    stats               : dict with matched / fallback counts per level
    """
    tours = df_tours.copy()
    tours['PRE_SUBZONE'] = tours[f'DESTINATION_SUBZONE_{i-1}']
    tours['PRE_PURPOSE'] = tours[f'TRIP_PURPOSE_{i-1}']
    tours['PRE_ENDTIME'] = tours[f'TRIP_ENDTIME_{i-1}']

    nst = df_next_start_times.rename(columns={
        'TRIP_PURPOSE':  'PRE_PURPOSE',
        'TRIP_ENDTIME':  'PRE_ENDTIME',
        'TRIP_STARTTIME': 'PRE_STARTTIME',
    }, errors='ignore')

    ith = df_ith_trips.copy()
    ith['PRE_SUBZONE']   = ith[f'ORIGIN_SUBZONE_{i}']
    ith['PRE_STARTTIME'] = ith[f'TRIP_STARTTIME_{i}']

    # ── Primary join ──────────────────────────────────────────────────────
    tours_joined = tours.merge(
        nst[['PRE_PURPOSE', 'PRE_ENDTIME', 'PRE_STARTTIME', 'Prob_cond']],
        on=['PRE_PURPOSE', 'PRE_ENDTIME'],
        how='left',
        copy=False,
    )

    join_keys = ['AGE', 'GENDER', 'INCOME', 'TRIP_MAX', 'PRE_SUBZONE', 'PRE_STARTTIME']
    tours_joined = tours_joined.merge(
        ith[join_keys + [
            f'TRIP_PURPOSE_{i}', f'TRAVEL_MODE_{i}',
            f'ORIGIN_SUBZONE_{i}', f'ORIGIN_SUBZONE_X_{i}', f'ORIGIN_SUBZONE_Y_{i}',
            f'DESTINATION_SUBZONE_{i}', f'DESTINATION_SUBZONE_X_{i}', f'DESTINATION_SUBZONE_Y_{i}',
            f'TRIP_STARTTIME_{i}', f'TRIP_ENDTIME_{i}', f'Prob_{i}']],
        on=join_keys,
        how='inner',
        copy=False,
    )

    matched_ids  = set(tours_joined['ID'].unique()) if len(tours_joined) > 0 else set()
    all_ids      = set(df_tours['ID'].unique())
    unmatched_ids = all_ids - matched_ids

    stats = {'matched': len(matched_ids), 'fallback_l1': 0,
             'fallback_l2': 0, 'fallback_l3': 0}

    # ── Fallback for unmatched individuals ───────────────────────────────
    if unmatched_ids and df_fus is not None:
        df_unmatched = tours[tours['ID'].isin(unmatched_ids)].copy()
        df_unmatched = df_unmatched.drop(
            columns=['PRE_SUBZONE', 'PRE_PURPOSE', 'PRE_ENDTIME'], errors='ignore'
        )

        df_filled, fb_levels = _fallback_trip_sample(df_unmatched, df_fus, i)
        stats['fallback_l1'] += fb_levels[1]
        stats['fallback_l2'] += fb_levels[2]
        stats['fallback_l3'] += fb_levels[3]

        # Assign nominal probability for filled rows so Prob column exists
        if len(tours_joined) > 0 and f'Prob_{i}' in tours_joined.columns:
            nominal_prob = float(tours_joined[f'Prob_{i}'].median())
        else:
            nominal_prob = 1e-6
        df_filled['Prob'] = nominal_prob

        # Finalise primary matched rows
        tours_joined['Prob'] = (
            tours_joined[f'Prob_{i}'] * tours_joined['Prob_cond']
        ).astype(np.float32)
        tours_joined.drop(
            columns=[f'Prob_{i}', 'Prob_cond',
                     'PRE_SUBZONE', 'PRE_PURPOSE', 'PRE_ENDTIME', 'PRE_STARTTIME'],
            inplace=True, errors='ignore'
        )

        # Align columns before concat (filled rows won't have Prob_cond etc.)
        shared_cols = [c for c in tours_joined.columns if c in df_filled.columns]
        tours_all = pd.concat(
            [tours_joined[shared_cols], df_filled[shared_cols]],
            ignore_index=True
        )
    else:
        # Normal path — no fallback needed
        tours_joined['Prob'] = (
            tours_joined[f'Prob_{i}'] * tours_joined['Prob_cond']
        ).astype(np.float32)
        tours_joined.drop(
            columns=[f'Prob_{i}', 'Prob_cond',
                     'PRE_SUBZONE', 'PRE_PURPOSE', 'PRE_ENDTIME', 'PRE_STARTTIME'],
            inplace=True, errors='ignore'
        )
        tours_all = tours_joined

    # ── Split complete vs incomplete ──────────────────────────────────────
    mask_done   = tours_all['TRIP_MAX'] == i
    mask_home   = mask_done & (tours_all[f'TRIP_PURPOSE_{i}'] == 0)

    complete_home = tours_all.loc[mask_home].copy()
    complete_home[f'DESTINATION_SUBZONE_{i}']   = complete_home['ORIGIN_SUBZONE_1']
    complete_home[f'DESTINATION_SUBZONE_X_{i}'] = complete_home['ORIGIN_SUBZONE_X_1']
    complete_home[f'DESTINATION_SUBZONE_Y_{i}'] = complete_home['ORIGIN_SUBZONE_Y_1']

    complete_nohome = tours_all.loc[mask_done & ~mask_home]
    df_tours_complete   = pd.concat([complete_home, complete_nohome], ignore_index=True)
    df_tours_incomplete = tours_all.loc[~mask_done].copy()

    return (reduce_mem_usage(df_tours_complete),
            reduce_mem_usage(df_tours_incomplete),
            stats)


# ============================================================
# Main generation function
# ============================================================

def generate_tours(df_fus: pd.DataFrame,
                   df_next_start_times: pd.DataFrame,
                   N: int,
                   verbose: bool = True) -> pd.DataFrame:
    """
    Generate N synthetic tour records.

    Changes vs original
    -------------------
    * concat_trips now returns (complete, incomplete, stats) — call sites updated.
    * Unmatched individuals are filled via OD-preserving fallback rather than dropped.
    * _prep_ith_trips is pre-computed once before the chunk loop (not rebuilt per chunk).
    * verbose flag prints per-trip match / fallback statistics.

    Notes on scale
    --------------
    scale > 1 serves two distinct purposes and should be kept:
      1. Seed diversity  : scale*N individuals are seeded so the chunk pool
                           starts with enough variety across cohorts.
      2. Pool diversity  : simulate_tours re-samples scale*N rows after each
                           inner join step so the incomplete pool does not
                           collapse to a handful of rows and produce near-clone tours.
    The fallback fixes zero-row failures; scale fixes low-diversity. They are
    complementary and both are needed.
    """
    scale = 2 #5

    # ── Seed tours from TRIP_CNT == 1 ────────────────────────────────────
    df_tours_seed = df_fus[df_fus['TRIP_CNT'] == 1]
    df_tours = simulate_trips(df_tours_seed, 1, int(scale * N)).reset_index(drop=True)
    df_tours = df_tours.reset_index()
    df_tours['ID'] = df_tours['index'].astype(np.int64)
    df_tours.drop(columns=['index'], inplace=True)

    # Trip-max proportions per (AGE, GENDER, INCOME, TRIP_MAX)
    df_tm = df_tours[['AGE', 'GENDER', 'INCOME', 'TRIP_MAX']].copy()
    df_tm['Numbers'] = 1.0
    df_tm = (df_tm.groupby(['AGE', 'GENDER', 'INCOME', 'TRIP_MAX'], observed=True)
             ['Numbers'].sum().reset_index())
    df_tm['Pop'] = (df_tm['Numbers'] / df_tm['Numbers'].sum()).astype(np.float32)
    df_tm.drop(columns=['Numbers'], inplace=True)
    trip_max = {(r.AGE, r.GENDER, r.INCOME, r.TRIP_MAX): r.Pop
                for r in df_tm.itertuples(index=False)}

    # ── Pre-build conditional trip tables once (avoids re-compute per chunk) ──
    if verbose:
        print("Pre-building conditional trip tables for trips 2 …", TRIP_MAX_)
    cond_tables = {i: _prep_ith_trips(df_fus, i) for i in range(2, TRIP_MAX_ + 1)}
    df_next_start_times = reduce_mem_usage(df_next_start_times.copy())

    # ── Chunk processing ──────────────────────────────────────────────────
    chunk_size  = 10_000
    Z           = (len(df_tours) + chunk_size - 1) // chunk_size
    out_chunks  = []
    total_stats = {'matched': 0, 'fallback_l1': 0, 'fallback_l2': 0, 'fallback_l3': 0}

    if verbose:
        print(f"Processing {Z} chunks of ≤{chunk_size} rows …")

    for z in range(Z):
        df_tours_ = df_tours.iloc[z * chunk_size:(z + 1) * chunk_size].copy()
        df_tours_['Prob'] = 1.0
        df_result = []

        for i in range(2, TRIP_MAX_ + 1):
            df_ith_trips = cond_tables[i]

            df_tours_complete, df_tours_incomplete, stats = concat_trips(
                df_tours_, df_ith_trips, df_next_start_times, i, df_fus=df_fus
            )

            for k in total_stats:
                total_stats[k] += stats.get(k, 0)

            if verbose:
                print(f"  chunk {z:3d} | trip {i}: "
                      f"matched={stats['matched']}, "
                      f"fb_l1={stats['fallback_l1']}, "
                      f"fb_l2={stats['fallback_l2']}, "
                      f"fb_l3={stats['fallback_l3']}, "
                      f"complete={len(df_tours_complete)}, "
                      f"incomplete={len(df_tours_incomplete)}")

            if len(df_tours_complete) > 0:
                df_result.append(
                    simulate_tours(df_tours_complete, i, int(scale * N))
                )

            if len(df_tours_incomplete) > 0:
                df_tours_ = simulate_tours(df_tours_incomplete, i, int(scale * N))
            else:
                break

        if df_result:
            res = pd.concat(df_result, ignore_index=True)
            res = res.drop_duplicates(subset=['ID'], keep='first')
            out_chunks.append(res)

        if verbose:
            print(f"  chunk {z} done  ({len(out_chunks[-1]) if out_chunks else 0} unique tours)")

    if not out_chunks:
        return pd.DataFrame(columns=['ID', 'AGE', 'GENDER', 'INCOME', 'TRIP_MAX'])

    df_result_ = pd.concat(out_chunks, ignore_index=True)
    df_result_ = df_result_.drop_duplicates(subset=['ID'], keep='first')

    # ── Final cohort sampling to match TRIP_MAX proportions ──────────────
    final_chunks = []
    for (age, gender, income, tmax), pop in trip_max.items():
        df_tmp = df_result_[
            (df_result_['TRIP_MAX'] == tmax)   &
            (df_result_['AGE']      == age)    &
            (df_result_['INCOME']   == income) &
            (df_result_['GENDER']   == gender)
        ]
        target = int(N * pop)
        n2 = len(df_tmp)
        if n2 == 0:
            if verbose:
                print(f"  WARNING: no tours for cohort "
                      f"(AGE={age}, GENDER={gender}, INCOME={income}, TRIP_MAX={tmax})")
            continue
        if n2 > target and target > 0:
            df_tmp = df_tmp.sample(n=target, replace=False, random_state=None)
        final_chunks.append(df_tmp)

    df_result_final = (pd.concat(final_chunks, ignore_index=True)
                       if final_chunks else df_result_.iloc[0:0])

    # Assign globally unique sequential ID
    df_result_final = df_result_final.reset_index(drop=True)
    df_result_final['ID'] = df_result_final.index

    if verbose:
        total_fb = total_stats['fallback_l1'] + total_stats['fallback_l2'] + total_stats['fallback_l3']
        total_all = total_stats['matched'] + total_fb
        pct = 100 * total_fb / max(total_all, 1)
        print(f"\n── Generation complete ──────────────────────────────────")
        print(f"   Output rows : {len(df_result_final)}")
        print(f"   Matched     : {total_stats['matched']}")
        print(f"   Fallback L1 : {total_stats['fallback_l1']}  "
              f"(demo+spatial, time relaxed)")
        print(f"   Fallback L2 : {total_stats['fallback_l2']}  "
              f"(spatial only)")
        print(f"   Fallback L3 : {total_stats['fallback_l3']}  "
              f"(unconditional OD draw)")
        print(f"   Total fallback: {total_fb} / {total_all}  ({pct:.1f}%)")

    return reduce_mem_usage(df_result_final)


# ============================================================
# Entry point
# ============================================================

if __name__ == '__main__':
    ff = 0.3

    df_hts = pd.read_csv(path_data + f'val_data_hts_trip_{ff}.csv')

    start_time = time.time()

    df_fus = pd.read_csv(path_result + f'val_sim_trip_proposed_{ff}.csv')
    df_fus = reduce_mem_usage(df_fus)
    print('df_fus length:', len(df_fus))

    df_next_start_times = next_start_times(df_hts)

    NN = 50_000
    df_tours = generate_tours(df_fus, df_next_start_times, NN, verbose=True)
    print(df_tours.info())
    df_tours.to_csv(path_result + f'val_sim_tour_proposed_{ff}.csv', index=False)

    print('Time taken: %s seconds' % (time.time() - start_time))
    print('Done!')

df_fus length: 3735279
Pre-building conditional trip tables for trips 2 … 7
Processing 10 chunks of ≤10000 rows …
  chunk   0 | trip 2: matched=9758, fb_l1=242, fb_l2=0, fb_l3=0, complete=860184, incomplete=104613
  chunk   0 | trip 3: matched=2837, fb_l1=908, fb_l2=0, fb_l3=0, complete=2083041, incomplete=974830
  chunk   0 | trip 4: matched=1352, fb_l1=196, fb_l2=0, fb_l3=0, complete=1860042, incomplete=399980
  chunk   0 | trip 5: matched=588, fb_l1=336, fb_l2=0, fb_l3=0, complete=991549, incomplete=299480
  chunk   0 | trip 6: matched=247, fb_l1=641, fb_l2=0, fb_l3=0, complete=881516, incomplete=46668
  chunk   0 | trip 7: matched=74, fb_l1=46, fb_l2=0, fb_l3=0, complete=515702, incomplete=0
  chunk 0 done  (9660 unique tours)
  chunk   1 | trip 2: matched=9793, fb_l1=207, fb_l2=0, fb_l3=0, complete=873222, incomplete=99037
  chunk   1 | trip 3: matched=2731, fb_l1=853, fb_l2=0, fb_l3=0, complete=2057083, incomplete=1000296
  chunk   1 | trip 4: matched=1353, fb_l1=243, fb_l2=0, fb

In [ ]:
#With income
import pandas as pd
import numpy as np
from sklearn import preprocessing
from sklearn.cluster import Birch
from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from scipy.special import rel_entr, kl_div
import math
import time

import warnings
warnings.filterwarnings('ignore')

TRIP_MAX_ = 7

# --------------- Memory helpers ---------------

def _downcast_int(s: pd.Series):
    if s.isnull().any():
        return s
    i32 = s.astype(np.int64)
    if i32.min() >= np.iinfo(np.int8).min and i32.max() <= np.iinfo(np.int8).max:
        return i32.astype(np.int8)
    if i32.min() >= np.iinfo(np.int16).min and i32.max() <= np.iinfo(np.int16).max:
        return i32.astype(np.int16)
    if i32.min() >= np.iinfo(np.int32).min and i32.max() <= np.iinfo(np.int32).max:
        return i32.astype(np.int32)
    return i32

def _downcast_float(s: pd.Series):
    return s.astype(np.float32)

def reduce_mem_usage(df: pd.DataFrame, use_cats: bool = True):
    for col in df.columns:
        if pd.api.types.is_integer_dtype(df[col]):
            df[col] = _downcast_int(df[col])
        elif pd.api.types.is_float_dtype(df[col]):
            df[col] = _downcast_float(df[col])
        elif use_cats and (df[col].nunique(dropna=False) / max(1, len(df[col])) < 0.5):
            df[col] = df[col].astype('category')
    return df

def as_codes(df: pd.DataFrame, cols):
    """Convert columns to categorical codes (int32) consistently, preserving mapping via .cat.categories."""
    for c in cols:
        if not pd.api.types.is_categorical_dtype(df[c]):
            df[c] = df[c].astype('category')
        df[c] = df[c].cat.codes.astype(np.int32)
    return df

# --------------- Math helpers ---------------

def js_div(p, q):
    h = 0.5*np.add(p, q)
    return 0.5*rel_entr(p, h) + 0.5*rel_entr(q, h)

# --------------- Column ops (zero-copy where possible) ---------------

def _renmap(i: int):
    return {
        'TRIP_PURPOSE': f'TRIP_PURPOSE_{i}',
        'TRAVEL_MODE': f'TRAVEL_MODE_{i}',
        'ORIGIN_SUBZONE': f'ORIGIN_SUBZONE_{i}',
        'DESTINATION_SUBZONE': f'DESTINATION_SUBZONE_{i}',
        'TRIP_STARTTIME': f'TRIP_STARTTIME_{i}',
        'TRIP_ENDTIME': f'TRIP_ENDTIME_{i}',
        'ORIGIN_SUBZONE_X': f'ORIGIN_SUBZONE_X_{i}',
        'ORIGIN_SUBZONE_Y': f'ORIGIN_SUBZONE_Y_{i}',
        'DESTINATION_SUBZONE_X': f'DESTINATION_SUBZONE_X_{i}',
        'DESTINATION_SUBZONE_Y': f'DESTINATION_SUBZONE_Y_{i}',
        'Prob_XYZ_fus': f'Prob_{i}',
    }

def rename_columns(df: pd.DataFrame, i: int):
    df.rename(columns=_renmap(i), inplace=True)

# --------------- Sampling utils (no-merge) ---------------

def _prob_sample_indices(prob: np.ndarray, N: int) -> np.ndarray:
    prob = np.asarray(prob, dtype=np.float64)
    prob_sum = prob.sum()
    if prob_sum <= 0:
        # Avoid divide-by-zero; return empty
        return np.array([], dtype=np.int64)
    prob = prob / prob_sum
    # Use cumulative trick to avoid materializing large intermediate df
    return np.random.choice(len(prob), size=N, replace=True, p=prob)

def _select_columns(df: pd.DataFrame, cols):
    # Avoid copying where possible; pandas will create a view if safe.
    return df.loc[:, cols]

# --------------- Core pipeline ---------------

def simulate_trips(df_fus: pd.DataFrame, i: int, N: int) -> pd.DataFrame:
    att_X = ['AGE', 'GENDER', 'INCOME', 'TRIP_MAX']
    att_Z = [f'TRIP_PURPOSE_{i}', f'TRAVEL_MODE_{i}',
             f'ORIGIN_SUBZONE_{i}', f'ORIGIN_SUBZONE_X_{i}', f'ORIGIN_SUBZONE_Y_{i}',
             f'DESTINATION_SUBZONE_{i}', f'DESTINATION_SUBZONE_X_{i}', f'DESTINATION_SUBZONE_Y_{i}',
             f'TRIP_STARTTIME_{i}', f'TRIP_ENDTIME_{i}']

    df_sim = df_fus[df_fus['TRIP_CNT'] == i].drop(columns=['TRIP_CNT'])
    rename_columns(df_sim, i)
    df_sim.rename(columns={f'Prob_{i}': 'Prob'}, inplace=True)

    df_sim = _select_columns(df_sim, att_X + att_Z + ['Prob'])
    reduce_mem_usage(df_sim)

    idx = _prob_sample_indices(df_sim['Prob'].to_numpy(), N)
    if len(idx) == 0:
        return df_sim.iloc[0:0][att_X + att_Z]

    # Use iloc/take instead of merging on an artificial index
    out = df_sim.iloc[idx][att_X + att_Z].reset_index(drop=True)
    return reduce_mem_usage(out)

def simulate_tours(df_tours: pd.DataFrame, order: int, N: int) -> pd.DataFrame:
    att_X = ['ID', 'AGE', 'GENDER', 'INCOME', 'TRIP_MAX']
    att_Z = []
    for i in range(1, order + 1):
        att_Z.extend([f'TRIP_PURPOSE_{i}', f'TRAVEL_MODE_{i}',
                      f'ORIGIN_SUBZONE_{i}', f'ORIGIN_SUBZONE_X_{i}', f'ORIGIN_SUBZONE_Y_{i}',
                      f'DESTINATION_SUBZONE_{i}', f'DESTINATION_SUBZONE_X_{i}', f'DESTINATION_SUBZONE_Y_{i}',
                      f'TRIP_STARTTIME_{i}', f'TRIP_ENDTIME_{i}'])

    # Prob normalization without extra copy
    probs = df_tours['Prob'].to_numpy(dtype=np.float64)
    probs = probs / probs.sum() if probs.sum() > 0 else probs
    idx = _prob_sample_indices(probs, N)
    if len(idx) == 0:
        return df_tours.iloc[0:0][att_X + att_Z]

    out = df_tours.iloc[idx][att_X + att_Z].reset_index(drop=True)
    return reduce_mem_usage(out)

def next_start_times(df_hts_: pd.DataFrame) -> pd.DataFrame:
    # Build minimal table, count combos, then convert to conditional P(start_next | end_prev, purpose_prev)
    t = df_hts_[['ACTIVITY_STARTTIME', 'TRIP_PURPOSE', 'ACTIVITY_DURATION']].copy()
    t['TRIP_STARTTIME'] = t['ACTIVITY_STARTTIME'] + t['ACTIVITY_DURATION']
    t = t.rename(columns={'ACTIVITY_STARTTIME': 'TRIP_ENDTIME'}).drop(columns=['ACTIVITY_DURATION'])

    # Use value_counts -> Prob -> conditional
    t['Numbers'] = 1.0
    grp_keys = ['TRIP_ENDTIME', 'TRIP_PURPOSE', 'TRIP_STARTTIME']
    agg = t.groupby(grp_keys, observed=True)['Numbers'].sum().astype(np.float64).reset_index()
    agg['Prob'] = agg['Numbers'] / agg['Numbers'].sum()
    agg.drop(columns=['Numbers'], inplace=True)

    base_keys = ['TRIP_ENDTIME', 'TRIP_PURPOSE']
    agg['Prob_base'] = agg.groupby(base_keys, observed=True)['Prob'].transform('sum')
    agg['Prob_cond'] = (agg['Prob'] / agg['Prob_base']).astype(np.float32)
    agg = agg.drop(columns=['Prob_base', 'Prob'])

    # Downcast & codes for join keys
    reduce_mem_usage(agg)
    return agg.dropna()

def _prep_ith_trips(df_fus: pd.DataFrame, i: int) -> pd.DataFrame:
    # Slice, rename, then build conditional Prob over (X, Y, O, S)
    cols_keep = ['AGE', 'GENDER', 'INCOME', 'TRIP_MAX', 'TRIP_CNT',
                 'TRIP_PURPOSE', 'TRAVEL_MODE',
                 'ORIGIN_SUBZONE', 'ORIGIN_SUBZONE_X', 'ORIGIN_SUBZONE_Y',
                 'DESTINATION_SUBZONE', 'DESTINATION_SUBZONE_X', 'DESTINATION_SUBZONE_Y',
                 'TRIP_STARTTIME', 'TRIP_ENDTIME', 'Prob_XYZ_fus']
    ith = df_fus.loc[df_fus['TRIP_CNT'] == i, cols_keep].copy()
    ith.drop(columns=['TRIP_CNT'], inplace=True)
    rename_columns(ith, i)

    # Group denominator: sum Prob over X,Y,O,S
    den_keys = ['AGE', 'GENDER', 'INCOME', 'TRIP_MAX', f'ORIGIN_SUBZONE_{i}', f'TRIP_STARTTIME_{i}']
    ith['Prob_den'] = ith.groupby(den_keys, observed=True)[f'Prob_{i}'].transform('sum')
    ith[f'Prob_{i}'] = (ith[f'Prob_{i}'] / ith['Prob_den']).astype(np.float32)
    ith.drop(columns=['Prob_den'], inplace=True)

    return reduce_mem_usage(ith)

def concat_trips(df_tours: pd.DataFrame,
                 df_ith_trips: pd.DataFrame,
                 df_next_start_times: pd.DataFrame,
                 i: int):
    # Build minimal PRE_* keys to join
    tours = df_tours.copy()
    tours['PRE_SUBZONE'] = tours[f'DESTINATION_SUBZONE_{i-1}']
    tours['PRE_PURPOSE'] = tours[f'TRIP_PURPOSE_{i-1}']
    tours['PRE_ENDTIME'] = tours[f'TRIP_ENDTIME_{i-1}']

    nst = df_next_start_times.rename(columns={
        'TRIP_PURPOSE': 'PRE_PURPOSE',
        'TRIP_ENDTIME': 'PRE_ENDTIME',
        'TRIP_STARTTIME': 'PRE_STARTTIME'
    }, errors='ignore')

    ith = df_ith_trips.copy()
    ith['PRE_SUBZONE'] = ith[f'ORIGIN_SUBZONE_{i}']
    ith['PRE_STARTTIME'] = ith[f'TRIP_STARTTIME_{i}']

    # Left-join tours with cond P(next start | prev end, purpose) to get Prob_cond
    tours = tours.merge(
        nst[[#'AGE', 'GENDER', 'TRIP_MAX',
            'PRE_PURPOSE', 'PRE_ENDTIME', 'PRE_STARTTIME', 'Prob_cond']],
        on=[#'AGE', 'GENDER', 'TRIP_MAX',
            'PRE_PURPOSE', 'PRE_ENDTIME'],
        how='left',
        copy=False
    )

    # Keyed join with i-th trips on (X, Y, PRE_SUBZONE, PRE_STARTTIME)
    join_keys = ['AGE', 'GENDER', 'INCOME', 'TRIP_MAX', 'PRE_SUBZONE', 'PRE_STARTTIME']
    tours = tours.merge(
        ith[join_keys + [f'TRIP_PURPOSE_{i}', f'TRAVEL_MODE_{i}',
                         f'ORIGIN_SUBZONE_{i}', f'ORIGIN_SUBZONE_X_{i}', f'ORIGIN_SUBZONE_Y_{i}',
                         f'DESTINATION_SUBZONE_{i}', f'DESTINATION_SUBZONE_X_{i}', f'DESTINATION_SUBZONE_Y_{i}',
                         f'TRIP_STARTTIME_{i}', f'TRIP_ENDTIME_{i}', f'Prob_{i}']],
        on=join_keys,
        how='inner',
        copy=False
    )

    # Combine probabilities
    tours['Prob'] = (tours[f'Prob_{i}'] * tours['Prob_cond']).astype(np.float32)
    tours.drop(columns=[f'Prob_{i}', 'Prob_cond', 'PRE_SUBZONE', 'PRE_PURPOSE', 'PRE_ENDTIME', 'PRE_STARTTIME'], inplace=True)

    # Split complete vs incomplete
    mask_done = (tours['TRIP_MAX'] == i)
    mask_home = mask_done & (tours[f'TRIP_PURPOSE_{i}'] == 0)

    complete_home = tours.loc[mask_home].copy()
    # Enforce return to home
    complete_home[f'DESTINATION_SUBZONE_{i}']   = complete_home['ORIGIN_SUBZONE_1']
    complete_home[f'DESTINATION_SUBZONE_X_{i}'] = complete_home['ORIGIN_SUBZONE_X_1']
    complete_home[f'DESTINATION_SUBZONE_Y_{i}'] = complete_home['ORIGIN_SUBZONE_Y_1']

    complete_nohome = tours.loc[mask_done & ~mask_home]
    df_tours_complete = pd.concat([complete_home, complete_nohome], ignore_index=True)

    df_tours_incomplete = tours.loc[~mask_done]

    return reduce_mem_usage(df_tours_complete), reduce_mem_usage(df_tours_incomplete)

def generate_tours(df_fus: pd.DataFrame, df_next_start_times: pd.DataFrame, N: int) -> pd.DataFrame:
    scale = 5

    # Seed tours from TRIP_CNT==1
    df_tours_seed = df_fus[df_fus['TRIP_CNT'] == 1]
    df_tours = simulate_trips(df_tours_seed, 1, int(scale * N)).reset_index(drop=True)
    df_tours = df_tours.reset_index()
    df_tours['ID'] = df_tours['index'].astype(np.int64)
    df_tours.drop(columns=['index'], inplace=True)

    # Trip-max proportions per (AGE, GENDER, TRIP_MAX)
    df_tm = df_tours[['AGE', 'GENDER', 'INCOME', 'TRIP_MAX']].copy()
    df_tm['Numbers'] = 1.0
    df_tm = df_tm.groupby(['AGE', 'GENDER', 'INCOME', 'TRIP_MAX'], observed=True)['Numbers'].sum().reset_index()
    df_tm['Pop'] = (df_tm['Numbers'] / df_tm['Numbers'].sum()).astype(np.float32)
    df_tm.drop(columns=['Numbers'], inplace=True)
    trip_max = {(r.AGE, r.GENDER, r.INCOME, r.TRIP_MAX): r.Pop for r in df_tm.itertuples(index=False)}

    # Chunk process to cap peak RAM
    out_chunks = []
    chunk_size = 10000
    Z = (len(df_tours) + chunk_size - 1) // chunk_size
    print('Number of chunks ', Z)

    # Precompute next-start table as int codes to reduce join memory
    # (keep original too; merge handles both)
    df_next_start_times = reduce_mem_usage(df_next_start_times.copy())

    for z in range(Z):
        df_tours_ = df_tours.iloc[z*chunk_size:(z+1)*chunk_size].copy()
        df_tours_['Prob'] = 1.0  # starts uniform before conditionals
        df_result = []

        for i in range(2, TRIP_MAX_ + 1):
            df_ith_trips = _prep_ith_trips(df_fus, i)
            df_tours_complete, df_tours_incomplete = concat_trips(df_tours_, df_ith_trips, df_next_start_times, i)

            if len(df_tours_complete) > 0:
                df_result.append(simulate_tours(df_tours_complete, i, int(scale * N)))
            if len(df_tours_incomplete) > 0:
                # Advance partial tours for next step; reweight by sampling
                df_tours_ = simulate_tours(df_tours_incomplete, i, int(scale * N))
            else:
                break

        if len(df_result):
            res = pd.concat(df_result, ignore_index=True)
            # Keep first completion per ID
            res = res.drop_duplicates(subset=['ID'], keep='first')
            out_chunks.append(res)
        print('Finish chunk ', str(z))

    if not out_chunks:
        return pd.DataFrame(columns=['ID', 'AGE', 'GENDER', 'INCOME', 'TRIP_MAX'])

    df_result_ = pd.concat(out_chunks, ignore_index=True)
    df_result_ = df_result_.drop_duplicates(subset=['ID'], keep='first')

    # Final cohort-downsampling to match TRIP_MAX proportions
    final_chunks = []
    for (age, gender, income, tmax), pop in trip_max.items():
        df_tmp = df_result_[(df_result_['TRIP_MAX'] == tmax) &
                            (df_result_['AGE'] == age) &
                            (df_result_['INCOME'] == income) &
                            (df_result_['GENDER'] == gender)]
        target = int(N * pop)
        n2 = len(df_tmp)
        if n2 > target and target > 0:
            df_tmp = df_tmp.sample(n=target, replace=False, random_state=None)
        final_chunks.append(df_tmp)

    df_result_final = pd.concat(final_chunks, ignore_index=True) if final_chunks else df_result_.iloc[0:0]
    return reduce_mem_usage(df_result_final)

# ------------------------------------------------------------
# __main__: Example end-to-end run (adjust paths)
# ------------------------------------------------------------

ff = 0.1

df_hts = pd.read_csv(path_data + f'val_data_hts_trip_{ff}.csv')
#df_true_trip = pd.read_csv(path_data + 'data_sgp_hts_trip.csv')
#df_pcm = pd.read_csv(path_data + 'val_data_pcm_trip.csv')

start_time = time.time()

df_fus = pd.read_csv(path_result + f'val_sim_trip_proposed_{ff}.csv')
df_fus = reduce_mem_usage(df_fus)
print('df_fus lenght', len(df_fus))

df_next_start_times = next_start_times(df_hts)

#
NN = 50000 #len(df_true_trip[['ID']].drop_duplicates())
df_tours = generate_tours(df_fus, df_next_start_times, NN)
print(df_tours.info())
df_tours.to_csv(path_result + f'val_sim_tour_proposed_{ff}.csv', index = False)

print('Time take %s seconds' % (time.time() - start_time))
print('Done!')

df_fus lenght 3055883
Number of chunks  25
Finish chunk  0
Finish chunk  1
Finish chunk  2
Finish chunk  3
Finish chunk  4
Finish chunk  5
Finish chunk  6
Finish chunk  7
Finish chunk  8
Finish chunk  9
Finish chunk  10
Finish chunk  11
Finish chunk  12
Finish chunk  13
Finish chunk  14
Finish chunk  15
Finish chunk  16
Finish chunk  17
Finish chunk  18
Finish chunk  19
Finish chunk  20
Finish chunk  21
Finish chunk  22
Finish chunk  23
Finish chunk  24
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49789 entries, 0 to 49788
Data columns (total 75 columns):
 #   Column                   Non-Null Count  Dtype   
---  ------                   --------------  -----   
 0   ID                       49789 non-null  int32   
 1   AGE                      49789 non-null  int8    
 2   GENDER                   49789 non-null  int8    
 3   INCOME                   49789 non-null  int8    
 4   TRIP_MAX                 49789 non-null  int8    
 5   TRIP_PURPOSE_1           49789 non-null  in

#Validation: Stage 1 Finding K

In [ ]:
import os, math, gc, torch, numpy as np, pandas as pd
from dataclasses import replace

# ---------------------------
# Minimal, RAM-friendly train
# ---------------------------
def train_encoder_only_fusion_finding_K_MINRAM(
    df_hts: pd.DataFrame,
    df_pcm: pd.DataFrame,
    cfg: "TrainConfig",
    phi_segments_meta: Dict[str,int],
    use_mixed_precision: bool = True,   # safe no-op on CPU; helpful on CUDA
) -> Dict[str, float]:
    """
    RAM-efficient variant:
      - No 'pack' and no model object returned.
      - No per-epoch history stored.
      - Temporary tensors are freed aggressively.
    Returns tiny dict of scalars: {"K", "L_hts_final", "fusion_consistency_js"}.
    """
    device = cfg.device
    dtype = cfg.dtype

    # 1) Prepare data and loaders (do NOT keep extra returns around)
    n_xy, _, _, hts_loader_full, pcm_loader_full, _phi_dim = _prepare_xy_and_loaders(df_hts, df_pcm, cfg)

    # 2) Build encoder (embedding version)
    cardinals = (
        phi_segments_meta["n_dist"],
        phi_segments_meta["n_start"],
        phi_segments_meta["n_end"],
        phi_segments_meta["n_cluster"],
    )
    enc = EncoderEmbed(
        cardinals=cardinals,
        K=cfg.K,
        emb_dims=(16, 16, 16, 16),
        hidden=cfg.enc_hidden,
        num_layers=cfg.enc_layers,
        dropout=cfg.enc_dropout,
    ).to(device=device, dtype=torch.float32)  # keep params in fp32 for stability

    opt = torch.optim.AdamW(enc.parameters(), lr=cfg.lr, weight_decay=cfg.wd)
    logXY = math.log(max(n_xy, 2))

    # 3) Temperature & support policy
    tau = get_tau(1, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
    support_policy = _make_support_policy(enc, hts_loader_full, tau, cfg)

    # Initial non-parametric decoder; compute under no_grad and on CPU
    with torch.no_grad():
        P_xy_h = build_nonparam_decoder_xy_given_h(
            encoder=enc,
            hts_loader=hts_loader_full,
            n_xy=n_xy,
            K=cfg.K,
            device=device,
            dtype=dtype,
            progress=cfg.progress,
            tau=tau,
            support_policy=support_policy,
        )

    # 4) Train loop — no history stored
    scaler = torch.cuda.amp.GradScaler(enabled=(use_mixed_precision and torch.cuda.is_available()))
    for ep in range(1, cfg.epochs + 1):
        tau = get_tau(ep, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
        enc.train()
        opt.zero_grad(set_to_none=True)

        if (
            cfg.use_bucket_candidates
            and cfg.refresh_bucket_every > 0
            and (ep == 1 or (ep % cfg.refresh_bucket_every) == 0)
        ):
            support_policy = _make_support_policy(enc, hts_loader_full, tau, cfg)

        # Mixed precision autocast (safe no-op on CPU)
        autocast_enabled = (use_mixed_precision and torch.cuda.is_available())
        with torch.cuda.amp.autocast(enabled=autocast_enabled):
            L_hts, _ = compute_hts_nll(
                encoder=enc,
                hts_loader=hts_loader_full,
                P_xy_given_h=P_xy_h,
                log_norm=(logXY if cfg.use_normalized_objective else 1.0),
                device=device,
                dtype=dtype,
                progress=cfg.progress,
                tau=tau,
                support_policy=support_policy,
            )
            total = L_hts

        # Backprop + step
        if scaler.is_enabled():
            scaler.scale(total).backward()
            scaler.step(opt)
            scaler.update()
        else:
            total.backward()
            opt.step()

        # Rebuild decoder periodically without storing old copies
        if (ep % cfg.rebuild_every) == 0:
            with torch.no_grad():
                P_xy_h = build_nonparam_decoder_xy_given_h(
                    encoder=enc,
                    hts_loader=hts_loader_full,
                    n_xy=n_xy,
                    K=cfg.K,
                    device=device,
                    dtype=dtype,
                    progress=cfg.progress,
                    tau=tau,
                    support_policy=support_policy,
                )

        print(f"[epoch {ep:03d}] tau={tau:.4f}  L_hts={float(L_hts.detach().cpu()):.6f}")

        # Proactively free ephemeral graph/tensors
        del L_hts, total
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # 5) Final evaluation — compute scalars only
    tau_final = get_tau(cfg.epochs, cfg.epochs, cfg.tau_start, cfg.tau_end, cfg.tau_schedule)
    support_policy = _make_support_policy(enc, hts_loader_full, tau_final, cfg)

    with torch.no_grad():
        P_xy_h_final = build_nonparam_decoder_xy_given_h(
            encoder=enc,
            hts_loader=hts_loader_full,
            n_xy=n_xy,
            K=cfg.K,
            device=device,
            dtype=dtype,
            progress=False,
            tau=tau_final,
            support_policy=support_policy,
        )

        L_hts_final, _ = compute_hts_nll(
            encoder=enc,
            hts_loader=hts_loader_full,
            P_xy_given_h=P_xy_h_final,
            log_norm=(math.log(max(n_xy, 2)) if cfg.use_normalized_objective else 1.0),
            device=device,
            dtype=dtype,
            progress=False,
            tau=tau_final,
            support_policy=support_policy,
        )

        L_fus_final = compute_fusion_js(
            encoder=enc,
            hts_loader=hts_loader_full,
            pcm_loader=pcm_loader_full,
            K=cfg.K,
            device=device,
            dtype=dtype,
            use_normalized=cfg.use_normalized_objective,
            tau=tau_final,
            support_policy=support_policy,
        )

    print(f"[FINAL] K={cfg.K}  tau={tau_final:.4f}")
    print(f"        Final likelihood (HTS NLL): {float(L_hts_final):.6f}")
    print(f"        Fusion consistency (JS):    {float(L_fus_final):.6f}")

    # Pull out tiny scalars and immediately free everything heavy
    K_val  = int(cfg.K)
    L_val  = float(L_hts_final.detach().cpu().item())
    JS_val = float(L_fus_final.detach().cpu().item())

    # Drop references to big objects
    del P_xy_h, P_xy_h_final, enc, opt, support_policy
    del hts_loader_full, pcm_loader_full
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

    return {"K": K_val, "L_hts_final": L_val, "fusion_consistency_js": JS_val}


# ---------------------------
# Normalization & elbow (as-is)
# ---------------------------
def _minmax_norm(arr: np.ndarray) -> np.ndarray:
    return arr

def find_elbow_by_max_distance(K_vals: np.ndarray, scores: np.ndarray) -> int:
    assert len(K_vals) == len(scores) and len(K_vals) >= 2
    x = _minmax_norm(np.asarray(K_vals, dtype=float))
    y = np.asarray(scores, dtype=float)
    x1, y1 = x[0], y[0]; x2, y2 = x[-1], y[-1]
    denom = np.hypot(y2 - y1, x2 - x1)
    if denom < 1e-12:
        return int(np.argmin(y))
    num = np.abs((y2 - y1) * x - (x2 - x1) * y + (x2 * y1 - y2 * x1))
    return int(np.argmax(num))


# ---------------------------
# K sweep — store metrics only
# ---------------------------
if __name__ == "__main__":
    # --- Load data (unchanged) ---
    df_hts = pd.read_csv(os.path.join(path_data, "val_data_hts_trip.csv"))
    df_pcm = pd.read_csv(os.path.join(path_data, "val_data_pcm_trip.csv"))
    df_true_trip = pd.read_csv(os.path.join(path_data, "data_sgp_hts_trip.csv"))

    if "COUNT" not in df_pcm.columns:
        df_pcm["COUNT"] = 1.0
        df_pcm["COUNT"] = df_pcm.groupby(
            ["ORIGIN_SUBZONE","DESTINATION_SUBZONE","TRIP_STARTTIME","TRIP_ENDTIME"]
        )["COUNT"].transform("sum")
        df_pcm = df_pcm.drop_duplicates()

    # --- Build Φ and slim tables (unchanged structure) ---
    df_hts["TRIP_DISTANCE"] = distance(df_hts, ['ORIGIN_SUBZONE_X','DESTINATION_SUBZONE_X','ORIGIN_SUBZONE_Y','DESTINATION_SUBZONE_Y'])
    df_pcm["TRIP_DISTANCE"] = distance(df_pcm, ['ORIGIN_SUBZONE_X','DESTINATION_SUBZONE_X','ORIGIN_SUBZONE_Y','DESTINATION_SUBZONE_Y'])
    df_land_use = land_use(df_true_trip)
    df_mode = mode_share(df_true_trip)
    df_hts = df_hts.merge(df_land_use, on='DESTINATION_SUBZONE').merge(df_mode, on='DESTINATION_SUBZONE')
    df_pcm = df_pcm.merge(df_land_use, on='DESTINATION_SUBZONE').merge(df_mode, on='DESTINATION_SUBZONE')

    nb = 10
    spec = DiscreteSpec(n_dist=nb, n_start=nb, n_end=nb, n_lu_mode_clusters=nb,
                        dist_binning="uniform", time_binning="uniform")
    fitted = fit_semantic_spec_on_hts(df_hts, spec)
    df_hts = apply_semantic_spec(df_hts, fitted)
    df_pcm = apply_semantic_spec(df_pcm, fitted)
    phi_segments_meta = fitted["phi_segments"]

    df_hts, phi_dim = build_phi_indices_from_bins(df_hts)
    df_pcm, _       = build_phi_indices_from_bins(df_pcm)

    att_X = ['AGE','GENDER', 'INCOME']
    att_Y = ['TRIP_CNT','TRIP_MAX','TRIP_PURPOSE','TRAVEL_MODE']
    att_Z = ['ORIGIN_SUBZONE','DESTINATION_SUBZONE','TRIP_STARTTIME','TRIP_ENDTIME',
             'ORIGIN_SUBZONE_X','DESTINATION_SUBZONE_X','ORIGIN_SUBZONE_Y','DESTINATION_SUBZONE_Y']
    df_hts = df_hts[att_X + att_Y + att_Z + ['PHI_IDX']]
    df_pcm = df_pcm[att_Z + ['PHI_IDX','COUNT']]
    df_hts['ATT_X'] = df_hts[att_X].astype(str).apply('_'.join, axis=1)
    df_hts['ATT_Y'] = df_hts[att_Y].astype(str).apply('_'.join, axis=1)
    df_hts['ATT_Z'] = df_hts[att_Z].astype(str).apply('_'.join, axis=1)
    df_pcm['ATT_Z'] = df_pcm[att_Z].astype(str).apply('_'.join, axis=1)
    df_hts = reduce_mem_usage(df_hts)
    df_pcm = reduce_mem_usage(df_pcm)

    # --- Config baseline (unchanged) ---
    cfg_base = TrainConfig(
        K=2500, enc_hidden=256, enc_layers=2, enc_dropout=0.0,
        epochs=5, batch_size_hts=8192, batch_size_pcm=16384,
        lr=2e-3, wd=1e-4, device=("cuda" if torch.cuda.is_available() else "cpu"),
        use_normalized_objective=True, rebuild_every=1, progress=True,
        tau_start=0.01, tau_end=0.01, tau_schedule="cosine",
        topk_h=10, use_bucket_candidates=False,
    )

    # --- Sweep K WITHOUT storing models or packs ---
    K_grid = [i*500 for i in range(1,20)]
    rows = []
    for K in K_grid:
        print(f"\n=== Running K={K} ===")
        cfgK = replace(cfg_base, K=K)
        out = train_encoder_only_fusion_finding_K_MINRAM(
            df_hts=df_hts, df_pcm=df_pcm, cfg=cfgK,
            phi_segments_meta=phi_segments_meta, use_mixed_precision=True,
        )
        rows.append(out)
        # hard cleanup between runs
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    dfK = pd.DataFrame(rows).sort_values("K").reset_index(drop=True)

    # Normalize (you kept identity; leave as-is)
    L_norm  = _minmax_norm(dfK["L_hts_final"].to_numpy())
    JS_norm = _minmax_norm(dfK["fusion_consistency_js"].to_numpy())
    score_norm = L_norm + JS_norm

    K_vals = dfK["K"].to_numpy(dtype=int)
    elbow_idx = find_elbow_by_max_distance(K_vals, score_norm)
    best_K = int(K_vals[elbow_idx])

    dfK["L_hts_final_norm"] = L_norm
    dfK["fusion_js_norm"]   = JS_norm
    dfK["score_norm"]       = score_norm
    dfK["is_elbow"]         = False
    dfK.loc[elbow_idx, "is_elbow"] = True

    print("\n=== K sweep results (elbow on normalized sum) ===")
    print(dfK[["K", "L_hts_final", "fusion_consistency_js", "L_hts_final_norm", "fusion_js_norm", "score_norm", "is_elbow"]])
    print(f"\n>>> Selected elbow K = {best_K}")




=== Running K=500 ===
[decoder] pass chunk 0
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 001] tau=0.0100  L_hts=0.611694
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 002] tau=0.0100  L_hts=0.541638
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 003] tau=0.0100  L_hts=0.513787
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 004] tau=0.0100  L_hts=0.500731
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 005] tau=0.0100  L_hts=0.492762
[FINAL] K=500  tau=0.0100
        Final likelihood (HTS NLL): 0.484104
        Fusion consistency (JS):    0.005122

=== Running K=1000 ===
[decoder] pass chunk 0
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 001] tau=0.0100  L_hts=0.573641
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 002] tau=0.0100  L_hts=0.521965
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 003] tau=0.0100  L_hts=0.491059
[hts-nll] pass chunk 0
[decoder] pass chunk 0
[epoch 004] tau=0.0100  L_hts=0.469222
[hts-nll] pass chunk 0
[de

#=======================

In [ ]:
df_pcm = pd.read_csv(path_data + 'val_data_pcm_trip.csv')
df_hts = pd.read_csv(path_data + 'val_data_hts_trip.csv')

print(df_hts.info())
print(df_pcm.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5068 entries, 0 to 5067
Data columns (total 23 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   ID                     5068 non-null   object 
 1   AGE                    5068 non-null   int64  
 2   GENDER                 5068 non-null   int64  
 3   INCOME                 5068 non-null   int64  
 4   TRIP_CNT               5068 non-null   int64  
 5   TRIP_MAX               5068 non-null   int64  
 6   TRIP_PURPOSE           5068 non-null   int64  
 7   TRIP_STARTTIME         5068 non-null   int64  
 8   TRIP_ENDTIME           5068 non-null   int64  
 9   TRAVEL_MODE            5068 non-null   int64  
 10  ORIGIN_SUBZONE         5068 non-null   object 
 11  ORIGIN_SUBZONE_X       5068 non-null   float64
 12  ORIGIN_SUBZONE_Y       5068 non-null   float64
 13  DESTINATION_SUBZONE    5068 non-null   object 
 14  DESTINATION_SUBZONE_X  5068 non-null   float64
 15  DEST

# Length-Stratified Tour Markov Model (LST-MM)

In [ ]:
import pandas as pd
import numpy as np
from scipy.special import rel_entr, kl_div
import math
import time
import warnings
warnings.filterwarnings('ignore')

# NOTE: reduce_mem_usage must be defined or imported before using this module.


# ============================================================
# Column renaming helper
# ============================================================

def rename_columns(df, i):
    """Rename generic trip columns to trip-index-suffixed versions (in-place)."""
    df.rename(columns={
        'TRIP_PURPOSE':          'TRIP_PURPOSE_'          + str(i),
        'ORIGIN_SUBZONE':        'ORIGIN_SUBZONE_'        + str(i),
        'DESTINATION_SUBZONE':   'DESTINATION_SUBZONE_'   + str(i),
        'TRIP_STARTTIME':        'TRIP_STARTTIME_'        + str(i),
        'TRIP_ENDTIME':          'TRIP_ENDTIME_'          + str(i),
        'ORIGIN_SUBZONE_X':      'ORIGIN_SUBZONE_X_'      + str(i),
        'ORIGIN_SUBZONE_Y':      'ORIGIN_SUBZONE_Y_'      + str(i),
        'DESTINATION_SUBZONE_X': 'DESTINATION_SUBZONE_X_' + str(i),
        'DESTINATION_SUBZONE_Y': 'DESTINATION_SUBZONE_Y_' + str(i),
        'Prob_XYZ_fus':          'Prob_'                  + str(i),
    }, inplace=True)


# ============================================================
# Trip simulation (marginal draw for trip i)
# ============================================================

def simulate_trips(df_fus, i, N, rng):
    """
    Draw N trip-i records from df_fus weighted by Prob_XYZ_fus.
    Preserves the full OD distribution from the source table.
    """
    att_Z = [
        'ORIGIN_SUBZONE_'        + str(i),
        'ORIGIN_SUBZONE_X_'      + str(i),
        'ORIGIN_SUBZONE_Y_'      + str(i),
        'DESTINATION_SUBZONE_'   + str(i),
        'DESTINATION_SUBZONE_X_' + str(i),
        'DESTINATION_SUBZONE_Y_' + str(i),
        'TRIP_STARTTIME_'        + str(i),
        'TRIP_ENDTIME_'          + str(i),
    ]
    df_sim = df_fus.copy()
    rename_columns(df_sim, i)
    df_sim.rename(columns={'Prob_' + str(i): 'Prob'}, inplace=True)
    df_sim = df_sim[att_Z + ['Prob']].copy()
    df_sim['Prob'] = df_sim['Prob'] / df_sim['Prob'].sum()
    df_sim = df_sim.reset_index(drop=True)

    prob = df_sim['Prob'].to_numpy()
    idx  = rng.choice(len(df_sim), size=N, replace=True, p=prob)

    df_out = df_sim.iloc[idx][att_Z].reset_index(drop=True)
    return reduce_mem_usage(df_out)


# ============================================================
# Tour re-sampling (weighted draw over already-built partial tours)
# ============================================================

def simulate_tours(df_tours, order, N, rng):
    """
    Re-sample N rows from df_tours weighted by Prob, keeping columns
    ID, TRIP_MAX, and all trip columns up to trip `order`.
    """
    att_X = ['ID', 'TRIP_MAX']
    att_Z = []
    for i in range(1, order + 1):
        att_Z += [
            'ORIGIN_SUBZONE_'        + str(i),
            'ORIGIN_SUBZONE_X_'      + str(i),
            'ORIGIN_SUBZONE_Y_'      + str(i),
            'DESTINATION_SUBZONE_'   + str(i),
            'DESTINATION_SUBZONE_X_' + str(i),
            'DESTINATION_SUBZONE_Y_' + str(i),
            'TRIP_STARTTIME_'        + str(i),
            'TRIP_ENDTIME_'          + str(i),
        ]

    df_t = df_tours[att_X + att_Z + ['Prob']].copy()
    df_t['Prob'] = df_t['Prob'] / df_t['Prob'].sum()
    df_t = df_t.reset_index(drop=True)

    prob = df_t['Prob'].to_numpy()
    idx  = rng.choice(len(df_t), size=N, replace=True, p=prob)

    df_out = df_t.iloc[idx][att_X + att_Z].reset_index(drop=True)
    return reduce_mem_usage(df_out)


# ============================================================
# Build conditional next-trip-start-time distribution from HTS
# ============================================================

def next_start_times(df_hts_):
    """
    Compute P(TRIP_STARTTIME | TRIP_ENDTIME) from HTS activity records.

    Requires columns: ACTIVITY_STARTTIME, ACTIVITY_DURATION.
    Returns DataFrame with: TRIP_ENDTIME, TRIP_STARTTIME, Prob_cond
    """
    df = df_hts_[['ACTIVITY_STARTTIME', 'ACTIVITY_DURATION']].copy()
    df['TRIP_STARTTIME'] = df['ACTIVITY_STARTTIME'] + df['ACTIVITY_DURATION']
    df.rename(columns={'ACTIVITY_STARTTIME': 'TRIP_ENDTIME'}, inplace=True)
    df.drop(columns=['ACTIVITY_DURATION'], inplace=True)

    df['Numbers'] = 1.0
    df['Numbers'] = df.groupby(['TRIP_ENDTIME', 'TRIP_STARTTIME'])['Numbers'].transform('sum')
    df = df.drop_duplicates()

    df['Prob']  = df['Numbers'] / df['Numbers'].sum()
    df['Prob_'] = df.groupby(['TRIP_ENDTIME'])['Prob'].transform('sum')
    df['Prob_cond'] = df['Prob'] / df['Prob_']
    df = df.drop(columns=['Numbers', 'Prob_', 'Prob'])

    return df.dropna()


# ============================================================
# Build OD-preserving conditional trip table for trip i
# ============================================================

def build_conditional_trip_table(df_fus, i):
    """
    Compute P(trip_i | ORIGIN_i, STARTTIME_i) from the fused OD matrix,
    preserving the full OD pair distribution.

    The conditional is: P(DEST, ENDTIME | ORIGIN, STARTTIME) so that
    when we condition on where/when the traveller departs, the drawn
    destination and arrival time honour the OD matrix weights.

    Returns
    -------
    df_cond : DataFrame with all trip-i attribute columns plus Prob_{i}
              representing P(row | ORIGIN_i, STARTTIME_i).
    """
    df_cond = df_fus.copy()
    rename_columns(df_cond, i)
    prob_col = 'Prob_' + str(i)

    # Compute conditional probability: P(dest,endtime | origin,starttime)
    group_keys = ['ORIGIN_SUBZONE_' + str(i), 'TRIP_STARTTIME_' + str(i)]
    df_cond['_marginal'] = df_cond.groupby(group_keys)[prob_col].transform('sum')

    # Guard: if marginal is 0 (shouldn't happen but be safe)
    df_cond[prob_col] = np.where(
        df_cond['_marginal'] > 0,
        df_cond[prob_col] / df_cond['_marginal'],
        0.0
    )
    df_cond = df_cond.drop(columns=['_marginal'])

    # Drop rows with zero probability to keep table lean
    df_cond = df_cond[df_cond[prob_col] > 0].copy()
    df_cond.drop_duplicates(inplace=True)
    df_cond.reset_index(drop=True, inplace=True)

    return df_cond


# ============================================================
# Fallback: sample trip i purely from OD matrix for unmatched rows
# ============================================================

def fallback_trip_sample(df_unmatched, df_fus, i, rng):
    """
    For individuals that could not be matched via spatial+temporal
    continuity, draw trip i directly from the OD matrix conditioned
    only on ORIGIN (relaxing the time constraint).

    This preserves the OD distribution while ensuring every individual
    gets a complete tour rather than being silently dropped.

    Strategy (cascading relaxation):
      1. Try to match on ORIGIN_SUBZONE only (relax time).
      2. If still no match, draw unconditionally from the OD matrix
         (full fallback) — this is rare and means the traveller's
         previous destination has no outgoing trips in df_fus at all.

    Returns
    -------
    df_filled : same schema as df_unmatched plus trip-i columns appended
    """
    trip_cols = [
        'ORIGIN_SUBZONE_'        + str(i),
        'ORIGIN_SUBZONE_X_'      + str(i),
        'ORIGIN_SUBZONE_Y_'      + str(i),
        'DESTINATION_SUBZONE_'   + str(i),
        'DESTINATION_SUBZONE_X_' + str(i),
        'DESTINATION_SUBZONE_Y_' + str(i),
        'TRIP_STARTTIME_'        + str(i),
        'TRIP_ENDTIME_'          + str(i),
    ]

    df_fus_i = df_fus.copy()
    rename_columns(df_fus_i, i)
    prob_col = 'Prob_' + str(i)

    # Normalise OD weights once
    df_fus_i[prob_col] = df_fus_i[prob_col] / df_fus_i[prob_col].sum()

    origin_col = 'ORIGIN_SUBZONE_' + str(i)
    prev_dest  = 'DESTINATION_SUBZONE_' + str(i - 1)

    rows_out = []
    for _, row in df_unmatched.iterrows():
        prev_zone = row[prev_dest]

        # --- Level 1: filter by origin (spatial continuity only) ----------
        pool = df_fus_i[df_fus_i[origin_col] == prev_zone].copy()

        if len(pool) == 0:
            # --- Level 2: unconditional draw from full OD matrix ----------
            pool = df_fus_i.copy()

        # Renormalise within the pool and draw one record
        pool = pool.copy()
        pool[prob_col] = pool[prob_col] / pool[prob_col].sum()
        idx = rng.choice(len(pool), p=pool[prob_col].to_numpy())
        drawn = pool.iloc[idx][trip_cols].to_dict()

        # Merge drawn trip onto the unmatched row
        combined = row.to_dict()
        combined.update(drawn)
        rows_out.append(combined)

    df_filled = pd.DataFrame(rows_out)
    return df_filled


# ============================================================
# Concatenate trip i onto partial tours and split complete/incomplete
# ============================================================

def concat_trips(df_tours, df_ith_trips, df_next_start_times, i,
                 df_fus=None, rng=None, time_tolerance=0):
    """
    Attach trip i to partial tours using spatial and temporal continuity,
    with a fallback mechanism to preserve OD distribution for unmatched rows.

    Spatial continuity : ORIGIN_SUBZONE_i == DESTINATION_SUBZONE_{i-1}
    Temporal continuity: TRIP_STARTTIME_i ~ P(start | end_{i-1}) (conditional)

    Parameters
    ----------
    df_tours            : partial tours DataFrame
    df_ith_trips        : conditional trip-i table (from build_conditional_trip_table)
    df_next_start_times : P(STARTTIME | ENDTIME) lookup
    i                   : current trip index
    df_fus              : original fused OD table (needed for fallback)
    rng                 : np.random.Generator (needed for fallback)
    time_tolerance      : number of time-bins to expand temporal match window
                          (0 = exact match only; increase to relax time constraint)

    Returns
    -------
    df_tours_complete   : rows where TRIP_MAX == i
    df_tours_incomplete : rows where TRIP_MAX >  i
    matched_count       : number of rows matched via primary join
    fallback_count      : number of rows filled via fallback
    """
    df_tours = df_tours.copy()
    prob_col = 'Prob_' + str(i)

    # Carry forward the spatial / temporal link from the previous trip
    df_tours['PRE_SUBZONE'] = df_tours['DESTINATION_SUBZONE_' + str(i - 1)]
    df_tours['PRE_ENDTIME'] = df_tours['TRIP_ENDTIME_'        + str(i - 1)]

    df_nst = df_next_start_times.copy()
    df_nst.rename(columns={
        'TRIP_ENDTIME':   'PRE_ENDTIME',
        'TRIP_STARTTIME': 'PRE_STARTTIME',
    }, inplace=True)

    df_ith = df_ith_trips.copy()
    df_ith['PRE_SUBZONE']   = df_ith['ORIGIN_SUBZONE_'  + str(i)]
    df_ith['PRE_STARTTIME'] = df_ith['TRIP_STARTTIME_'  + str(i)]

    # ── Primary join: temporal then spatial+temporal ──────────────────────
    df_matched = pd.merge(df_tours, df_nst,  on=['PRE_ENDTIME'])
    df_matched = pd.merge(df_matched, df_ith, on=['PRE_SUBZONE', 'PRE_STARTTIME'])
    df_matched = df_matched.drop(columns=['PRE_SUBZONE', 'PRE_ENDTIME', 'PRE_STARTTIME'])

    matched_ids  = set(df_matched['ID'].unique()) if len(df_matched) > 0 else set()
    all_ids      = set(df_tours['ID'].unique())
    unmatched_ids = all_ids - matched_ids
    fallback_count = 0

    # ── Fallback for unmatched individuals ───────────────────────────────
    if unmatched_ids and df_fus is not None and rng is not None:
        df_unmatched = df_tours[df_tours['ID'].isin(unmatched_ids)].copy()
        df_unmatched = df_unmatched.drop(columns=['PRE_SUBZONE', 'PRE_ENDTIME'])
        df_filled    = fallback_trip_sample(df_unmatched, df_fus, i, rng)
        fallback_count = len(df_filled)

        # Assign a nominal joint probability for filled rows
        # (use median of matched prob so they don't dominate resampling)
        if len(df_matched) > 0 and prob_col in df_matched.columns:
            nominal_prob = df_matched[prob_col].median()
        else:
            nominal_prob = 1e-6

        df_filled['Prob'] = (
            nominal_prob * df_filled.get('Prob_cond', pd.Series(1.0, index=df_filled.index))
        )
        # Drop helper columns that may be missing in df_filled
        for col in ['Prob_cond', prob_col]:
            if col in df_filled.columns:
                df_filled = df_filled.drop(columns=[col])

        df_matched['Prob'] = df_matched[prob_col] * df_matched['Prob_cond']
        df_matched = df_matched.drop(columns=[prob_col, 'Prob_cond'])

        df_tours = pd.concat([df_matched, df_filled], ignore_index=True)
    else:
        # Normal path: no fallback needed
        df_matched['Prob'] = df_matched[prob_col] * df_matched['Prob_cond']
        df_matched = df_matched.drop(columns=[prob_col, 'Prob_cond'])
        df_tours = df_matched

    matched_count = len(matched_ids)

    # ── Complete tours: last trip returns home ────────────────────────────
    df_tours_complete = df_tours[df_tours['TRIP_MAX'] == i].copy()
    df_tours_complete['DESTINATION_SUBZONE_'   + str(i)] = df_tours_complete['ORIGIN_SUBZONE_1']
    df_tours_complete['DESTINATION_SUBZONE_X_' + str(i)] = df_tours_complete['ORIGIN_SUBZONE_X_1']
    df_tours_complete['DESTINATION_SUBZONE_Y_' + str(i)] = df_tours_complete['ORIGIN_SUBZONE_Y_1']

    # ── Incomplete tours: still need more trips ───────────────────────────
    df_tours_incomplete = df_tours[df_tours['TRIP_MAX'] > i].copy()

    return df_tours_complete, df_tours_incomplete, matched_count, fallback_count


# ============================================================
# Main generation function
# ============================================================

def generate_tours(df_fus, df_next_start_times, df_hts, N,
                   trip_max_upper=7, scale=10, seed=42, verbose=True):
    """
    Generate N synthetic tour records matching the TRIP_MAX distribution
    of df_hts, with fallback to preserve OD matrix distribution.

    Parameters
    ----------
    df_fus              : fused trip probability table (df_pcm with Prob_XYZ_fus)
    df_next_start_times : conditional start-time table from next_start_times()
    df_hts              : DataFrame with columns ID, TRIP_MAX
    N                   : target number of synthetic individuals
    trip_max_upper      : maximum TRIP_MAX to generate (default 7)
    scale               : re-sampling multiplier for pool diversity
    seed                : random seed
    verbose             : print progress and fallback statistics

    Returns
    -------
    df_result_final : wide-format tour DataFrame, one row per individual
    """
    rng = np.random.default_rng(seed)

    # ── Step 1: TRIP_MAX proportions from ground truth ────────────────────
    df_trip_max = df_hts[['ID', 'TRIP_MAX']].drop_duplicates().copy()
    df_trip_max = df_trip_max.drop(columns=['ID'])
    df_trip_max['Numbers'] = 1.0
    df_trip_max['Numbers'] = df_trip_max.groupby(['TRIP_MAX'])['Numbers'].transform('sum')
    df_trip_max = df_trip_max.drop_duplicates()
    df_trip_max['Pop'] = df_trip_max['Numbers'] / df_trip_max['Numbers'].sum()
    df_trip_max = df_trip_max.drop(columns=['Numbers'])
    df_trip_max = df_trip_max.sort_values('TRIP_MAX', ascending=True)

    tm_vals  = [n for n in range(2, trip_max_upper + 1)
                if n in df_trip_max['TRIP_MAX'].values]
    props    = np.array([
        df_trip_max.loc[df_trip_max['TRIP_MAX'] == n, 'Pop'].values[0]
        for n in tm_vals
    ])
    props    = props / props.sum()
    exact    = props * N
    floors   = np.floor(exact).astype(int)
    deficit  = N - floors.sum()
    top_idx  = np.argsort(exact - floors)[::-1][:deficit]
    floors[top_idx] += 1
    N_k_dict = dict(zip(tm_vals, floors))

    # ── Step 2: Pre-build conditional trip tables for all trip indices ─────
    # This avoids re-computing inside the loop and prevents df_fus mutation.
    if verbose:
        print("Pre-building conditional trip tables …")
    cond_tables = {}
    for i in range(2, trip_max_upper + 1):
        cond_tables[i] = build_conditional_trip_table(df_fus, i)

    # ── Step 3: generate tours bucket by bucket ───────────────────────────
    df_result_final = pd.DataFrame()
    total_fallback  = 0
    total_matched   = 0

    for n in range(2, trip_max_upper + 1):
        N1 = N_k_dict.get(n, 0)
        if N1 == 0:
            continue

        if verbose:
            print(f"\n── TRIP_MAX = {n}  (target N = {N1}) ──")

        # Seed trip 1 for this TRIP_MAX bucket
        df_tours = simulate_trips(df_fus, 1, int(scale * N1), rng)
        df_tours = df_tours.reset_index(drop=True)
        df_tours['ID']       = df_tours.index
        df_tours['TRIP_MAX'] = n

        df_result = pd.DataFrame()

        for i in range(2, n + 1):
            df_ith_trips = cond_tables[i]

            df_tours_complete, df_tours_incomplete, mc, fc = concat_trips(
                df_tours,
                df_ith_trips,
                df_next_start_times,
                i,
                df_fus=df_fus,
                rng=rng,
            )
            total_matched  += mc
            total_fallback += fc

            if verbose:
                print(f"  trip {i}: matched={mc}, fallback={fc}, "
                      f"complete={len(df_tours_complete)}, "
                      f"incomplete={len(df_tours_incomplete)}")

            if len(df_tours_complete) > 0:
                df_result = pd.concat([
                    df_result,
                    simulate_tours(df_tours_complete, i, int(scale * N1), rng)
                ])

            if len(df_tours_incomplete) > 0:
                df_tours = simulate_tours(
                    df_tours_incomplete, i, int(scale * N1), rng
                )
            else:
                # All tours completed (or fallback filled them all as complete)
                break

        if len(df_result) == 0:
            if verbose:
                print(f"  WARNING: no complete tours generated for TRIP_MAX={n}")
            continue

        # Keep one row per local ID, then sample exactly N1 rows
        df_result = df_result.drop_duplicates(subset=['ID'], keep='first')

        # If we have fewer rows than needed, sample with replacement
        if len(df_result) < N1:
            if verbose:
                print(f"  NOTE: only {len(df_result)} unique tours for "
                      f"TRIP_MAX={n}, sampling {N1} with replacement.")
            df_result = df_result.sample(n=N1, replace=True,
                                         random_state=int(rng.integers(1e9)))
        else:
            df_result = df_result.sample(n=N1, replace=False,
                                         random_state=int(rng.integers(1e9)))

        df_result_final = pd.concat([df_result_final, df_result],
                                    ignore_index=True)

    # Assign globally unique ID
    df_result_final = df_result_final.reset_index(drop=True)
    df_result_final['ID'] = df_result_final.index

    if verbose:
        fallback_pct = (100 * total_fallback / max(total_matched + total_fallback, 1))
        print(f"\n── Generation complete ──")
        print(f"   Total rows : {len(df_result_final)}")
        print(f"   Matched    : {total_matched}")
        print(f"   Fallback   : {total_fallback}  ({fallback_pct:.1f}%)")
        print(f"   (Fallback rows used OD-matrix-conditioned draw, "
              f"relaxed time constraint)")

    return df_result_final


# ============================================================
# Minimal reduce_mem_usage
# ============================================================

def reduce_mem_usage(df):
    """Downcast numeric columns to reduce memory footprint."""
    for col in df.columns:
        col_type = df[col].dtype
        if col_type != object:
            c_min = df[col].min()
            c_max = df[col].max()
            if str(col_type).startswith('int'):
                if   c_min >= np.iinfo(np.int8).min  and c_max <= np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
            elif str(col_type).startswith('float'):
                if c_min >= np.finfo(np.float32).min and c_max <= np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    return df


# ============================================================
# Entry point
# ============================================================

if __name__ == '__main__':
    ff = 0.5
    TRIP_MAX_ = 7

    df_pcm = pd.read_csv(path_data + 'val_data_pcm_trip.csv')
    df_hts = pd.read_csv(path_data + f'val_data_hts_trip_{ff}.csv')

    # Assign uniform weight to each unique PCM record
    columns_ = df_pcm.columns.to_list()
    df_pcm['Prob_XYZ_fus'] = 1
    df_pcm['Prob_XYZ_fus'] = df_pcm.groupby(columns_)['Prob_XYZ_fus'].transform('sum')
    df_pcm.drop_duplicates(inplace=True)
    df_pcm['Prob_XYZ_fus'] = df_pcm['Prob_XYZ_fus'] / df_pcm['Prob_XYZ_fus'].sum()

    df_next_start_times_ = next_start_times(df_hts)

    NN = 50000
    df_tours = generate_tours(
        df_pcm, df_next_start_times_, df_hts,
        N=NN,
        trip_max_upper=TRIP_MAX_,
        scale=10,
        seed=42,
        verbose=True,
    )

    print(df_tours.info())
    print(df_tours["TRIP_MAX"].value_counts())
    df_tours.to_csv(path_result + f'val_sim_tour_context_free_{ff}.csv', index=False)

Pre-building conditional trip tables …

── TRIP_MAX = 2  (target N = 35633) ──
  trip 2: matched=353036, fallback=3294, complete=37304054, incomplete=0

── TRIP_MAX = 3  (target N = 7002) ──
  trip 2: matched=69386, fallback=634, complete=0, incomplete=7328132
  trip 3: matched=39971, fallback=526, complete=5061258, incomplete=0

── TRIP_MAX = 4  (target N = 4043) ──
  trip 2: matched=40046, fallback=384, complete=0, incomplete=4257578
  trip 3: matched=23131, fallback=283, complete=0, incomplete=2953660
  trip 4: matched=15489, fallback=295, complete=2366465, incomplete=0

── TRIP_MAX = 5  (target N = 1752) ──
  trip 2: matched=17342, fallback=178, complete=0, incomplete=1838432
  trip 3: matched=9923, fallback=125, complete=0, incomplete=1294696
  trip 4: matched=6627, fallback=127, complete=0, incomplete=1062246
  trip 5: matched=4707, fallback=108, complete=910467, incomplete=0

── TRIP_MAX = 6  (target N = 1122) ──
  trip 2: matched=11111, fallback=109, complete=0, incomplete=1174

# Explore and Return

In [ ]:
"""
Explore-and-Return (XR) Model for Individual Travel Demand Synthesis
====================================================================
Adapted from:
Anda et al. (2021), Transportation Research Part C 128, 103118

Supported generation modes
--------------------------
  method = "stratified_pool"
      Build a large unconditional pool, bucket trajectories by final tour code,
      and sample to match the survey tour-code distribution.

  method = "none"
      No global tour-code correction. Raw forward simulation from the XR model.

Main idea
---------
1. Fit local XR CPDs:
   Survey:
     P1   : P(Z1 | S1)
     PE1  : P(E1 | Z1, S1)
     PD   : P(Dk | Zk, Sk)
     XR   : P(XRk | Zk, Ek)

   OD matrix:
     PZ   : P(Zk | Zk-1, Ek-1)
     PS   : P(Sk | Zk, Zk-1, Ek-1)

2. Optionally compute the target final tour-code distribution directly from the
   input survey.

3. If method="stratified_pool":
   - convert target probabilities into target counts N_c for each tour code c
   - generate a large unconditional pool from the XR model
   - bucket trajectories by final tour code
   - top up shortfalls for rare codes
   - sample exactly N_c from each bucket

4. If method="none":
   - generate N trajectories directly from the XR model without any stratification

Important time mapping from OD trips to stay-points
---------------------------------------------------
OD trip table:
  TRIP_STARTTIME = trip departure time
  TRIP_ENDTIME   = trip arrival time

Stay-point mapping:
  E_{k-1} = TRIP_STARTTIME   (end time of previous stay-point)
  S_k     = TRIP_ENDTIME     (start time of next stay-point)

Therefore:
  PZ[(origin_zone, TRIP_STARTTIME_H)][dest_zone]
  PS[(dest_zone, origin_zone, TRIP_STARTTIME_H)][TRIP_ENDTIME_H]
"""

import numpy as np
import pandas as pd
from collections import defaultdict
import warnings

warnings.filterwarnings("ignore")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 — DATA LOADING & PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────

def load_survey(path: str) -> pd.DataFrame:
    """
    Load household travel survey and return one row per trip.

    Expected columns
    ----------------
    ID, TRIP_CNT, TRIP_MAX, TRIP_STARTTIME, TRIP_ENDTIME,
    ORIGIN_SUBZONE, ORIGIN_SUBZONE_X, ORIGIN_SUBZONE_Y,
    DESTINATION_SUBZONE, DESTINATION_SUBZONE_X, DESTINATION_SUBZONE_Y,
    HOME_SUBZONE, HOME_SUBZONE_X, HOME_SUBZONE_Y
    """
    df = pd.read_csv(path)
    df.columns = df.columns.str.upper().str.strip()
    return df


def load_od(path: str) -> pd.DataFrame:
    """
    Load hourly OD matrix.

    Expected columns
    ----------------
    TRIP_STARTTIME, TRIP_ENDTIME,
    ORIGIN_SUBZONE, ORIGIN_SUBZONE_X, ORIGIN_SUBZONE_Y,
    DESTINATION_SUBZONE, DESTINATION_SUBZONE_X, DESTINATION_SUBZONE_Y
    """
    df = pd.read_csv(path)
    df.columns = df.columns.str.upper().str.strip()
    return df


def to_hour(t) -> int:
    """
    Convert a time value to integer hour (0-23).

    Accepts:
      - int / float
      - 'HH:MM'
      - 'HH:MM:SS'
    """
    if pd.isna(t):
        return np.nan
    if isinstance(t, (int, float)):
        return int(t) % 24
    t = str(t).strip()
    parts = t.split(":")
    return int(parts[0]) % 24


def build_stay_points_from_survey(survey: pd.DataFrame) -> pd.DataFrame:
    """
    Convert trip-based survey into stay-point trajectories per person.

    Logic
    -----
      - Each person starts the day at HOME_SUBZONE (stay-point 0).
      - Each trip's DESTINATION_SUBZONE becomes the next stay-point.
      - Start time of stay-point i = TRIP_ENDTIME of trip i
      - End time of stay-point i   = TRIP_STARTTIME of trip i+1
      - The last stay-point ends at hour 24
      - Stay-point 0 ends at the departure time of the first trip

    Returns
    -------
    DataFrame with columns:
      ID, sp_idx, zone, x, y, start_h, end_h, duration_h
    """
    survey = survey.copy()
    survey["TRIP_STARTTIME_H"] = survey["TRIP_STARTTIME"].apply(to_hour)
    survey["TRIP_ENDTIME_H"] = survey["TRIP_ENDTIME"].apply(to_hour)
    survey = survey.sort_values(["ID", "TRIP_CNT"]).reset_index(drop=True)

    records = []

    for pid, grp in survey.groupby("ID"):
        grp = grp.sort_values("TRIP_CNT").reset_index(drop=True)

        home_zone = grp.iloc[0]["HOME_SUBZONE"]
        home_x = grp.iloc[0]["HOME_SUBZONE_X"]
        home_y = grp.iloc[0]["HOME_SUBZONE_Y"]
        first_dep = grp.iloc[0]["TRIP_STARTTIME_H"]

        sp0_end = first_dep if pd.notna(first_dep) else 8
        records.append({
            "ID": pid,
            "sp_idx": 0,
            "zone": home_zone,
            "x": home_x,
            "y": home_y,
            "start_h": 0,
            "end_h": sp0_end,
            "duration_h": max(sp0_end, 0)
        })

        for i, row in grp.iterrows():
            dest_zone = row["DESTINATION_SUBZONE"]
            dest_x = row["DESTINATION_SUBZONE_X"]
            dest_y = row["DESTINATION_SUBZONE_Y"]
            arr_h = row["TRIP_ENDTIME_H"]

            if i < len(grp) - 1:
                dep_h = grp.iloc[i + 1]["TRIP_STARTTIME_H"]
            else:
                dep_h = 24

            if pd.isna(arr_h):
                arr_h = sp0_end
            if pd.isna(dep_h):
                dep_h = arr_h + 1

            dur = max(dep_h - arr_h, 0)

            records.append({
                "ID": pid,
                "sp_idx": int(row["TRIP_CNT"]),
                "zone": dest_zone,
                "x": dest_x,
                "y": dest_y,
                "start_h": arr_h,
                "end_h": dep_h,
                "duration_h": dur
            })

    return pd.DataFrame(records)


def distribution_from_series(s: pd.Series) -> dict:
    """
    Convert a categorical series into a probability dictionary.
    """
    probs = s.value_counts(normalize=True)
    return probs.to_dict()


def allocate_target_counts(target_dist: dict, n_agents: int) -> dict:
    """
    Convert probability distribution to exact integer counts summing to n_agents.
    """
    if not target_dist:
        return {}

    codes = list(target_dist.keys())
    probs = np.array([target_dist[c] for c in codes], dtype=float)
    probs = probs / probs.sum()

    exact = probs * n_agents
    floors = np.floor(exact).astype(int)
    deficit = n_agents - floors.sum()

    if deficit > 0:
        frac = exact - floors
        top_idx = np.argsort(frac)[::-1][:deficit]
        floors[top_idx] += 1

    return dict(zip(codes, floors))


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — MLE HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def _norm(d: dict) -> dict:
    """
    Normalize a frequency dict to a probability dict.
    """
    total = sum(d.values())
    if total == 0:
        return {}
    return {k: v / total for k, v in d.items()}


def _extract_final_tour(recs: list) -> str:
    """
    Final zone-pattern code used for stratified pool sampling.
    """
    if not recs:
        return ""

    zone_to_digit = {}
    next_digit = 0
    digits = []

    for r in recs:
        z = r["zone"]
        if z not in zone_to_digit:
            zone_to_digit[z] = next_digit
            next_digit += 1
        digits.append(str(zone_to_digit[z]))

    return "".join(digits)


def build_survey_tour_distribution(survey_df: pd.DataFrame) -> dict:
    """
    Build the final tour-code distribution directly from the input survey.
    """
    sp_df = build_stay_points_from_survey(survey_df)
    survey_tours = (
        sp_df.sort_values(["ID", "sp_idx"])
             .groupby("ID")["zone"]
             .apply(lambda s: _extract_final_tour([{"zone": z} for z in s.tolist()]))
    )
    return distribution_from_series(survey_tours)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 — XR MODEL
# ─────────────────────────────────────────────────────────────────────────────

class XRModel:
    """
    Explore-and-Return model.

    CPDs
    ----
    Survey:
      P1   : P(Z1 | S1)
      PE1  : P(E1 | Z1, S1)
      PD   : P(Dk | Zk, Sk)
      XR   : P(XRk | Zk, Ek)   where XRk in {"explore", "return"}

    OD matrix:
      PZ   : P(Zk | Zk-1, Ek-1)
      PS   : P(Sk | Zk, Zk-1, Ek-1)
    """

    def __init__(self):
        self.P1 = {}
        self.PE1 = {}
        self.PZ = {}
        self.PS = {}
        self.PD = {}
        self.XR = {}
        self.zone_coords = {}

    # ── helpers ──────────────────────────────────────────────────────────────

    def _sample(self, prob_dict: dict):
        """
        Draw one sample from a probability dict {value: prob}.
        """
        if not prob_dict:
            return None

        keys = list(prob_dict.keys())
        probs = np.array([prob_dict[k] for k in keys], dtype=float)
        s = probs.sum()

        if s <= 0:
            return None

        probs = probs / s
        idx = np.searchsorted(np.cumsum(probs), np.random.random(), side="right")
        return keys[min(idx, len(keys) - 1)]

    def _fallback_zone_dist(self, prev_zone):
        """
        Fallback marginal next-zone distribution for a given previous zone.
        Aggregates over all end-hour conditions in PZ.
        """
        agg = defaultdict(float)

        for (pz, _), d in self.PZ.items():
            if pz == prev_zone:
                for z, p in d.items():
                    agg[z] += p

        if agg:
            return _norm(dict(agg))

        zones = list(self.zone_coords.keys())
        if not zones:
            return {}

        return {z: 1 / len(zones) for z in zones}

    def _fallback_ps_dist(self, next_zone, prev_zone):
        """
        Fallback for PS by aggregating over end-hour conditions.
        """
        agg = defaultdict(float)

        for (z, pz, _), d in self.PS.items():
            if z == next_zone and pz == prev_zone:
                for sh, p in d.items():
                    agg[sh] += p

        if agg:
            return _norm(dict(agg))

        return {}

    def _fallback_pd_dist(self, zone):
        """
        Fallback for PD by aggregating over start-hour conditions.
        """
        agg = defaultdict(float)

        for (z, _), d in self.PD.items():
            if z == zone:
                for dur, p in d.items():
                    agg[dur] += p

        if agg:
            return _norm(dict(agg))

        return {1: 1.0}

    def _fallback_xr_dist(self, zone):
        """
        Fallback for XR by aggregating over end-hour conditions.
        """
        agg = defaultdict(float)

        for (z, _), d in self.XR.items():
            if z == zone:
                for xr_state, p in d.items():
                    agg[xr_state] += p

        if agg:
            return _norm(dict(agg))

        return {"explore": 0.5, "return": 0.5}

    # ── fitting subroutines ─────────────────────────────────────────────────

    def _fit_survey_cpds(self, sp_df: pd.DataFrame):
        """
        Fit P1, PE1, PD, XR from survey stay-points.

        XR fitting rule
        ---------------
        At a current stay-point k-1 with state (Z_{k-1}, E_{k-1}), classify the
        transition to stay-point k as:
          - "explore" if next zone has not been visited before
          - "explore" also if next zone == current zone
          - "return" otherwise
        """
        freq_P1 = defaultdict(lambda: defaultdict(float))
        freq_PE1 = defaultdict(lambda: defaultdict(float))
        freq_PD = defaultdict(lambda: defaultdict(float))
        freq_XR = defaultdict(lambda: defaultdict(float))

        for _, row in sp_df.iterrows():
            self.zone_coords[row["zone"]] = (row["x"], row["y"])

        for pid, grp in sp_df.groupby("ID"):
            grp = grp.sort_values("sp_idx").reset_index(drop=True)

            for k in range(len(grp)):
                row = grp.iloc[k]
                zone = row["zone"]
                start_h = int(row["start_h"]) % 24
                end_h = int(row["end_h"]) % 24
                dur_h = min(int(max(row["duration_h"], 0)), 23)

                if k == 0:
                    freq_P1[start_h][zone] += 1
                    freq_PE1[(zone, start_h)][end_h] += 1
                else:
                    freq_PD[(zone, start_h)][dur_h] += 1

            visited = {grp.iloc[0]["zone"]}

            for k in range(1, len(grp)):
                current_row = grp.iloc[k - 1]
                next_row = grp.iloc[k]

                current_zone = current_row["zone"]
                current_end_h = int(current_row["end_h"]) % 24
                next_zone = next_row["zone"]

                if next_zone == current_zone:
                    xr_state = "explore"
                elif next_zone in visited:
                    xr_state = "return"
                else:
                    xr_state = "explore"

                freq_XR[(current_zone, current_end_h)][xr_state] += 1
                visited.add(next_zone)

        self.P1 = {k: _norm(dict(v)) for k, v in freq_P1.items()}
        self.PE1 = {k: _norm(dict(v)) for k, v in freq_PE1.items()}
        self.PD = {k: _norm(dict(v)) for k, v in freq_PD.items()}
        self.XR = {k: _norm(dict(v)) for k, v in freq_XR.items()}

    def _fit_od_cpds(self, od_df: pd.DataFrame):
        """
        Fit PZ and PS from OD matrix.

        Mapping from OD trips to stay-point semantics
        ---------------------------------------------
        origin zone      = Z_{k-1}
        destination zone = Z_k
        trip start time  = E_{k-1}
        trip end time    = S_k
        """
        od_df = od_df.copy()
        od_df["TRIP_STARTTIME_H"] = od_df["TRIP_STARTTIME"].apply(to_hour)
        od_df["TRIP_ENDTIME_H"] = od_df["TRIP_ENDTIME"].apply(to_hour)

        freq_PZ = defaultdict(lambda: defaultdict(float))
        freq_PS = defaultdict(lambda: defaultdict(float))

        for _, row in od_df.iterrows():
            origin_zone = row["ORIGIN_SUBZONE"]
            dest_zone = row["DESTINATION_SUBZONE"]

            prev_end_h = row["TRIP_STARTTIME_H"]   # E_{k-1}
            start_h = row["TRIP_ENDTIME_H"]        # S_k

            if pd.isna(prev_end_h) or pd.isna(start_h):
                continue

            prev_end_h = int(prev_end_h) % 24
            start_h = int(start_h) % 24

            freq_PZ[(origin_zone, prev_end_h)][dest_zone] += 1
            freq_PS[(dest_zone, origin_zone, prev_end_h)][start_h] += 1

            if origin_zone not in self.zone_coords:
                self.zone_coords[origin_zone] = (
                    row.get("ORIGIN_SUBZONE_X", None),
                    row.get("ORIGIN_SUBZONE_Y", None)
                )
            if dest_zone not in self.zone_coords:
                self.zone_coords[dest_zone] = (
                    row.get("DESTINATION_SUBZONE_X", None),
                    row.get("DESTINATION_SUBZONE_Y", None)
                )

        self.PZ = {k: _norm(dict(v)) for k, v in freq_PZ.items()}
        self.PS = {k: _norm(dict(v)) for k, v in freq_PS.items()}

    # ── public fit ──────────────────────────────────────────────────────────

    def fit(self, sp_df: pd.DataFrame, od_df: pd.DataFrame):
        """
        Fit all XR-model CPDs.
        """
        print("Fitting XR model...")
        self._fit_survey_cpds(sp_df.copy())
        self._fit_od_cpds(od_df.copy())

        print(f"  P1  : {len(self.P1)} conditions  (survey)")
        print(f"  PE1 : {len(self.PE1)} conditions (survey)")
        print(f"  PZ  : {len(self.PZ)} conditions  (OD)")
        print(f"  PS  : {len(self.PS)} conditions  (OD)")
        print(f"  PD  : {len(self.PD)} conditions  (survey)")
        print(f"  XR  : {len(self.XR)} conditions  (survey)")
        print(f"  Zones known: {len(self.zone_coords)}")
        print("Done.")

    # ── generation ───────────────────────────────────────────────────────────

    def _sample_next_zone_with_xr(self, prev_zone, prev_end_h, visited):
        """
        Sample the next zone using:
          1. XR state from current zone and current end time
          2. PZ transition distribution from OD matrix

        Explore:
          - exclude already visited zones
          - but allow current zone itself

        Return:
          - allow only previously visited zones
          - exclude current zone
        """
        xr_dist = self.XR.get((prev_zone, prev_end_h)) or self._fallback_xr_dist(prev_zone)
        xr_state = self._sample(xr_dist)
        if xr_state is None:
            xr_state = "explore"

        base_pz = self.PZ.get((prev_zone, prev_end_h)) or self._fallback_zone_dist(prev_zone)
        if not base_pz:
            return None, xr_state

        if xr_state == "explore":
            filtered = {
                z: p for z, p in base_pz.items()
                if (z not in visited) or (z == prev_zone)
            }
            if not filtered:
                filtered = {z: p for z, p in base_pz.items() if z == prev_zone} or base_pz
        else:
            filtered = {
                z: p for z, p in base_pz.items()
                if (z in visited) and (z != prev_zone)
            }
            if not filtered:
                filtered = {z: p for z, p in base_pz.items() if z in visited} or base_pz

        next_zone = self._sample(_norm(filtered))
        return next_zone, xr_state

    def generate_one(self, agent_id: int, start_h: int = 0, max_stay_points: int = 12) -> list:
        """
        Forward-sample one synthetic agent's daily stay-point trajectory.
        """
        records = []
        current_h = start_h % 24

        p1_dist = self.P1.get(current_h) or (next(iter(self.P1.values())) if self.P1 else {})
        if not p1_dist:
            return records

        zone = self._sample(p1_dist)
        if zone is None:
            return records

        pe1_dist = self.PE1.get((zone, current_h), {})
        end_h = self._sample(pe1_dist) if pe1_dist else (current_h + 2)

        x, y = self.zone_coords.get(zone, (None, None))
        records.append({
            "id": agent_id,
            "sp_idx": 0,
            "zone": zone,
            "x": x,
            "y": y,
            "start_h": current_h,
            "end_h": end_h,
            "duration_h": max(end_h - current_h, 0),
            "xr_state": None
        })

        visited = {zone}
        prev_zone = zone
        prev_end_h = int(end_h) % 24
        sp_idx = 1

        while sp_idx <= max_stay_points and prev_end_h < 24:
            next_zone, xr_state = self._sample_next_zone_with_xr(prev_zone, prev_end_h, visited)
            if next_zone is None:
                break

            ps_dist = self.PS.get((next_zone, prev_zone, prev_end_h)) \
                      or self._fallback_ps_dist(next_zone, prev_zone)
            start_h_sp = self._sample(ps_dist) if ps_dist else prev_end_h

            if start_h_sp is None:
                start_h_sp = prev_end_h

            if start_h_sp < prev_end_h:
                start_h_sp = prev_end_h

            sh_mod = int(start_h_sp) % 24
            pd_dist = self.PD.get((next_zone, sh_mod)) or self._fallback_pd_dist(next_zone)
            dur = self._sample(pd_dist) if pd_dist else 1
            dur = max(int(dur), 0)

            end_h_sp = start_h_sp + dur
            if end_h_sp > 24:
                end_h_sp = 24
                dur = max(end_h_sp - start_h_sp, 0)

            x, y = self.zone_coords.get(next_zone, (None, None))
            records.append({
                "id": agent_id,
                "sp_idx": sp_idx,
                "zone": next_zone,
                "x": x,
                "y": y,
                "start_h": start_h_sp,
                "end_h": end_h_sp,
                "duration_h": dur,
                "xr_state": xr_state
            })

            visited.add(next_zone)
            prev_zone = next_zone
            prev_end_h = int(end_h_sp) % 24
            sp_idx += 1

            if end_h_sp >= 24:
                break

        return records

    def generate_population(self, n_agents: int, start_h: int = 0, max_stay_points: int = 12) -> pd.DataFrame:
        """
        Generate a synthetic population without any global tour-code correction.
        """
        print(f"Generating {n_agents:,} synthetic agents (no stratification)...")
        all_records = []
        tenth = max(1, n_agents // 10)

        for i in range(n_agents):
            all_records.extend(self.generate_one(i, start_h=start_h, max_stay_points=max_stay_points))
            if (i + 1) % tenth == 0:
                print(f"  {i + 1:,}/{n_agents:,} done")

        print("Generation complete.")
        return pd.DataFrame(all_records)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 — STRATIFIED POOL SAMPLING
# ─────────────────────────────────────────────────────────────────────────────

def build_pool_by_tour_code(model: XRModel,
                            pool_size: int,
                            max_stay_points: int = 12,
                            start_id: int = 0):
    """
    Generate an unconditional pool and bucket trajectories by final tour code.

    Returns
    -------
    pool_by_code : {tour_code: [trajectory, ...]}
    next_id      : next available synthetic id after pool generation
    """
    pool_by_code = defaultdict(list)
    next_id = start_id

    for _ in range(pool_size):
        recs = model.generate_one(next_id, max_stay_points=max_stay_points)
        code = _extract_final_tour(recs)
        pool_by_code[code].append(recs)
        next_id += 1

    return pool_by_code, next_id


def top_up_pool_by_code(model: XRModel,
                        pool_by_code: dict,
                        target_counts: dict,
                        max_stay_points: int = 12,
                        batch_size: int = 5_000,
                        max_rounds: int = 20,
                        start_id: int = 0):
    """
    Top up only the tour-code buckets that are still short.
    """
    next_id = start_id

    for round_idx in range(max_rounds):
        shortfall = {
            c: max(target_counts.get(c, 0) - len(pool_by_code.get(c, [])), 0)
            for c in target_counts
        }
        total_short = sum(shortfall.values())

        if total_short == 0:
            print("  No shortfall remains after pool generation.")
            break

        active_short = sum(v > 0 for v in shortfall.values())
        print(f"  Top-up round {round_idx + 1}: total shortfall={total_short:,}, active codes={active_short:,}")

        for _ in range(batch_size):
            recs = model.generate_one(next_id, max_stay_points=max_stay_points)
            code = _extract_final_tour(recs)
            if code in target_counts and len(pool_by_code[code]) < target_counts[code]:
                pool_by_code[code].append(recs)
            next_id += 1

    return pool_by_code, next_id


def sample_from_stratified_pool(pool_by_code: dict,
                                target_counts: dict,
                                rng_seed: int = 42) -> pd.DataFrame:
    """
    Sample exactly N_c trajectories from each tour-code bucket.
    Uses replacement only when a bucket remains smaller than N_c.
    """
    rng = np.random.default_rng(rng_seed)
    accepted_rows = []
    new_id = 0

    target_counts_sorted = dict(sorted(target_counts.items(), key=lambda x: (-x[1], x[0])))

    for code, n_target in target_counts_sorted.items():
        if n_target <= 0:
            continue

        bucket = pool_by_code.get(code, [])

        if len(bucket) == 0:
            print(f"  WARNING: no trajectories available for code {code}; skipping.")
            continue

        replace = len(bucket) < n_target
        idx = rng.choice(len(bucket), size=n_target, replace=replace)

        if replace:
            print(f"  WARNING: bucket {code} has only {len(bucket):,} trajectories; "
                  f"sampling {n_target:,} with replacement.")

        for k in idx:
            recs = bucket[k]
            for r in recs:
                row = dict(r)
                row["id"] = new_id
                accepted_rows.append(row)
            new_id += 1

    return pd.DataFrame(accepted_rows)


def stratified_pool_sampling(model: XRModel,
                             target_counts: dict,
                             initial_pool_factor: int = 5,
                             min_pool_size: int = 10_000,
                             max_stay_points: int = 12,
                             topup_batch_size: int = 5_000,
                             topup_max_rounds: int = 20,
                             rng_seed: int = 42) -> pd.DataFrame:
    """
    Pool-based stratified sampling by final tour code.

    Algorithm
    ---------
    1. Compute target counts N_c.
    2. Generate a large unconditional pool.
    3. Bucket trajectories by final tour code.
    4. Top up only short buckets.
    5. Sample exactly N_c from each bucket.

    Returns
    -------
    DataFrame of synthetic stay-point trajectories.
    """
    total_n = sum(target_counts.values())
    initial_pool_size = max(initial_pool_factor * total_n, min_pool_size)

    print(f"Stratified pool sampling: target={total_n:,}, initial_pool={initial_pool_size:,}")

    pool_by_code, next_id = build_pool_by_tour_code(
        model=model,
        pool_size=initial_pool_size,
        max_stay_points=max_stay_points,
        start_id=0
    )

    target_codes = set(target_counts.keys())
    initial_cover = sum(len(pool_by_code.get(c, [])) > 0 for c in target_codes)
    print(f"  Initial bucket coverage: {initial_cover}/{len(target_codes)} target tour codes")

    pool_by_code, next_id = top_up_pool_by_code(
        model=model,
        pool_by_code=pool_by_code,
        target_counts=target_counts,
        max_stay_points=max_stay_points,
        batch_size=topup_batch_size,
        max_rounds=topup_max_rounds,
        start_id=next_id
    )

    synth_df = sample_from_stratified_pool(
        pool_by_code=pool_by_code,
        target_counts=target_counts,
        rng_seed=rng_seed
    )

    n_agents_out = synth_df["id"].nunique() if not synth_df.empty else 0
    print(f"  Stratified pool sampling complete: generated {n_agents_out:,} agents")

    return synth_df


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 — OUTPUT CONVERSION
# ─────────────────────────────────────────────────────────────────────────────

def stay_points_to_trips(synth_df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert synthetic stay-point trajectories into survey-format trip records.

    Mapping
    -------
      TRIP_STARTTIME = end_h of origin stay-point
      TRIP_ENDTIME   = start_h of destination stay-point
      TRIP_CNT       = sp_idx of destination
      TRIP_MAX       = total trips that day for that agent
    """
    synth_df = synth_df.sort_values(["id", "sp_idx"]).reset_index(drop=True)
    records = []

    for agent_id, grp in synth_df.groupby("id"):
        grp = grp.sort_values("sp_idx").reset_index(drop=True)
        n = len(grp)

        if n < 2:
            continue

        trip_max = n - 1

        for k in range(1, n):
            o = grp.iloc[k - 1]
            d = grp.iloc[k]

            records.append({
                "ID": agent_id,
                "TRIP_CNT": k,
                "TRIP_MAX": trip_max,
                "TRIP_STARTTIME": o["end_h"],
                "TRIP_ENDTIME": d["start_h"],
                "ORIGIN_SUBZONE": o["zone"],
                "ORIGIN_SUBZONE_X": o["x"],
                "ORIGIN_SUBZONE_Y": o["y"],
                "DESTINATION_SUBZONE": d["zone"],
                "DESTINATION_SUBZONE_X": d["x"],
                "DESTINATION_SUBZONE_Y": d["y"],
            })

    return pd.DataFrame(records, columns=[
        "ID", "TRIP_CNT", "TRIP_MAX",
        "TRIP_STARTTIME", "TRIP_ENDTIME",
        "ORIGIN_SUBZONE", "ORIGIN_SUBZONE_X", "ORIGIN_SUBZONE_Y",
        "DESTINATION_SUBZONE", "DESTINATION_SUBZONE_X", "DESTINATION_SUBZONE_Y",
    ])


def trip_diaries_to_wide(df_trip: pd.DataFrame) -> pd.DataFrame:
    """
    Convert long-format trip diaries to wide tour format.
    """
    trip_cols = [
        "ORIGIN_SUBZONE", "ORIGIN_SUBZONE_X", "ORIGIN_SUBZONE_Y",
        "DESTINATION_SUBZONE", "DESTINATION_SUBZONE_X", "DESTINATION_SUBZONE_Y",
        "TRIP_STARTTIME", "TRIP_ENDTIME",
    ]

    df = df_trip.copy()
    df["ID"] = df["ID"].astype(str)

    df_wide = df.pivot_table(
        index="ID",
        columns="TRIP_CNT",
        values=trip_cols,
        aggfunc="first"
    )
    df_wide.columns = [f"{col}_{int(k)}" for col, k in df_wide.columns]
    df_wide = df_wide.reset_index()

    trip_max_map = df.groupby("ID")["TRIP_MAX"].max()
    df_wide = df_wide.merge(trip_max_map.rename("TRIP_MAX"), on="ID")

    max_trips = int(df["TRIP_MAX"].max()) if not df.empty else 0
    ordered = ["ID", "TRIP_MAX"]

    for k in range(1, max_trips + 1):
        for col in trip_cols:
            c = f"{col}_{k}"
            if c in df_wide.columns:
                ordered.append(c)

    return df_wide[[c for c in ordered if c in df_wide.columns]]


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 — END-TO-END PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

def run_pipeline(survey_path: str,
                 od_path: str,
                 n_agents: int = 1000,
                 method: str = "stratified_pool",
                 train_ratio: float = 0.7,
                 random_seed: int = 42,
                 max_stay_points: int = 12,
                 initial_pool_factor: int = 5,
                 min_pool_size: int = 10_000,
                 topup_batch_size: int = 5_000,
                 topup_max_rounds: int = 20):
    """
    Full XR-model pipeline.

    Parameters
    ----------
    survey_path : path to household travel survey CSV
    od_path     : path to hourly OD matrix CSV
    n_agents    : number of synthetic agents to generate
    method      : one of:
                    "stratified_pool"  generate a large pool, bucket by final
                                       tour code, and sample to match the survey
                                       tour-code distribution
                    "none"             no global tour-code correction; raw forward
                                       simulation from the XR model
    train_ratio : share of persons used for fitting
    random_seed : random seed
    max_stay_points : maximum stay-points per generated agent
    initial_pool_factor : initial unconditional pool size = factor * n_agents
    min_pool_size : minimum initial pool size
    topup_batch_size : number of unconditional trajectories per top-up round
    topup_max_rounds : maximum number of top-up rounds

    Returns
    -------
    trips_df : survey-format long trip DataFrame
    synth_df : synthetic stay-point DataFrame
    model    : fitted XRModel
    """
    print("=" * 60)
    print("STEP 1: Loading data")
    survey = load_survey(survey_path)
    od = load_od(od_path)
    print(f"  Survey rows : {len(survey):,}")
    print(f"  OD rows     : {len(od):,}")

    print("\nSTEP 2: Building stay-point trajectories")
    sp_df = build_stay_points_from_survey(survey)
    n_persons = sp_df["ID"].nunique()
    print(f"  Persons: {n_persons:,}  Stay-points: {len(sp_df):,}")

    np.random.seed(random_seed)
    all_ids = sp_df["ID"].unique()
    train_size = int(train_ratio * len(all_ids))
    train_ids = np.random.choice(all_ids, size=train_size, replace=False)
    sp_train = sp_df[sp_df["ID"].isin(train_ids)]

    print(f"  Train persons: {len(train_ids):,}")

    print("\nSTEP 3: Fitting XR model")
    model = XRModel()
    model.fit(sp_train, od)

    if method == "stratified_pool":
        print(f"\nSTEP 4: Building survey target tour distribution")
        target_dist = build_survey_tour_distribution(survey)
        target_counts = allocate_target_counts(target_dist, n_agents)
        print(f"  Target tour codes: {len(target_counts):,}")
        print(f"  Target agents    : {sum(target_counts.values()):,}")

        print(f"\nSTEP 5: Generating {n_agents:,} synthetic travellers (method=stratified_pool)")
        synth_df = stratified_pool_sampling(
            model=model,
            target_counts=target_counts,
            initial_pool_factor=initial_pool_factor,
            min_pool_size=min_pool_size,
            max_stay_points=max_stay_points,
            topup_batch_size=topup_batch_size,
            topup_max_rounds=topup_max_rounds,
            rng_seed=random_seed
        )

    elif method == "none":
        print(f"\nSTEP 4: No tour-code stratification")
        print(f"\nSTEP 5: Generating {n_agents:,} synthetic travellers (method=none)")
        synth_df = model.generate_population(
            n_agents=n_agents,
            start_h=0,
            max_stay_points=max_stay_points
        )

    else:
        raise ValueError("Unknown method. Choose 'stratified_pool' or 'none'.")

    print("\nSTEP 6: Converting stay-points -> survey-format trips")
    trips_df = stay_points_to_trips(synth_df)
    n_agents_out = trips_df["ID"].nunique()

    print(f"  Agents with >=1 trip : {n_agents_out:,}")
    print(f"  Total trip records   : {len(trips_df):,}")
    print(f"  Avg trips per agent  : {len(trips_df) / max(n_agents_out, 1):.2f}")
    print("=" * 60)

    return trips_df, synth_df, model


# ─────────────────────────────────────────────────────────────────────────────
# USAGE EXAMPLE
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    ff = 0.5

    trips_df, synth_df, model = run_pipeline(
        survey_path=path_data + f"val_data_hts_trip_{ff}.csv",
        od_path=path_data + "val_data_pcm_trip.csv",
        n_agents=50_000,
        method="stratified_pool",  # "stratified_pool" | "none"
        train_ratio=0.7,
        random_seed=42,
        max_stay_points=12,
        initial_pool_factor=5,
        min_pool_size=10_000,
        topup_batch_size=5_000,
        topup_max_rounds=20,
    )

    trips_wide = trip_diaries_to_wide(trips_df)
    trips_wide.to_csv(path_result + f"val_sim_tour_xr_{ff}.csv", index=False)

    print(trips_wide.info())
    print(trips_wide["TRIP_MAX"].value_counts())

STEP 1: Loading data
  Survey rows : 19,551
  OD rows     : 39,247

STEP 2: Building stay-point trajectories
  Persons: 7,705  Stay-points: 27,256
  Train persons: 5,393

STEP 3: Fitting XR model
Fitting XR model...
  P1  : 1 conditions  (survey)
  PE1 : 207 conditions (survey)
  PZ  : 4373 conditions  (OD)
  PS  : 31477 conditions  (OD)
  PD  : 3345 conditions  (survey)
  XR  : 3346 conditions  (survey)
  Zones known: 309
Done.

STEP 4: Building survey target tour distribution
  Target tour codes: 210
  Target agents    : 50,000

STEP 5: Generating 50,000 synthetic travellers (method=stratified_pool)
Stratified pool sampling: target=50,000, initial_pool=250,000
  Initial bucket coverage: 168/210 target tour codes
  Top-up round 1: total shortfall=4,228, active codes=142
  Top-up round 2: total shortfall=4,100, active codes=140
  Top-up round 3: total shortfall=4,004, active codes=140
  Top-up round 4: total shortfall=3,902, active codes=139
  Top-up round 5: total shortfall=3,789, act

# TourExplicit

In [ ]:
"""
Tour Explicit (TX) Model for Individual Travel Demand Synthesis
===============================================================
Based on: Anda et al. (2021) "Synthesising digital twin travellers:
Individual travel demand from aggregated mobile phone data"

Cleaned hybrid version:
  - Removes OD-matrix blending from PZ
  - Keeps PS estimated from the OD matrix dataset
  - Estimates P1, PE1, PZ, PD, PT from household travel survey stay-points
  - Retains optional tour-network correction via rejection sampling (RS)
    or direct pool sampling (DPS)

Inputs:
  1. Household Travel Survey — individual trip chains used to learn:
       P1, PE1, PZ, PD, PT
  2. Hourly OD Matrix — used to learn:
       PS = P(Sk | Zk, Zk-1, Ek-1)

Output:
  Synthetic stay-point trajectories (Digital Twin Travellers), one full
  day per agent:
      [id, sp_idx, zone, x, y, start_h, end_h, duration_h, tour_code]

  and trip-format diaries matching the survey column schema.
"""

import numpy as np
import pandas as pd
from collections import defaultdict
import warnings

warnings.filterwarnings("ignore")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 — DATA LOADING & PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────

def load_survey(path: str) -> pd.DataFrame:
    """
    Load household travel survey and return one row per trip.

    Expected columns
    ----------------
    ID, TRIP_CNT, TRIP_MAX, TRIP_STARTTIME, TRIP_ENDTIME,
    ORIGIN_SUBZONE, ORIGIN_SUBZONE_X, ORIGIN_SUBZONE_Y,
    DESTINATION_SUBZONE, DESTINATION_SUBZONE_X, DESTINATION_SUBZONE_Y,
    HOME_SUBZONE, HOME_SUBZONE_X, HOME_SUBZONE_Y
    """
    df = pd.read_csv(path)
    df.columns = df.columns.str.upper().str.strip()
    return df


def load_od(path: str) -> pd.DataFrame:
    """
    Load hourly OD matrix.

    Expected columns
    ----------------
    TRIP_STARTTIME, TRIP_ENDTIME,
    ORIGIN_SUBZONE, ORIGIN_SUBZONE_X, ORIGIN_SUBZONE_Y,
    DESTINATION_SUBZONE, DESTINATION_SUBZONE_X, DESTINATION_SUBZONE_Y
    """
    df = pd.read_csv(path)
    df.columns = df.columns.str.upper().str.strip()
    return df


def to_hour(t) -> int:
    """
    Convert a time value to integer hour (0-23).

    Accepts:
      - int / float  : already hourly
      - 'HH:MM'
      - 'HH:MM:SS'
    """
    if pd.isna(t):
        return np.nan
    if isinstance(t, (int, float)):
        return int(t) % 24
    t = str(t).strip()
    parts = t.split(":")
    return int(parts[0]) % 24


def build_stay_points_from_survey(survey: pd.DataFrame) -> pd.DataFrame:
    """
    Convert trip-based survey into stay-point trajectories per person.

    Logic
    -----
      - Each person starts the day at HOME_SUBZONE (stay-point 0).
      - Each trip's DESTINATION_SUBZONE becomes the next stay-point.
      - Start time of stay-point i  = TRIP_ENDTIME of trip i
      - End time of stay-point i    = TRIP_STARTTIME of trip i+1
      - The last stay-point ends at hour 24
      - Stay-point 0 ends at the departure time of the first trip

    Returns
    -------
    DataFrame with columns:
      ID, sp_idx, zone, x, y, start_h, end_h, duration_h
    """
    survey = survey.copy()
    survey["TRIP_STARTTIME_H"] = survey["TRIP_STARTTIME"].apply(to_hour)
    survey["TRIP_ENDTIME_H"] = survey["TRIP_ENDTIME"].apply(to_hour)
    survey = survey.sort_values(["ID", "TRIP_CNT"]).reset_index(drop=True)

    records = []

    for pid, grp in survey.groupby("ID"):
        grp = grp.sort_values("TRIP_CNT").reset_index(drop=True)

        home_zone = grp.iloc[0]["HOME_SUBZONE"]
        home_x = grp.iloc[0]["HOME_SUBZONE_X"]
        home_y = grp.iloc[0]["HOME_SUBZONE_Y"]
        first_dep = grp.iloc[0]["TRIP_STARTTIME_H"]

        # Stay-point 0: home at start of day
        sp0_end = first_dep if pd.notna(first_dep) else 8
        records.append({
            "ID": pid,
            "sp_idx": 0,
            "zone": home_zone,
            "x": home_x,
            "y": home_y,
            "start_h": 0,
            "end_h": sp0_end,
            "duration_h": max(sp0_end, 0)
        })

        for i, row in grp.iterrows():
            dest_zone = row["DESTINATION_SUBZONE"]
            dest_x = row["DESTINATION_SUBZONE_X"]
            dest_y = row["DESTINATION_SUBZONE_Y"]
            arr_h = row["TRIP_ENDTIME_H"]

            if i < len(grp) - 1:
                dep_h = grp.iloc[i + 1]["TRIP_STARTTIME_H"]
            else:
                dep_h = 24

            if pd.isna(arr_h):
                arr_h = sp0_end
            if pd.isna(dep_h):
                dep_h = arr_h + 1

            dur = max(dep_h - arr_h, 0)

            records.append({
                "ID": pid,
                "sp_idx": int(row["TRIP_CNT"]),
                "zone": dest_zone,
                "x": dest_x,
                "y": dest_y,
                "start_h": arr_h,
                "end_h": dep_h,
                "duration_h": dur
            })

    return pd.DataFrame(records)


def distribution_from_series(s: pd.Series) -> dict:
    """
    Convert a categorical series into a probability dictionary.
    """
    probs = s.value_counts(normalize=True)
    return probs.to_dict()


def assign_tour_codes(sp_df: pd.DataFrame) -> pd.DataFrame:
    """
    Assign tour-code digit sequence to each person's stay-points.

    Each unique zone within a person's day gets a unique digit.
    The same zone revisited gets the same digit.

    Example
    -------
    [Home, Work, Home, Shop, Home] -> '01020'
    """
    sp_df = sp_df.copy()
    tour_codes = []

    for pid, grp in sp_df.groupby("ID"):
        grp = grp.sort_values("sp_idx")
        zone_to_digit = {}
        next_digit = 0
        digits = []

        for zone in grp["zone"]:
            if zone not in zone_to_digit:
                zone_to_digit[zone] = next_digit
                next_digit += 1
            digits.append(zone_to_digit[zone])

        for k in range(len(digits)):
            tour_codes.append("".join(str(d) for d in digits[:k + 1]))

    sp_df = sp_df.sort_values(["ID", "sp_idx"]).reset_index(drop=True)
    sp_df["tour_code"] = tour_codes
    return sp_df


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — CPD ESTIMATION (MLE via frequency counting)
# ─────────────────────────────────────────────────────────────────────────────

def _norm(d: dict) -> dict:
    """
    Normalize a frequency dict to a probability dict.
    """
    total = sum(d.values())
    if total == 0:
        return d
    return {k: v / total for k, v in d.items()}


class TXModel:
    """
    Tour Explicit (TX) Markov model.

    Conditional Probability Distributions (CPDs):
      P1  : P(Z1 | S1)               — initial zone given start hour
      PE1 : P(E1 | Z1, S1)           — end hour of first stay-point given zone & start
      PZ  : P(Zk | Zk-1, Ek-1, i)    — next zone given previous zone, hour, stay-point count
      PS  : P(Sk | Zk, Zk-1, Ek-1)   — start time given OD pair & previous end time
      PD  : P(Dk | Zk, Sk, tc)       — duration given zone, start time, and tour code
      PT  : P(T  | Zk, Ek, tc)       — next tour-code digit given zone, time, and tour code

    Estimation source:
      - Survey stay-points: P1, PE1, PZ, PD, PT
      - OD matrix dataset : PS
    """

    def __init__(self):
        self.P1 = {}
        self.PE1 = {}
        self.PZ = {}
        self.PS = {}
        self.PD = {}
        self.PT = {}
        self.zone_coords = {}

    # ── helpers ──────────────────────────────────────────────────────────────

    def _sample(self, prob_dict: dict):
        """
        Draw one sample from a probability dict {value: prob}.
        """
        if not prob_dict:
            return None

        keys = list(prob_dict.keys())
        probs = np.array([prob_dict[k] for k in keys], dtype=float)
        probs_sum = probs.sum()

        if probs_sum <= 0:
            return None

        probs /= probs_sum
        idx = np.searchsorted(np.cumsum(probs), np.random.random(), side="right")
        return keys[min(idx, len(keys) - 1)]

    def _fallback_zone(self, prev_zone, end_h):
        """
        Fallback marginal zone distribution over all stay-point counts for a
        given previous zone.
        """
        agg = defaultdict(float)

        for (pz, _, __), d in self.PZ.items():
            if pz == prev_zone:
                for z, p in d.items():
                    agg[z] += p

        if agg:
            return _norm(dict(agg))

        zones = list(self.zone_coords.keys())
        if not zones:
            return {}

        return {z: 1 / len(zones) for z in zones}

    def _fit_ps_from_od(self, od_df: pd.DataFrame):
        """
        Estimate PS = P(Sk | Zk, Zk-1, Ek-1) from the OD matrix dataset.

        Mapping from OD trips to PS:
          Zk-1   = ORIGIN_SUBZONE
          Zk     = DESTINATION_SUBZONE
          Ek-1   = TRIP_ENDTIME
          Sk     = TRIP_STARTTIME

        So we count:
          PS[(dest_zone, origin_zone, prev_end_h)][start_h] += 1
        """
        od_df = od_df.copy()
        od_df["TRIP_STARTTIME_H"] = od_df["TRIP_STARTTIME"].apply(to_hour)
        od_df["TRIP_ENDTIME_H"] = od_df["TRIP_ENDTIME"].apply(to_hour)

        freq_PS = defaultdict(lambda: defaultdict(float))

        for _, row in od_df.iterrows():
            origin_zone = row["ORIGIN_SUBZONE"]
            dest_zone = row["DESTINATION_SUBZONE"]
            start_h = row["TRIP_STARTTIME_H"]
            prev_end_h = row["TRIP_ENDTIME_H"]

            if pd.isna(start_h) or pd.isna(prev_end_h):
                continue

            start_h = int(start_h) % 24
            prev_end_h = int(prev_end_h) % 24

            freq_PS[(dest_zone, origin_zone, prev_end_h)][start_h] += 1

            # store coordinates when available
            if origin_zone not in self.zone_coords:
                ox = row.get("ORIGIN_SUBZONE_X", None)
                oy = row.get("ORIGIN_SUBZONE_Y", None)
                self.zone_coords[origin_zone] = (ox, oy)

            if dest_zone not in self.zone_coords:
                dx = row.get("DESTINATION_SUBZONE_X", None)
                dy = row.get("DESTINATION_SUBZONE_Y", None)
                self.zone_coords[dest_zone] = (dx, dy)

        self.PS = {k: _norm(dict(v)) for k, v in freq_PS.items()}

    # ── fitting ──────────────────────────────────────────────────────────────

    def fit(self, sp_df: pd.DataFrame, od_df: pd.DataFrame):
        """
        Estimate all CPDs.

        From survey stay-points:
          P1, PE1, PZ, PD, PT

        From OD matrix:
          PS
        """
        print("Fitting TX model...")
        sp_df = sp_df.copy()

        for _, row in sp_df.iterrows():
            self.zone_coords[row["zone"]] = (row["x"], row["y"])

        freq_P1 = defaultdict(lambda: defaultdict(float))
        freq_PE1 = defaultdict(lambda: defaultdict(float))
        freq_PZ = defaultdict(lambda: defaultdict(float))
        freq_PD = defaultdict(lambda: defaultdict(float))
        freq_PT = defaultdict(lambda: defaultdict(float))

        for pid, grp in sp_df.groupby("ID"):
            grp = grp.sort_values("sp_idx").reset_index(drop=True)

            for k in range(len(grp)):
                row = grp.iloc[k]
                zone = row["zone"]
                start_h = int(row["start_h"]) % 24
                end_h = int(row["end_h"]) % 24
                dur_h = min(int(max(row["duration_h"], 0)), 23)
                tc = row["tour_code"]
                sp_idx = int(row["sp_idx"])

                if k == 0:
                    freq_P1[start_h][zone] += 1
                    freq_PE1[(zone, start_h)][end_h] += 1
                else:
                    prev_row = grp.iloc[k - 1]
                    prev_zone = prev_row["zone"]
                    prev_end_h = int(prev_row["end_h"]) % 24
                    prev_tc = prev_row["tour_code"]

                    freq_PZ[(prev_zone, prev_end_h, sp_idx)][zone] += 1
                    freq_PT[(prev_zone, prev_end_h, prev_tc)][int(tc[-1])] += 1

                freq_PD[(zone, start_h, tc)][dur_h] += 1

        self.P1 = {k: _norm(dict(v)) for k, v in freq_P1.items()}
        self.PE1 = {k: _norm(dict(v)) for k, v in freq_PE1.items()}
        self.PZ = {k: _norm(dict(v)) for k, v in freq_PZ.items()}
        self.PD = {k: _norm(dict(v)) for k, v in freq_PD.items()}
        self.PT = {k: _norm(dict(v)) for k, v in freq_PT.items()}

        # PS is estimated only from OD matrix
        self._fit_ps_from_od(od_df)

        print(f"  P1  : {len(self.P1)} conditions")
        print(f"  PE1 : {len(self.PE1)} conditions")
        print(f"  PZ  : {len(self.PZ)} conditions")
        print(f"  PS  : {len(self.PS)} conditions  (from OD matrix)")
        print(f"  PD  : {len(self.PD)} conditions")
        print(f"  PT  : {len(self.PT)} conditions")
        print(f"  Zones known: {len(self.zone_coords)}")
        print("Done.")

    # ── generation ───────────────────────────────────────────────────────────

    def generate_one(self, agent_id: int, start_h: int = 0,
                     max_stay_points: int = 12) -> list:
        """
        Forward-sample one synthetic agent's daily stay-point trajectory.

        Returns
        -------
        List of stay-point dictionaries:
          {id, sp_idx, zone, x, y, start_h, end_h, duration_h, tour_code}
        """
        records = []
        current_h = start_h % 24

        p1_dist = self.P1.get(current_h) or (next(iter(self.P1.values())) if self.P1 else {})
        if not p1_dist:
            return records

        zone = self._sample(p1_dist)
        if zone is None:
            return records

        pe1_dist = self.PE1.get((zone, current_h), {})
        end_h = self._sample(pe1_dist) if pe1_dist else (current_h + 2) % 24

        tc = "0"
        zone_to_digit = {zone: 0}
        digit_to_zone = {0: zone}
        next_digit = 1
        visited = {zone}

        x, y = self.zone_coords.get(zone, (None, None))
        records.append({
            "id": agent_id,
            "sp_idx": 0,
            "zone": zone,
            "x": x,
            "y": y,
            "start_h": current_h,
            "end_h": end_h,
            "duration_h": max(end_h - current_h, 0),
            "tour_code": tc
        })

        prev_zone = zone
        prev_end_h = end_h % 24
        sp_idx = 1

        while sp_idx <= max_stay_points and prev_end_h < 24:
            # 1) Sample next tour-code digit
            pt_dist = self.PT.get((prev_zone, prev_end_h, tc))
            if not pt_dist:
                break

            next_d = self._sample(pt_dist)
            if next_d is None:
                break

            # 2) Resolve next zone
            if next_d not in digit_to_zone:
                pz_dist = self.PZ.get((prev_zone, prev_end_h, sp_idx)) \
                          or self._fallback_zone(prev_zone, prev_end_h)

                explore_dist = {z: p for z, p in pz_dist.items() if z not in visited}
                next_zone = self._sample(explore_dist or pz_dist)

                if next_zone is None:
                    break

                zone_to_digit[next_zone] = next_digit
                digit_to_zone[next_digit] = next_zone
                next_digit += 1
                visited.add(next_zone)
            else:
                next_zone = digit_to_zone[next_d]
                if next_zone is None:
                    break

            new_tc = tc + str(zone_to_digit[next_zone])

            # 3) Sample start time from PS estimated from OD matrix
            ps_dist = self.PS.get((next_zone, prev_zone, prev_end_h), {})
            start_h_sp = self._sample(ps_dist) if ps_dist else prev_end_h

            # 4) Sample duration
            sh_mod = start_h_sp % 24
            pd_dist = self.PD.get((next_zone, sh_mod, new_tc)) \
                      or self.PD.get(
                          (next_zone, sh_mod, new_tc[:2] if len(new_tc) > 2 else new_tc)
                      )

            dur = self._sample(pd_dist) if pd_dist else 1
            end_h_sp = start_h_sp + dur

            x, y = self.zone_coords.get(next_zone, (None, None))
            records.append({
                "id": agent_id,
                "sp_idx": sp_idx,
                "zone": next_zone,
                "x": x,
                "y": y,
                "start_h": start_h_sp,
                "end_h": end_h_sp,
                "duration_h": dur,
                "tour_code": new_tc
            })

            tc = new_tc
            prev_zone = next_zone
            prev_end_h = end_h_sp % 24
            sp_idx += 1

        return records

    def generate_population(self, n_agents: int, start_h: int = 0,
                            max_stay_points: int = 12) -> pd.DataFrame:
        """
        Generate a synthetic population of n_agents travellers.
        """
        print(f"Generating {n_agents:,} synthetic agents...")
        all_records = []
        tenth = max(1, n_agents // 10)

        for i in range(n_agents):
            all_records.extend(self.generate_one(i, start_h, max_stay_points))
            if (i + 1) % tenth == 0:
                print(f"  {i + 1:,}/{n_agents:,} done")

        print("Generation complete.")
        return pd.DataFrame(all_records)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 — REJECTION SAMPLING (tour network correction)
# ─────────────────────────────────────────────────────────────────────────────

def _extract_final_tour(recs: list) -> str:
    """
    Extract the final tour code from one generated agent.
    """
    return recs[-1]["tour_code"] if recs else ""


def rejection_sampling(model: TXModel,
                       target_tour_dist: dict,
                       n_accepted: int,
                       pool_size: int = None,
                       lambda_cap_pct: float = 99.0,
                       batch_size: int = 512,
                       max_stay_points: int = 12) -> pd.DataFrame:
    """
    Fast rejection sampling based on the target tour-code distribution.
    """
    if pool_size is None:
        pool_size = max(2 * n_accepted, 2_000)

    print(f"Rejection sampling: target={n_accepted:,}, pool={pool_size:,}...")

    pool_tours = [
        _extract_final_tour(model.generate_one(i, max_stay_points=max_stay_points))
        for i in range(pool_size)
    ]

    tour_counts = defaultdict(int)
    for t in pool_tours:
        tour_counts[t] += 1

    smooth_floor = 1.0 / (pool_size + len(target_tour_dist))
    g_tour = {t: c / pool_size for t, c in tour_counts.items()}

    ratios = [
        target_tour_dist[tc] / g_tour.get(tc, smooth_floor)
        for tc in target_tour_dist
    ]
    lambda_val = max(float(np.percentile(ratios, lambda_cap_pct)), 1.0) if ratios else 1.0

    print(f"  λ = {lambda_val:.2f}  (capped at {lambda_cap_pct}th pct, {len(ratios)} tours in target)")

    accepted_recs = []
    cand_id = 0
    attempts = 0
    log_every = max(batch_size * 20, 10_000)

    while len(accepted_recs) < n_accepted:
        batch = [
            model.generate_one(cand_id + i, max_stay_points=max_stay_points)
            for i in range(batch_size)
        ]

        tours = [_extract_final_tour(r) for r in batch]
        f_arr = np.array([target_tour_dist.get(t, 0.0) for t in tours])
        g_arr = np.array([g_tour.get(t, smooth_floor) for t in tours])
        u_arr = np.random.uniform(0, lambda_val * g_arr)
        mask = u_arr <= f_arr

        for accept, recs in zip(mask, batch):
            if accept and len(accepted_recs) < n_accepted:
                accepted_recs.append(recs)

        cand_id += batch_size
        attempts += batch_size

        if attempts % log_every == 0:
            print(f"  accepted {len(accepted_recs):,}/{n_accepted:,}  (attempts: {attempts:,})")

    print(f"  Done. Acceptance rate: {n_accepted / attempts:.4f}  (attempts: {attempts:,})")

    all_rows = []
    for new_id, recs in enumerate(accepted_recs):
        for r in recs:
            row = dict(r)
            row["id"] = new_id
            all_rows.append(row)

    return pd.DataFrame(all_rows)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3b — DIRECT POOL SAMPLING (DPS)
# ─────────────────────────────────────────────────────────────────────────────

def direct_pool_sampling(model: TXModel,
                         target_tour_dist: dict,
                         n_accepted: int,
                         pool_size: int = None,
                         max_stay_points: int = 12) -> pd.DataFrame:
    """
    Direct Pool Sampling (DPS).
    """
    if pool_size is None:
        pool_size = max(5 * n_accepted, 10_000)

    print(f"Direct Pool Sampling: pool={pool_size:,}, target={n_accepted:,}...")

    pool_recs = []
    pool_tours = []

    for i in range(pool_size):
        recs = model.generate_one(i, max_stay_points=max_stay_points)
        pool_recs.append(recs)
        pool_tours.append(_extract_final_tour(recs))

    weights = np.array([target_tour_dist.get(t, 0.0) for t in pool_tours], dtype=float)
    total_w = weights.sum()

    n_covered_agents = int((weights > 0).sum())
    n_target_codes = len(target_tour_dist)
    coverage = n_covered_agents / max(n_target_codes, 1) * 100

    print(f"  Pool tour coverage : {n_covered_agents}/{n_target_codes} target codes ({coverage:.1f}%)")

    if total_w == 0:
        raise ValueError(
            "No pool agent matches any target tour code. "
            "Increase pool_size or check that target_tour_dist was built "
            "from the same survey used to fit the model."
        )

    probs = weights / total_w
    chosen_idx = np.random.choice(pool_size, size=n_accepted, replace=True, p=probs)

    n_unique = len(set(chosen_idx))
    duplication_rt = 1.0 - n_unique / n_accepted
    print(f"  Unique agents      : {n_unique:,}/{n_accepted:,}  (duplication rate: {duplication_rt:.3f})")

    if duplication_rt > 0.2:
        print(f"  WARNING: high duplication ({duplication_rt:.1%}). Consider increasing pool_size (currently {pool_size:,}).")

    all_rows = []
    for new_id, idx in enumerate(chosen_idx):
        for r in pool_recs[idx]:
            row = dict(r)
            row["id"] = new_id
            all_rows.append(row)

    print("  Done.")
    return pd.DataFrame(all_rows)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 4 — OUTPUT CONVERSION: stay-points → survey-format trips
# ─────────────────────────────────────────────────────────────────────────────

def stay_points_to_trips(synth_df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert synthetic stay-point trajectories into survey-format trip records.
    """
    synth_df = synth_df.sort_values(["id", "sp_idx"]).reset_index(drop=True)
    records = []

    for agent_id, grp in synth_df.groupby("id"):
        grp = grp.sort_values("sp_idx").reset_index(drop=True)
        n = len(grp)

        if n < 2:
            continue

        trip_max = n - 1

        for k in range(1, n):
            o = grp.iloc[k - 1]
            d = grp.iloc[k]

            records.append({
                "ID": agent_id,
                "TRIP_CNT": k,
                "TRIP_MAX": trip_max,
                "TRIP_STARTTIME": o["end_h"],
                "TRIP_ENDTIME": d["start_h"],
                "ORIGIN_SUBZONE": o["zone"],
                "ORIGIN_SUBZONE_X": o["x"],
                "ORIGIN_SUBZONE_Y": o["y"],
                "DESTINATION_SUBZONE": d["zone"],
                "DESTINATION_SUBZONE_X": d["x"],
                "DESTINATION_SUBZONE_Y": d["y"],
            })

    return pd.DataFrame(records, columns=[
        "ID", "TRIP_CNT", "TRIP_MAX",
        "TRIP_STARTTIME", "TRIP_ENDTIME",
        "ORIGIN_SUBZONE", "ORIGIN_SUBZONE_X", "ORIGIN_SUBZONE_Y",
        "DESTINATION_SUBZONE", "DESTINATION_SUBZONE_X", "DESTINATION_SUBZONE_Y",
    ])


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 — WIDE-FORMAT CONVERSION
# ─────────────────────────────────────────────────────────────────────────────

def trip_diaries_to_wide(df_trip: pd.DataFrame) -> pd.DataFrame:
    """
    Convert long-format trip diaries to wide tour format.
    """
    trip_cols = [
        "ORIGIN_SUBZONE", "ORIGIN_SUBZONE_X", "ORIGIN_SUBZONE_Y",
        "DESTINATION_SUBZONE", "DESTINATION_SUBZONE_X", "DESTINATION_SUBZONE_Y",
        "TRIP_STARTTIME", "TRIP_ENDTIME",
    ]

    df = df_trip.copy()
    df["ID"] = df["ID"].astype(str)

    df_wide = df.pivot_table(
        index="ID",
        columns="TRIP_CNT",
        values=trip_cols,
        aggfunc="first"
    )
    df_wide.columns = [f"{col}_{int(k)}" for col, k in df_wide.columns]
    df_wide = df_wide.reset_index()

    trip_max_map = df.groupby("ID")["TRIP_MAX"].max()
    df_wide = df_wide.merge(trip_max_map.rename("TRIP_MAX"), on="ID")

    max_trips = int(df["TRIP_MAX"].max()) if not df.empty else 0
    ordered = ["ID", "TRIP_MAX"]

    for k in range(1, max_trips + 1):
        for col in trip_cols:
            c = f"{col}_{k}"
            if c in df_wide.columns:
                ordered.append(c)

    return df_wide[[c for c in ordered if c in df_wide.columns]]


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 — END-TO-END PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

def run_pipeline(survey_path: str,
                 od_path: str,
                 n_agents: int = 1000,
                 method: str = "rs",
                 train_ratio: float = 0.7,
                 random_seed: int = 42):
    """
    Full TX model pipeline.

    Parameters
    ----------
    survey_path : path to household travel survey CSV
    od_path     : path to hourly OD matrix CSV
    n_agents    : number of synthetic agents to generate
    method      : one of:
                    "rs"   rejection sampling
                    "dps"  direct pool sampling
                    "none" no tour correction; raw forward sampling
    train_ratio : share of persons used for model fitting
    random_seed : random seed for train/test split and generation reproducibility

    Returns
    -------
    trips_df : survey-format long trip DataFrame
    synth_df : synthetic stay-point DataFrame
    model    : fitted TXModel
    """
    print("=" * 60)
    print("STEP 1: Loading data")
    survey = load_survey(survey_path)
    od = load_od(od_path)
    print(f"  Survey rows : {len(survey):,}")
    print(f"  OD rows     : {len(od):,}")

    print("\nSTEP 2: Building stay-point trajectories")
    sp_df = assign_tour_codes(build_stay_points_from_survey(survey))
    n_persons = sp_df["ID"].nunique()
    print(f"  Persons: {n_persons:,}  Stay-points: {len(sp_df):,}")

    np.random.seed(random_seed)
    all_ids = sp_df["ID"].unique()
    train_size = int(train_ratio * len(all_ids))
    train_ids = np.random.choice(all_ids, size=train_size, replace=False)
    test_ids = np.setdiff1d(all_ids, train_ids)

    sp_train = sp_df[sp_df["ID"].isin(train_ids)]
    sp_test = sp_df[sp_df["ID"].isin(test_ids)]

    print(f"  Train: {len(train_ids):,}  Test: {len(test_ids):,}")

    print("\nSTEP 3: Fitting TX model")
    model = TXModel()
    model.fit(sp_train, od)

    print(f"\nSTEP 4: Generating {n_agents:,} synthetic travellers  (method={method})")
    target_dist = distribution_from_series(
        sp_test.groupby("ID")["tour_code"].last()
    )

    if method == "rs":
        synth_df = rejection_sampling(model, target_dist, n_accepted=n_agents)
    elif method == "dps":
        synth_df = direct_pool_sampling(model, target_dist, n_accepted=n_agents)
    elif method == "none":
        synth_df = model.generate_population(n_agents)
    else:
        raise ValueError(f"Unknown method '{method}'. Choose 'rs', 'dps', or 'none'.")

    print("\nSTEP 5: Converting stay-points -> survey-format trips")
    trips_df = stay_points_to_trips(synth_df)
    n_agents_out = trips_df["ID"].nunique()

    print(f"  Agents with >=1 trip : {n_agents_out:,}")
    print(f"  Total trip records   : {len(trips_df):,}")
    print(f"  Avg trips per agent  : {len(trips_df) / max(n_agents_out, 1):.2f}")
    print("=" * 60)

    return trips_df, synth_df, model


# ─────────────────────────────────────────────────────────────────────────────
# USAGE EXAMPLE
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    ff = 0.1

    trips_df, synth_df, model = run_pipeline(
        survey_path=path_data + f"val_data_hts_trip_{ff}.csv",
        od_path=path_data + "val_data_pcm_trip.csv",
        n_agents=30_000,
        method="rs",   # "rs" | "dps" | "none"
    )

    trips_wide = trip_diaries_to_wide(trips_df)
    trips_wide.to_csv(path_result + f"val_sim_tour_tx_{ff}.csv", index=False)

    print(trips_wide.info())
    print(trips_wide["TRIP_MAX"].value_counts())

STEP 1: Loading data
  Survey rows : 3,950
  OD rows     : 39,247

STEP 2: Building stay-point trajectories
  Persons: 1,541  Stay-points: 5,491
  Train: 1,078  Test: 463

STEP 3: Fitting TX model
Fitting TX model...
  P1  : 1 conditions
  PE1 : 174 conditions
  PZ  : 1992 conditions
  PS  : 31508 conditions  (from OD matrix)
  PD  : 2397 conditions
  PT  : 2047 conditions
  Zones known: 309
Done.

STEP 4: Generating 30,000 synthetic travellers  (method=rs)
Rejection sampling: target=30,000, pool=60,000...
  λ = 414.97  (capped at 99.0th pct, 41 tours in target)
  accepted 18/30,000  (attempts: 10,240)
  accepted 43/30,000  (attempts: 20,480)
  accepted 71/30,000  (attempts: 30,720)
  accepted 95/30,000  (attempts: 40,960)
  accepted 122/30,000  (attempts: 51,200)
  accepted 134/30,000  (attempts: 61,440)
  accepted 158/30,000  (attempts: 71,680)
  accepted 178/30,000  (attempts: 81,920)
  accepted 211/30,000  (attempts: 92,160)
  accepted 238/30,000  (attempts: 102,400)
  accepted 262

In [ ]:
"""
Stable

Tour Explicit (TX) Model for Individual Travel Demand Synthesis
===============================================================
Based on: Anda et al. (2021) "Synthesising digital twin travellers:
Individual travel demand from aggregated mobile phone data"

Inputs:
  1. Household Travel Survey  — individual trip chains (used to learn CPDs)
  2. Hourly OD Matrix         — aggregated trip flows (used to supplement
                                 spatial transition probabilities)

Output:
  Synthetic stay-point trajectories (Digital Twin Travellers), one full
  day per agent: [id, sp_idx, zone, x, y, start_h, end_h, duration_h, tour_code]
  …and trip-format diaries matching the survey column schema.
"""

import numpy as np
import pandas as pd
from collections import defaultdict
import warnings
warnings.filterwarnings("ignore")


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 1 — DATA LOADING & PREPROCESSING
# ─────────────────────────────────────────────────────────────────────────────

def load_survey(path: str) -> pd.DataFrame:
    """
    Load household travel survey and return one row per trip.
    Expects columns: ID, TRIP_CNT, TRIP_MAX, TRIP_STARTTIME, TRIP_ENDTIME,
                     ORIGIN_SUBZONE, ORIGIN_SUBZONE_X, ORIGIN_SUBZONE_Y,
                     DESTINATION_SUBZONE, DESTINATION_SUBZONE_X,
                     DESTINATION_SUBZONE_Y, HOME_SUBZONE,
                     HOME_SUBZONE_X, HOME_SUBZONE_Y
    """
    df = pd.read_csv(path)
    df.columns = df.columns.str.upper().str.strip()
    return df


def load_od(path: str) -> pd.DataFrame:
    """
    Load hourly OD matrix.
    Expects columns: TRIP_STARTTIME, TRIP_ENDTIME,
                     ORIGIN_SUBZONE, ORIGIN_SUBZONE_X, ORIGIN_SUBZONE_Y,
                     DESTINATION_SUBZONE, DESTINATION_SUBZONE_X,
                     DESTINATION_SUBZONE_Y
    """
    df = pd.read_csv(path)
    df.columns = df.columns.str.upper().str.strip()
    return df


def to_hour(t) -> int:
    """Convert a time value to integer hour (0-23).
    Accepts: int/float (already hours), 'HH:MM' string, or 'HH:MM:SS' string.
    """
    if pd.isna(t):
        return np.nan
    if isinstance(t, (int, float)):
        return int(t) % 24
    t = str(t).strip()
    parts = t.split(":")
    return int(parts[0]) % 24


def build_stay_points_from_survey(survey: pd.DataFrame) -> pd.DataFrame:
    """
    Convert trip-based survey into stay-point trajectories per person.

    Logic:
      - Each person starts the day at HOME_SUBZONE (stay-point 0).
      - Each trip's DESTINATION_SUBZONE becomes the next stay-point.
      - Start time of stay-point i  = TRIP_ENDTIME of trip i
        (person arrived, activity begins).
      - End time of stay-point i    = TRIP_STARTTIME of trip i+1
        (person departs for next trip).
      - The last stay-point of the day ends at midnight (hour 24).
      - Stay-point 0 (home at start of day) ends at TRIP_STARTTIME of trip 1.

    Returns DataFrame with columns:
      ID, sp_idx, zone, x, y, start_h, end_h, duration_h
    """
    survey = survey.copy()
    survey["TRIP_STARTTIME_H"] = survey["TRIP_STARTTIME"].apply(to_hour)
    survey["TRIP_ENDTIME_H"]   = survey["TRIP_ENDTIME"].apply(to_hour)
    survey = survey.sort_values(["ID", "TRIP_CNT"]).reset_index(drop=True)

    records = []
    for pid, grp in survey.groupby("ID"):
        grp = grp.sort_values("TRIP_CNT").reset_index(drop=True)
        home_zone = grp.iloc[0]["HOME_SUBZONE"]
        home_x    = grp.iloc[0]["HOME_SUBZONE_X"]
        home_y    = grp.iloc[0]["HOME_SUBZONE_Y"]
        first_dep = grp.iloc[0]["TRIP_STARTTIME_H"]

        # Stay-point 0: home at start of day
        sp0_end = first_dep if pd.notna(first_dep) else 8
        records.append({
            "ID": pid, "sp_idx": 0,
            "zone": home_zone, "x": home_x, "y": home_y,
            "start_h": 0, "end_h": sp0_end,
            "duration_h": max(sp0_end, 0)
        })

        for i, row in grp.iterrows():
            dest_zone = row["DESTINATION_SUBZONE"]
            dest_x    = row["DESTINATION_SUBZONE_X"]
            dest_y    = row["DESTINATION_SUBZONE_Y"]
            arr_h     = row["TRIP_ENDTIME_H"]

            # Determine when this activity ends
            if i < len(grp) - 1:
                dep_h = grp.iloc[i + 1]["TRIP_STARTTIME_H"]
            else:
                dep_h = 24  # last activity ends at midnight

            if pd.isna(arr_h):
                arr_h = sp0_end
            if pd.isna(dep_h):
                dep_h = arr_h + 1

            dur = max(dep_h - arr_h, 0)
            records.append({
                "ID": pid, "sp_idx": int(row["TRIP_CNT"]),
                "zone": dest_zone, "x": dest_x, "y": dest_y,
                "start_h": arr_h, "end_h": dep_h,
                "duration_h": dur
            })

    return pd.DataFrame(records)


def assign_tour_codes(sp_df: pd.DataFrame) -> pd.DataFrame:
    """
    Assign tour-code digit sequence to each person's stay-points.
    Each unique zone within a person's day gets a unique digit;
    the same zone revisited gets the same digit.
    e.g. [Home, Work, Home, Shop, Home] → '01020'
    """
    sp_df = sp_df.copy()
    tour_codes = []
    for pid, grp in sp_df.groupby("ID"):
        grp = grp.sort_values("sp_idx")
        zone_to_digit = {}
        next_digit = 0
        digits = []
        for zone in grp["zone"]:
            if zone not in zone_to_digit:
                zone_to_digit[zone] = next_digit
                next_digit += 1
            digits.append(zone_to_digit[zone])
        for k in range(len(digits)):
            tour_codes.append("".join(str(d) for d in digits[:k+1]))

    sp_df = sp_df.sort_values(["ID", "sp_idx"]).reset_index(drop=True)
    sp_df["tour_code"] = tour_codes
    return sp_df


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 2 — CPD ESTIMATION (MLE via frequency counting)
# ─────────────────────────────────────────────────────────────────────────────

def _norm(d: dict) -> dict:
    """Normalise a frequency dict to a probability dict."""
    total = sum(d.values())
    if total == 0:
        return d
    return {k: v / total for k, v in d.items()}


class TXModel:
    """
    Tour Explicit (TX) Markov model.

    Conditional Probability Distributions (CPDs) estimated from survey:
      P1  : P(Z1 | S1)               — initial zone given start hour
      PE1 : P(E1 | Z1, S1)          — first departure hour given zone & start
      PZ  : P(Zk | Zk-1, Ek-1, i)  — next zone given prev zone, hour, sp count
      PS  : P(Sk | Zk, Zk-1, Ek-1) — start time given OD pair & prev end time
      PD  : P(Dk | Zk, Sk, tc)      — duration given zone, start time, tour code
      PT  : P(T  | Zk, Ek, tc)      — next tour-code digit given zone, time, tc

    The OD matrix is used to augment/smooth PZ where survey data is sparse.
    """

    def __init__(self, od_weight: float = 0.0):
        self.od_weight  = od_weight
        self.P1         = {}
        self.PE1        = {}
        self.PZ         = {}
        self.PS         = {}
        self.PD         = {}
        self.PT         = {}
        self.zone_coords = {}

    # ── helpers ──────────────────────────────────────────────────────────────

    def _sample(self, prob_dict: dict):
        """Draw one sample from a probability dict {value: prob}."""
        if not prob_dict:
            return None
        keys  = list(prob_dict.keys())
        probs = np.array([prob_dict[k] for k in keys], dtype=float)
        probs /= probs.sum()
        idx = np.searchsorted(np.cumsum(probs), np.random.random(), side='right')
        return keys[min(idx, len(keys) - 1)]

    def _fallback_zone(self, prev_zone, end_h):
        """Marginal zone distribution over all sp_idx for prev_zone."""
        agg = defaultdict(float)
        for (pz, _, __), d in self.PZ.items():
            if pz == prev_zone:
                for z, p in d.items():
                    agg[z] += p
        if agg:
            return _norm(dict(agg))
        zones = list(self.zone_coords.keys())
        return {z: 1 / len(zones) for z in zones}

    # ── fitting ──────────────────────────────────────────────────────────────

    def fit(self, sp_df: pd.DataFrame, od_df: pd.DataFrame = None):
        """
        Estimate all CPDs from stay-point data (and optionally OD matrix).

        Parameters
        ----------
        sp_df  : stay-point DataFrame with 'tour_code' column
        od_df  : optional OD matrix DataFrame
        """
        print("Fitting TX model...")
        sp_df = sp_df.copy()

        for _, row in sp_df.iterrows():
            self.zone_coords[row["zone"]] = (row["x"], row["y"])

        freq_P1  = defaultdict(lambda: defaultdict(float))
        freq_PE1 = defaultdict(lambda: defaultdict(float))
        freq_PZ  = defaultdict(lambda: defaultdict(float))
        freq_PS  = defaultdict(lambda: defaultdict(float))
        freq_PD  = defaultdict(lambda: defaultdict(float))
        freq_PT  = defaultdict(lambda: defaultdict(float))

        for pid, grp in sp_df.groupby("ID"):
            grp = grp.sort_values("sp_idx").reset_index(drop=True)

            for k in range(len(grp)):
                row      = grp.iloc[k]
                zone     = row["zone"]
                start_h  = int(row["start_h"]) % 24
                end_h    = int(row["end_h"])   % 24
                dur_h    = min(int(max(row["duration_h"], 0)), 23)
                tc       = row["tour_code"]
                sp_idx   = int(row["sp_idx"])

                if k == 0:
                    freq_P1[start_h][zone]            += 1
                    freq_PE1[(zone, start_h)][end_h]  += 1
                else:
                    prev_row   = grp.iloc[k - 1]
                    prev_zone  = prev_row["zone"]
                    prev_end_h = int(prev_row["end_h"]) % 24
                    prev_tc    = prev_row["tour_code"]

                    freq_PZ[(prev_zone, prev_end_h, sp_idx)][zone]          += 1
                    freq_PS[(zone, prev_zone, prev_end_h)][start_h]         += 1
                    freq_PT[(prev_zone, prev_end_h, prev_tc)][int(tc[-1])]  += 1

                freq_PD[(zone, start_h, tc)][dur_h] += 1

        self.P1  = {k: _norm(dict(v)) for k, v in freq_P1.items()}
        self.PE1 = {k: _norm(dict(v)) for k, v in freq_PE1.items()}
        self.PZ  = {k: _norm(dict(v)) for k, v in freq_PZ.items()}
        self.PS  = {k: _norm(dict(v)) for k, v in freq_PS.items()}
        self.PD  = {k: _norm(dict(v)) for k, v in freq_PD.items()}
        self.PT  = {k: _norm(dict(v)) for k, v in freq_PT.items()}

        if od_df is not None and self.od_weight > 0:
            self._blend_od(od_df)

        print(f"  P1  : {len(self.P1)} conditions")
        print(f"  PE1 : {len(self.PE1)} conditions")
        print(f"  PZ  : {len(self.PZ)} conditions")
        print(f"  PS  : {len(self.PS)} conditions")
        print(f"  PD  : {len(self.PD)} conditions")
        print(f"  PT  : {len(self.PT)} conditions")
        print(f"  Zones known: {len(self.zone_coords)}")
        print("Done.")

    def _blend_od(self, od_df: pd.DataFrame):
        """Blend OD matrix flows into PZ to improve spatial coverage."""
        od_df = od_df.copy()
        od_df["START_H"] = od_df["TRIP_STARTTIME"].apply(to_hour)

        od_freq = defaultdict(lambda: defaultdict(float))
        for _, row in od_df.iterrows():
            oh = int(row["START_H"]) % 24
            od_freq[(row["ORIGIN_SUBZONE"], oh)][row["DESTINATION_SUBZONE"]] += 1
        od_prob = {k: _norm(dict(v)) for k, v in od_freq.items()}

        new_PZ = {}
        for (pz, peh, si), survey_dist in self.PZ.items():
            od_key = (pz, peh)
            if od_key in od_prob:
                od_dist   = od_prob[od_key]
                all_zones = set(survey_dist) | set(od_dist)
                blended   = {
                    z: (1 - self.od_weight) * survey_dist.get(z, 0.0)
                       + self.od_weight     * od_dist.get(z, 0.0)
                    for z in all_zones
                }
                new_PZ[(pz, peh, si)] = _norm(blended)
            else:
                new_PZ[(pz, peh, si)] = survey_dist
        self.PZ = new_PZ

    # ── generation ───────────────────────────────────────────────────────────

    def generate_one(self, agent_id: int, start_h: int = 0,
                     max_stay_points: int = 12) -> list:
        """
        Forward-sample one synthetic agent's daily stay-point trajectory.
        Returns a list of stay-point dicts:
          {id, sp_idx, zone, x, y, start_h, end_h, duration_h, tour_code}
        """
        records   = []
        current_h = start_h % 24

        p1_dist = self.P1.get(current_h) or (
            next(iter(self.P1.values())) if self.P1 else {})
        if not p1_dist:
            return records

        zone  = self._sample(p1_dist)
        if zone is None:
            return records

        pe1_dist = self.PE1.get((zone, current_h), {})
        end_h    = self._sample(pe1_dist) if pe1_dist else (current_h + 2) % 24

        tc            = "0"
        zone_to_digit = {zone: 0}
        digit_to_zone = {0: zone}
        next_digit    = 1
        visited       = {zone}

        x, y = self.zone_coords.get(zone, (None, None))
        records.append({
            "id": agent_id, "sp_idx": 0,
            "zone": zone, "x": x, "y": y,
            "start_h": current_h, "end_h": end_h,
            "duration_h": max(end_h - current_h, 0),
            "tour_code": tc
        })

        prev_zone  = zone
        prev_end_h = end_h % 24
        sp_idx     = 1

        while sp_idx <= max_stay_points and prev_end_h < 24:
            # 1. Sample next tour-code digit
            pt_dist = self.PT.get((prev_zone, prev_end_h, tc))
            if not pt_dist:
                break
            next_d = self._sample(pt_dist)

            # 2. Resolve next zone
            if next_d not in digit_to_zone:
                pz_dist = self.PZ.get((prev_zone, prev_end_h, sp_idx)) \
                          or self._fallback_zone(prev_zone, prev_end_h)
                explore_dist = {z: p for z, p in pz_dist.items()
                                if z not in visited}
                next_zone = self._sample(explore_dist or pz_dist)
                if next_zone is None:
                    break
                zone_to_digit[next_zone]  = next_digit
                digit_to_zone[next_digit] = next_zone
                next_digit += 1
                visited.add(next_zone)
            else:
                next_zone = digit_to_zone[next_d]
                if next_zone is None:
                    break

            new_tc = tc + str(zone_to_digit[next_zone])

            # 3. Sample start time
            ps_dist    = self.PS.get((next_zone, prev_zone, prev_end_h), {})
            start_h_sp = self._sample(ps_dist) if ps_dist else prev_end_h

            # 4. Sample duration
            sh_mod  = start_h_sp % 24
            pd_dist = self.PD.get((next_zone, sh_mod, new_tc)) \
                      or self.PD.get((next_zone, sh_mod,
                                      new_tc[:2] if len(new_tc) > 2 else new_tc))
            dur      = self._sample(pd_dist) if pd_dist else 1
            end_h_sp = start_h_sp + dur

            x, y = self.zone_coords.get(next_zone, (None, None))
            records.append({
                "id": agent_id, "sp_idx": sp_idx,
                "zone": next_zone, "x": x, "y": y,
                "start_h": start_h_sp, "end_h": end_h_sp,
                "duration_h": dur, "tour_code": new_tc
            })

            tc         = new_tc
            prev_zone  = next_zone
            prev_end_h = end_h_sp % 24
            sp_idx    += 1

        return records

    def generate_population(self, n_agents: int, start_h: int = 0,
                             max_stay_points: int = 12) -> pd.DataFrame:
        """Generate a synthetic population of n_agents travellers."""
        print(f"Generating {n_agents:,} synthetic agents...")
        all_records = []
        tenth = max(1, n_agents // 10)
        for i in range(n_agents):
            all_records.extend(self.generate_one(i, start_h, max_stay_points))
            if (i + 1) % tenth == 0:
                print(f"  {i+1:,}/{n_agents:,} done")
        print("Generation complete.")
        return pd.DataFrame(all_records)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3 — REJECTION SAMPLING (tour network correction)
# ─────────────────────────────────────────────────────────────────────────────
#
# Why the original was slow — and what changed
# ─────────────────────────────────────────────
# Problem 1 — Enormous warm-up pool
#   Original: pool_size = max(20 * n_accepted, 10_000)
#   With n_accepted=50 000 → 1 000 000 generate_one calls just to estimate
#   g_tour BEFORE rejection even starts.
#   Fix: pool_size defaults to max(2 * n_accepted, 2_000), cutting the
#   warm-up cost by ~10×. The rough g_tour estimate is sufficient for a
#   valid envelope.
#
# Problem 2 — λ blown up by tours absent from the pool
#   Any target tour not seen in the pool gets g_p = 1e-9, making
#   λ = f_p / 1e-9 astronomically large and causing nearly every
#   candidate to be rejected.
#   Fix: unseen tours get a Laplace-smoothed floor probability instead of
#   1e-9. λ is then capped at the ``lambda_cap_pct`` percentile of the
#   observed ratio distribution so a single rare tour cannot dominate.
#
# Problem 3 — Per-sample Python overhead in the rejection loop
#   The original loop called generate_one, tested one sample, and looped —
#   pure Python overhead for every single candidate.
#   Fix: candidates are generated in batches of ``batch_size``; the
#   accept/reject decision is vectorised with numpy across the full batch,
#   removing per-sample interpreter overhead from the test itself.
#
# Problem 4 — Redundant id reassignment inside the hot loop
#   Original: for r in recs: r["id"] = len(accepted) executed O(stay-points)
#   writes per candidate inside the tight loop.
#   Fix: agent IDs are assigned once after all accepted agents are collected.


def rejection_sampling(model: TXModel,
                       target_tour_dist: dict,
                       n_accepted: int,
                       pool_size: int = None,
                       lambda_cap_pct: float = 99.0,
                       batch_size: int = 512,
                       max_stay_points: int = 12) -> pd.DataFrame:
    """
    Fast rejection-sampling (Algorithm 1, Anda et al. 2021).

    Parameters
    ----------
    model             : fitted TXModel
    target_tour_dist  : {tour_code: probability}  empirical target distribution
    n_accepted        : desired number of accepted synthetic agents
    pool_size         : warm-up pool size (default: max(2*n_accepted, 2_000))
    lambda_cap_pct    : percentile at which to cap λ — prevents a single rare
                        tour from inflating λ to an extreme value (default 99)
    batch_size        : candidates generated per iteration for vectorised
                        accept/reject (default 512)
    max_stay_points   : max stay-points per agent

    Returns
    -------
    pd.DataFrame of accepted synthetic stay-point trajectories
    """
    # FIX 1: small warm-up pool
    if pool_size is None:
        pool_size = max(2 * n_accepted, 2_000)

    print(f"Rejection sampling: target={n_accepted:,}, pool={pool_size:,}...")

    # ── Step 1: warm-up pool → empirical proposal g_tour ─────────────────
    pool_tours = [
        _extract_final_tour(model.generate_one(i, max_stay_points=max_stay_points))
        for i in range(pool_size)
    ]
    tour_counts = defaultdict(int)
    for t in pool_tours:
        tour_counts[t] += 1

    # FIX 2a: Laplace-smoothed floor for tours absent from pool
    smooth_floor = 1.0 / (pool_size + len(target_tour_dist))
    g_tour = {t: c / pool_size for t, c in tour_counts.items()}

    # FIX 2b: cap λ at a percentile of observed ratios
    ratios = [
        target_tour_dist[tc] / g_tour.get(tc, smooth_floor)
        for tc in target_tour_dist
    ]
    lambda_val = max(float(np.percentile(ratios, lambda_cap_pct)), 1.0) \
        if ratios else 1.0
    print(f"  λ = {lambda_val:.2f}  "
          f"(capped at {lambda_cap_pct}th pct, {len(ratios)} tours in target)")

    # ── Step 2: batched rejection sampling ───────────────────────────────
    # FIX 3: generate in batches; vectorise accept/reject with numpy
    # FIX 4: assign IDs after collection, not inside the hot loop
    accepted_recs = []   # list of record-lists, one per accepted agent
    cand_id       = 0
    attempts      = 0
    log_every     = max(batch_size * 20, 10_000)

    while len(accepted_recs) < n_accepted:
        batch      = [model.generate_one(cand_id + i,
                                         max_stay_points=max_stay_points)
                      for i in range(batch_size)]
        tours      = [_extract_final_tour(r) for r in batch]
        f_arr      = np.array([target_tour_dist.get(t, 0.0)  for t in tours])
        g_arr      = np.array([g_tour.get(t, smooth_floor)   for t in tours])
        u_arr      = np.random.uniform(0, lambda_val * g_arr)
        mask       = u_arr <= f_arr

        for accept, recs in zip(mask, batch):
            if accept and len(accepted_recs) < n_accepted:
                accepted_recs.append(recs)

        cand_id  += batch_size
        attempts += batch_size
        if attempts % log_every == 0:
            print(f"  accepted {len(accepted_recs):,}/{n_accepted:,}  "
                  f"(attempts: {attempts:,})")

    print(f"  Done. Acceptance rate: {n_accepted / attempts:.4f}  "
          f"(attempts: {attempts:,})")

    # ── Step 3: assign clean sequential IDs and flatten ──────────────────
    all_rows = []
    for new_id, recs in enumerate(accepted_recs):
        for r in recs:
            row      = dict(r)
            row["id"] = new_id
            all_rows.append(row)

    return pd.DataFrame(all_rows)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 3b — DIRECT POOL SAMPLING (DPS)
# ─────────────────────────────────────────────────────────────────────────────
#
# Mechanism
# ─────────
# 1. Generate a pool of `pool_size` agents unconditionally from gX.
# 2. Compute each agent's tour code.
# 3. Assign each agent a sampling weight = f_tour[tour_code].
#    Agents whose tour code is absent from f_tour get weight 0.
# 4. Draw n_accepted agents from the pool WITH REPLACEMENT using those weights.
#
# Comparison with Rejection Sampling (RS)
# ────────────────────────────────────────
#
#  Property                  │ RS                          │ DPS
#  ──────────────────────────┼─────────────────────────────┼───────────────────────────────
#  Tour dist accuracy        │ Exact (as pool→∞)           │ Exact for tours in pool;
#                            │                             │ silent gap for absent tours
#  Within-tour diversity     │ Each agent independently    │ WITH REPLACEMENT → duplicates
#                            │ drawn from gX               │ when pool < n_accepted
#  Rare tour codes           │ Model can still generate    │ Need huge pool for rare tours
#                            │ them via CPDs               │ to appear even once
#  Speed                     │ Proportional to λ           │ One pool pass + O(pool) sample
#  Best use case             │ Small/sparse survey;        │ Large survey; speed priority;
#                            │ agent-based simulation      │ n_accepted << pool_size
#
# Rule of thumb: pool_size >= 5 × n_accepted to keep duplication rate low.

def _extract_final_tour(recs: list) -> str:
    return recs[-1]["tour_code"] if recs else ""


def direct_pool_sampling(model,
                         target_tour_dist: dict,
                         n_accepted: int,
                         pool_size: int = None,
                         max_stay_points: int = 12) -> pd.DataFrame:
    """
    Direct Pool Sampling: generate a pool unconditionally, then
    importance-resample using the target tour-network distribution.

    Parameters
    ----------
    model             : fitted TXModel
    target_tour_dist  : {tour_code: probability}  target tour distribution
                        (typically from the survey directly)
    n_accepted        : number of synthetic agents to return
    pool_size         : pool size before resampling
                        (default: max(5 * n_accepted, 10_000))
                        Rule of thumb: >= 5 x n_accepted to limit duplicates.
    max_stay_points   : max stay-points per agent

    Returns
    -------
    pd.DataFrame of resampled synthetic stay-point trajectories.

    Notes
    -----
    Agents are drawn WITH REPLACEMENT. If pool_size < n_accepted, many
    trajectories will be duplicated. Check duplication_rate in printed summary.
    """
    if pool_size is None:
        pool_size = max(5 * n_accepted, 10_000)

    print(f"Direct Pool Sampling: pool={pool_size:,}, target={n_accepted:,}...")

    # Step 1: generate unconditional pool
    pool_recs  = []
    pool_tours = []
    for i in range(pool_size):
        recs = model.generate_one(i, max_stay_points=max_stay_points)
        pool_recs.append(recs)
        pool_tours.append(_extract_final_tour(recs))

    # Step 2: assign weights from f_tour
    weights = np.array([target_tour_dist.get(t, 0.0) for t in pool_tours],
                       dtype=float)
    total_w = weights.sum()

    n_covered = int((weights > 0).sum())
    n_target  = len(target_tour_dist)
    coverage  = n_covered / max(n_target, 1) * 100
    print(f"  Pool tour coverage : {n_covered}/{n_target} target codes "
          f"({coverage:.1f}%)")

    if total_w == 0:
        raise ValueError(
            "No pool agent matches any target tour code. "
            "Increase pool_size or check that target_tour_dist was built "
            "from the same survey used to fit the model.")

    probs = weights / total_w

    # Step 3: weighted resample WITH REPLACEMENT
    chosen_idx     = np.random.choice(pool_size, size=n_accepted,
                                      replace=True, p=probs)
    n_unique       = len(set(chosen_idx))
    duplication_rt = 1.0 - n_unique / n_accepted
    print(f"  Unique agents      : {n_unique:,}/{n_accepted:,}  "
          f"(duplication rate: {duplication_rt:.3f})")
    if duplication_rt > 0.2:
        print(f"  WARNING: high duplication ({duplication_rt:.1%}). "
              f"Consider increasing pool_size (currently {pool_size:,}).")

    # Step 4: assemble output with clean sequential IDs
    all_rows = []
    for new_id, idx in enumerate(chosen_idx):
        for r in pool_recs[idx]:
            row       = dict(r)
            row["id"] = new_id
            all_rows.append(row)

    print("  Done.")
    return pd.DataFrame(all_rows)


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 5 — OUTPUT CONVERSION: stay-points → survey-format trips
# ─────────────────────────────────────────────────────────────────────────────

def stay_points_to_trips(synth_df: pd.DataFrame) -> pd.DataFrame:
    """
    Convert synthetic stay-point trajectories into survey-format trip records.

    Stay-point → trip mapping
    ─────────────────────────
      TRIP_STARTTIME  = end_h   of origin stay-point  (person departs)
      TRIP_ENDTIME    = start_h of destination stay-point  (person arrives)
      TRIP_CNT        = sp_idx of destination  (1-based)
      TRIP_MAX        = total trips that day  (= max sp_idx per agent)
    """
    synth_df = synth_df.sort_values(["id", "sp_idx"]).reset_index(drop=True)
    records  = []

    for agent_id, grp in synth_df.groupby("id"):
        grp = grp.sort_values("sp_idx").reset_index(drop=True)
        n   = len(grp)
        if n < 2:
            continue
        trip_max = n - 1
        for k in range(1, n):
            o = grp.iloc[k - 1]
            d = grp.iloc[k]
            records.append({
                "ID":                    agent_id,
                "TRIP_CNT":              k,
                "TRIP_MAX":              trip_max,
                "TRIP_STARTTIME":        o["end_h"],
                "TRIP_ENDTIME":          d["start_h"],
                "ORIGIN_SUBZONE":        o["zone"],
                "ORIGIN_SUBZONE_X":      o["x"],
                "ORIGIN_SUBZONE_Y":      o["y"],
                "DESTINATION_SUBZONE":   d["zone"],
                "DESTINATION_SUBZONE_X": d["x"],
                "DESTINATION_SUBZONE_Y": d["y"],
            })

    return pd.DataFrame(records, columns=[
        "ID", "TRIP_CNT", "TRIP_MAX",
        "TRIP_STARTTIME", "TRIP_ENDTIME",
        "ORIGIN_SUBZONE", "ORIGIN_SUBZONE_X", "ORIGIN_SUBZONE_Y",
        "DESTINATION_SUBZONE", "DESTINATION_SUBZONE_X", "DESTINATION_SUBZONE_Y",
    ])


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 6 — WIDE-FORMAT CONVERSION
# ─────────────────────────────────────────────────────────────────────────────

def trip_diaries_to_wide(df_trip: pd.DataFrame) -> pd.DataFrame:
    """
    Convert long-format trip diaries to wide tour format.

    Each agent becomes one row:
        ID, TRIP_MAX,
        ORIGIN_SUBZONE_1, ..., DESTINATION_SUBZONE_1, ...,
        TRIP_STARTTIME_1, TRIP_ENDTIME_1,
        ORIGIN_SUBZONE_2, ...

    TRIP_ENDTIME - TRIP_STARTTIME      = travel time per trip.
    TRIP_STARTTIME[k+1] - TRIP_ENDTIME[k] = activity duration at destination.
    """
    trip_cols = [
        'ORIGIN_SUBZONE', 'ORIGIN_SUBZONE_X', 'ORIGIN_SUBZONE_Y',
        'DESTINATION_SUBZONE', 'DESTINATION_SUBZONE_X', 'DESTINATION_SUBZONE_Y',
        'TRIP_STARTTIME', 'TRIP_ENDTIME',
    ]
    df = df_trip.copy()
    df['ID'] = df['ID'].astype(str)

    # pivot_table with aggfunc='first' handles agents with different TRIP_MAX
    df_wide = df.pivot_table(
        index='ID', columns='TRIP_CNT', values=trip_cols, aggfunc='first'
    )
    df_wide.columns = [f'{col}_{int(k)}' for col, k in df_wide.columns]
    df_wide = df_wide.reset_index()

    trip_max_map = df.groupby('ID')['TRIP_MAX'].max()
    df_wide = df_wide.merge(trip_max_map.rename('TRIP_MAX'), on='ID')

    max_trips = int(df['TRIP_MAX'].max()) if not df.empty else 0
    ordered   = ['ID', 'TRIP_MAX']
    for k in range(1, max_trips + 1):
        for col in trip_cols:
            c = f'{col}_{k}'
            if c in df_wide.columns:
                ordered.append(c)

    return df_wide[[c for c in ordered if c in df_wide.columns]]


# ─────────────────────────────────────────────────────────────────────────────
# SECTION 7 — END-TO-END PIPELINE
# ─────────────────────────────────────────────────────────────────────────────

def run_pipeline(survey_path: str,
                 od_path: str,
                 n_agents: int = 1000,
                 od_weight: float = 0.0,
                 method: str = "rs"):
    """
    Full TX model pipeline.

    Parameters
    ----------
    survey_path : path to household travel survey CSV
    od_path     : path to hourly OD matrix CSV
    n_agents    : number of synthetic agents to generate
    od_weight   : blend weight for OD matrix in PZ (0 = survey only)
    method      : sampling method — one of:
                    "rs"   Rejection Sampling (default).
                           Statistically exact; each agent is an independent
                           draw. Slower when λ is large.
                    "dps"  Direct Pool Sampling.
                           Faster; importance-resamples a pre-generated pool
                           using the survey tour distribution as weights.
                           May produce duplicate trajectories when
                           pool_size < 5 × n_agents.
                    "none" No tour correction; raw forward samples from gX.

    Returns
    -------
    trips_df : survey-format long trip DataFrame
    synth_df : stay-point DataFrame (intermediate)
    model    : fitted TXModel
    """
    print("=" * 60)
    print("STEP 1: Loading data")
    survey = load_survey(survey_path)
    od     = load_od(od_path)
    print(f"  Survey rows : {len(survey):,}")
    print(f"  OD rows     : {len(od):,}")

    print("\nSTEP 2: Building stay-point trajectories")
    sp_df     = assign_tour_codes(build_stay_points_from_survey(survey))
    n_persons = sp_df["ID"].nunique()
    print(f"  Persons: {n_persons:,}  Stay-points: {len(sp_df):,}")

    np.random.seed(42)
    all_ids   = sp_df["ID"].unique()
    train_ids = np.random.choice(all_ids, size=int(0.7 * len(all_ids)),
                                 replace=False)
    test_ids  = np.setdiff1d(all_ids, train_ids)
    sp_train  = sp_df[sp_df["ID"].isin(train_ids)]
    sp_test   = sp_df[sp_df["ID"].isin(test_ids)]
    print(f"  Train: {len(train_ids):,}  Test: {len(test_ids):,}")

    print("\nSTEP 3: Fitting TX model")
    model = TXModel(od_weight=od_weight)
    model.fit(sp_train, od_df=od)

    print(f"\nSTEP 4: Generating {n_agents:,} synthetic travellers  (method={method})")
    target_dist = distribution_from_series(
        sp_test.groupby("ID")["tour_code"].last())

    if method == "rs":
        synth_df = rejection_sampling(model, target_dist, n_accepted=n_agents)
    elif method == "dps":
        synth_df = direct_pool_sampling(model, target_dist, n_accepted=n_agents)
    elif method == "none":
        synth_df = model.generate_population(n_agents)
    else:
        raise ValueError(f"Unknown method '{method}'. Choose 'rs', 'dps', or 'none'.")

    print("\nSTEP 6: Converting stay-points → survey-format trips")
    trips_df     = stay_points_to_trips(synth_df)
    n_agents_out = trips_df["ID"].nunique()
    print(f"  Agents with ≥1 trip : {n_agents_out:,}")
    print(f"  Total trip records  : {len(trips_df):,}")
    print(f"  Avg trips per agent : {len(trips_df) / max(n_agents_out, 1):.2f}")
    print("=" * 60)

    return trips_df, synth_df, model


# ─────────────────────────────────────────────────────────────────────────────
# USAGE EXAMPLE
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    ff          = 0.1

    trips_df, synth_df, model = run_pipeline(
        survey_path = path_data + f"val_data_hts_trip_{ff}.csv",
        od_path     = path_data + "val_data_pcm_trip.csv",
        n_agents    = 50_000,
        method      = "rs",   # "rs" | "dps" | "none"
    )

    trips_wide = trip_diaries_to_wide(trips_df)
    trips_wide.to_csv(path_result + f"val_sim_tour_tx_{ff}.csv", index=False)
    print(trips_wide.info())
    print(trips_wide["TRIP_MAX"].value_counts())

STEP 1: Loading data
  Survey rows : 3,950
  OD rows     : 39,247

STEP 2: Building stay-point trajectories
  Persons: 1,541  Stay-points: 5,491
  Train: 1,078  Test: 463

STEP 3: Fitting TX model
Fitting TX model...
  P1  : 1 conditions
  PE1 : 174 conditions
  PZ  : 1992 conditions
  PS  : 2669 conditions
  PD  : 2397 conditions
  PT  : 2047 conditions
  Zones known: 280
Done.

STEP 4: Generating 50,000 synthetic travellers  (method=rs)
Rejection sampling: target=50,000, pool=100,000...
  λ = 345.71  (capped at 99.0th pct, 41 tours in target)
  accepted 32/50,000  (attempts: 10,240)
  accepted 53/50,000  (attempts: 20,480)
  accepted 85/50,000  (attempts: 30,720)
  accepted 124/50,000  (attempts: 40,960)
  accepted 149/50,000  (attempts: 51,200)
  accepted 174/50,000  (attempts: 61,440)
  accepted 204/50,000  (attempts: 71,680)
  accepted 236/50,000  (attempts: 81,920)
  accepted 269/50,000  (attempts: 92,160)
  accepted 299/50,000  (attempts: 102,400)
  accepted 331/50,000  (attempt

# DITRAS

In [ ]:
"""
DITRAS Adapted for Household Travel Survey + Hourly OD Matrix
=============================================================
Refactored fast stratified-pool implementation.

Key improvement
---------------
Tour-code stratification is now performed ONLINE at the agent level:

  1. generate one synthetic agent
  2. compute that agent's tour code immediately
  3. store the whole agent trip-chain directly in pool_by_code[code]

This avoids repeated expensive dataframe-wide regrouping and merging.

Supported methods
-----------------
method = "none"
    No global tour-code correction; unconditional DITRAS generation.

method = "stratified_pool"
    Generate a large unconditional pool, bucket agents directly by tour code,
    top up rare codes, and sample to match the survey tour-code distribution.
"""

import os
import sys
import time
import warnings
from collections import defaultdict
from math import sqrt, sin, cos, pi, asin
from random import random, uniform, sample

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")


# ---------------------------------------------------------------------------
# Utilities
# ---------------------------------------------------------------------------

def earth_distance(lat_lng1, lat_lng2):
    """Haversine distance in km between two (lat, lon) pairs."""
    lat1, lng1 = [l * pi / 180 for l in lat_lng1]
    lat2, lng2 = [l * pi / 180 for l in lat_lng2]
    dlat, dlng = lat1 - lat2, lng1 - lng2
    ds = 2 * asin(sqrt(sin(dlat / 2.0) ** 2 + cos(lat1) * cos(lat2) * sin(dlng / 2.0) ** 2))
    return 6371.01 * ds


def weighted_random_selection(weights):
    """
    Sample an index from a list of nonnegative weights.
    Falls back to uniform random index if all weights are zero.
    """
    totals = []
    running_total = 0.0

    for w in weights:
        running_total += w
        totals.append(running_total)

    if running_total == 0:
        return np.random.randint(0, len(weights))

    rnd = random() * running_total
    for index, total in enumerate(totals):
        if rnd < total:
            return index

    return len(totals) - 1


def timeit(method):
    """Simple timing decorator."""
    def timed(*args, **kwargs):
        ts = time.time()
        result = method(*args, **kwargs)
        te = time.time()
        print(f"\n\t\ttime: {(te - ts) / 60:.2f} min")
        return result
    return timed


def distribution_from_series(s: pd.Series) -> dict:
    """Convert a categorical series into a probability dictionary."""
    probs = s.value_counts(normalize=True)
    return probs.to_dict()


def allocate_target_counts(target_dist: dict, n_agents: int) -> dict:
    """
    Convert a probability distribution into exact integer counts summing to n_agents.
    """
    if not target_dist:
        return {}

    codes = list(target_dist.keys())
    probs = np.array([target_dist[c] for c in codes], dtype=float)
    probs = probs / probs.sum()

    exact = probs * n_agents
    floors = np.floor(exact).astype(int)
    deficit = n_agents - floors.sum()

    if deficit > 0:
        frac = exact - floors
        top_idx = np.argsort(frac)[::-1][:deficit]
        floors[top_idx] += 1

    return dict(zip(codes, floors))


def encode_zone_sequence_to_tour_code(zones):
    """
    Encode a sequence of zones into a compact tour code.

    Example
    -------
    [A, B, A, C, A] -> "01020"
    """
    zone_to_digit = {}
    next_digit = 0
    digits = []

    for z in zones:
        if z not in zone_to_digit:
            zone_to_digit[z] = next_digit
            next_digit += 1
        digits.append(str(zone_to_digit[z]))

    return "".join(digits)


def build_survey_tour_distribution(survey_df: pd.DataFrame) -> dict:
    """
    Build final survey tour-code distribution from observed trip chains.

    Survey tour code is computed from:
      [HOME_SUBZONE, DESTINATION_1, DESTINATION_2, ..., DESTINATION_m]
    """
    required = {"ID", "TRIP_CNT", "HOME_SUBZONE", "DESTINATION_SUBZONE"}
    missing = required - set(survey_df.columns)
    if missing:
        raise KeyError(f"Survey dataframe missing columns required for tour code: {sorted(missing)}")

    code_series = (
        survey_df.sort_values(["ID", "TRIP_CNT"])
        .groupby("ID")
        .apply(
            lambda g: encode_zone_sequence_to_tour_code(
                [g.iloc[0]["HOME_SUBZONE"]] + g["DESTINATION_SUBZONE"].tolist()
            ),
            include_groups=False,
        )
    )
    return distribution_from_series(code_series)


def compute_agent_tour_code_from_trips(agent_trips):
    """
    Compute one agent's final tour code directly from that agent's trip rows.

    Tour code is computed from:
      [ORIGIN_SUBZONE_1, DESTINATION_1, DESTINATION_2, ..., DESTINATION_m]
    """
    if not agent_trips:
        return ""

    trips_sorted = sorted(agent_trips, key=lambda r: r["TRIP_CNT"])
    zone_seq = [trips_sorted[0]["ORIGIN_SUBZONE"]] + [r["DESTINATION_SUBZONE"] for r in trips_sorted]
    return encode_zone_sequence_to_tour_code(zone_seq)


def attach_tour_code_to_output(df_trip: pd.DataFrame) -> pd.DataFrame:
    """
    Attach TOUR_CODE to each row of a trip dataframe.

    This function is used only once at the end in the refactored pipeline,
    not repeatedly during bucket construction.
    """
    if df_trip is None:
        return pd.DataFrame(columns=["TOUR_CODE"])

    out = df_trip.copy()

    if len(out) == 0:
        if "TOUR_CODE" not in out.columns:
            out["TOUR_CODE"] = pd.Series(dtype="object")
        return out

    required_cols = {"ID", "TRIP_CNT", "ORIGIN_SUBZONE", "DESTINATION_SUBZONE"}
    missing = required_cols - set(out.columns)

    if missing:
        out["TOUR_CODE"] = np.nan
        return out

    def _agent_code(g):
        g = g.sort_values("TRIP_CNT")
        seq = [g.iloc[0]["ORIGIN_SUBZONE"]] + g["DESTINATION_SUBZONE"].tolist()
        return encode_zone_sequence_to_tour_code(seq)

    code_map = (
        out.groupby("ID", group_keys=False)
           .apply(_agent_code, include_groups=False)
           .rename("TOUR_CODE")
           .reset_index()
    )

    out = out.merge(code_map, on="ID", how="left")
    return out


# ---------------------------------------------------------------------------
# Spatial Tessellation
# ---------------------------------------------------------------------------

def build_spatial_tessellation(survey_df, od_df):
    """
    Build spatial tessellation from all unique subzones across survey and OD data.
    """
    records = {}

    def _add(subzone, x, y):
        if pd.notna(subzone) and subzone not in records:
            records[subzone] = {"lat": float(y), "lon": float(x)}

    for df in [survey_df, od_df]:
        if "ORIGIN_SUBZONE" in df.columns:
            for _, row in df[["ORIGIN_SUBZONE", "ORIGIN_SUBZONE_X", "ORIGIN_SUBZONE_Y"]].drop_duplicates().iterrows():
                _add(row["ORIGIN_SUBZONE"], row["ORIGIN_SUBZONE_X"], row["ORIGIN_SUBZONE_Y"])

        if "DESTINATION_SUBZONE" in df.columns:
            for _, row in df[["DESTINATION_SUBZONE", "DESTINATION_SUBZONE_X", "DESTINATION_SUBZONE_Y"]].drop_duplicates().iterrows():
                _add(row["DESTINATION_SUBZONE"], row["DESTINATION_SUBZONE_X"], row["DESTINATION_SUBZONE_Y"])

    if "HOME_SUBZONE" in survey_df.columns:
        for _, row in survey_df[["HOME_SUBZONE", "HOME_SUBZONE_X", "HOME_SUBZONE_Y"]].drop_duplicates().iterrows():
            _add(row["HOME_SUBZONE"], row["HOME_SUBZONE_X"], row["HOME_SUBZONE_Y"])

    subzone_to_id = {sz: i for i, sz in enumerate(sorted(records.keys()))}

    spatial_tessellation = {
        subzone_to_id[sz]: {"subzone": sz, "lat": info["lat"], "lon": info["lon"]}
        for sz, info in records.items()
    }

    print(f"[Spatial Tessellation] {len(spatial_tessellation)} unique subzones found.")
    return spatial_tessellation, subzone_to_id


# ---------------------------------------------------------------------------
# Hourly OD Matrix
# ---------------------------------------------------------------------------

def build_hourly_od_matrix(od_df, subzone_to_id, n_zones):
    """
    Build a separate OD probability matrix for each hour of the day (0-23).
    """
    print("[Building Hourly OD Matrices]")
    hourly_od = {}

    for hour in range(24):
        mat = np.zeros((n_zones, n_zones))
        hour_df = od_df[od_df["TRIP_STARTTIME"] == hour]

        for _, row in hour_df.iterrows():
            o = subzone_to_id.get(row["ORIGIN_SUBZONE"])
            d = subzone_to_id.get(row["DESTINATION_SUBZONE"])

            if o is not None and d is not None and o != d:
                mat[o, d] += 1

        row_sums = mat.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1
        mat = mat / row_sums

        hourly_od[hour] = mat
        sys.stdout.write(f"\r  Hour {hour:02d}/23 done")
        sys.stdout.flush()

    print(f"\n[Hourly OD Matrices] Built {len(hourly_od)} matrices of shape ({n_zones} x {n_zones}).")
    return hourly_od


# ---------------------------------------------------------------------------
# Survey Diary Generator
# ---------------------------------------------------------------------------

class SurveyDiaryGenerator:
    """
    Generates mobility diaries by sampling real trip chains from the survey.
    """

    def __init__(self, survey_df):
        self.name = "SurveyDiaryGenerator"
        self.persons = {
            pid: grp.sort_values("TRIP_CNT").reset_index(drop=True)
            for pid, grp in survey_df.groupby("ID")
        }
        self.person_ids = list(self.persons.keys())
        print(f"[SurveyDiaryGenerator] {len(self.person_ids)} persons loaded.")

    def start_simulation(self, diary_length=24):
        """
        Sample one person's trip chain and convert it into an hourly diary.
        """
        pid = sample(self.person_ids, 1)[0]
        trips_sorted = self.persons[pid].sort_values("TRIP_CNT").reset_index(drop=True)
        n_trips = len(trips_sorted)

        diary = [0] * diary_length
        trip_endtimes = {}

        for idx in range(n_trips):
            trip = trips_sorted.iloc[idx]
            start = int(trip["TRIP_STARTTIME"])
            end = int(trip["TRIP_ENDTIME"])
            trip_idx = int(trip["TRIP_CNT"])

            if end <= start:
                end = start + 1

            trip_endtimes[trip_idx] = end

            # Trip segment itself
            for h in range(start, min(end, diary_length)):
                diary[h] = trip_idx

            # Stay after arrival until next departure
            if idx < n_trips - 1:
                next_start = int(trips_sorted.iloc[idx + 1]["TRIP_STARTTIME"])
                for h in range(end, min(next_start, diary_length)):
                    diary[h] = trip_idx

        return diary, trips_sorted, trip_endtimes


# ---------------------------------------------------------------------------
# Hourly d-EPR Trajectory Generator
# ---------------------------------------------------------------------------

class HourlyDEPR:
    """
    Extended d-EPR model using the hour-specific OD matrix for exploration.
    """

    def __init__(self, rho=0.6, gamma=0.21):
        self.name = "HourlyDEPR"
        self.rho = rho
        self.gamma = gamma

    def _preferential_return(self):
        """Return to a previously visited location weighted by visit frequency."""
        locs = list(self.location2visits.keys())
        visits = list(self.location2visits.values())
        index = weighted_random_selection(visits)
        return locs[index]

    def _preferential_exploration(self, current_location, hour):
        """Explore a new location using the OD matrix for the given hour."""
        od = self.hourly_od[hour % 24]
        weights = od[current_location].tolist()
        return weighted_random_selection(weights)

    def choose_location(self, current_location, hour):
        """
        Choose next location using d-EPR:
            probability(new) = rho * S^{-gamma}
        """
        S = len(self.location2visits)

        if S == 0:
            self.home = self._preferential_exploration(self.home, hour)
            return self.home

        p_new = uniform(0, 1)
        if p_new <= self.rho * (S ** -self.gamma):
            return self._preferential_exploration(current_location, hour)

        return self._preferential_return()

    def start_simulation(self, spatial_tessellation, subzone_to_id,
                         mobility_diary, sampled_trips, hourly_od,
                         trip_endtimes=None, home_subzone=None):
        """
        Simulate one trajectory conditioned on a sampled diary and return trip rows.
        """
        self.trajectory = []
        self.location2visits = defaultdict(int)
        self.hourly_od = hourly_od
        self.spatial_tessellation = spatial_tessellation

        if home_subzone and home_subzone in subzone_to_id:
            self.home = subzone_to_id[home_subzone]
        elif "HOME_SUBZONE" in sampled_trips.columns and len(sampled_trips) > 0:
            hsz = sampled_trips.iloc[0]["HOME_SUBZONE"]
            self.home = subzone_to_id.get(hsz, sample(list(spatial_tessellation.keys()), 1)[0])
        else:
            self.home = sample(list(spatial_tessellation.keys()), 1)[0]

        segments = []
        i = 0
        prev_zone = self.home

        while i < len(mobility_diary):
            hour = i % 24
            slot = mobility_diary[i]

            if slot == 0:
                self.trajectory.append(self.home)
                self.location2visits[self.home] += 1
                prev_zone = self.home
                i += 1
            else:
                dest_zone = self.choose_location(prev_zone, hour)
                start_hour = i

                j = i + 1
                while j < len(mobility_diary) and mobility_diary[j] == slot:
                    self.trajectory.append(dest_zone)
                    self.location2visits[dest_zone] += 1
                    j += 1

                end_hour = min(j, len(mobility_diary))
                self.trajectory.append(dest_zone)
                self.location2visits[dest_zone] += 1

                trip_endtime = trip_endtimes.get(slot, end_hour) if trip_endtimes else end_hour

                segments.append({
                    "trip_idx": slot,
                    "origin_zone": prev_zone,
                    "dest_zone": dest_zone,
                    "start_hour": start_hour,
                    "trip_endtime": trip_endtime,
                    "end_hour": end_hour,
                })

                prev_zone = dest_zone
                i = j

        trips = []
        trip_max = len(segments)

        for rank, seg in enumerate(segments, start=1):
            o_info = spatial_tessellation[seg["origin_zone"]]
            d_info = spatial_tessellation[seg["dest_zone"]]

            trips.append({
                "TRIP_CNT": rank,
                "TRIP_MAX": trip_max,
                "TRIP_STARTTIME": seg["start_hour"],
                "TRIP_ENDTIME": seg["trip_endtime"],
                "ORIGIN_SUBZONE": o_info["subzone"],
                "ORIGIN_SUBZONE_X": o_info["lon"],
                "ORIGIN_SUBZONE_Y": o_info["lat"],
                "DESTINATION_SUBZONE": d_info["subzone"],
                "DESTINATION_SUBZONE_X": d_info["lon"],
                "DESTINATION_SUBZONE_Y": d_info["lat"],
            })

        return trips, self.home


# ---------------------------------------------------------------------------
# DITRAS Adapted (Fast Stratified Pool)
# ---------------------------------------------------------------------------

class DITRASAdapted:
    """
    DITRAS adapted for household travel survey + hourly OD matrix.

    method = "none"
        Unconditional generation.

    method = "stratified_pool"
        Generate a large unconditional pool, compute one tour code per agent
        immediately, bucket agents directly by code, top up rare codes, and
        resample to match the survey tour-code distribution.
    """

    def __init__(self, n_agents=1000, diary_length=24,
                 rho=0.6, gamma=0.21,
                 method="none",
                 initial_pool_factor=5,
                 min_pool_size=10000,
                 topup_batch_size=5000,
                 topup_max_rounds=20,
                 output_file="synthetic_trips.csv",
                 random_seed=42):
        self.n_agents = n_agents
        self.diary_length = diary_length
        self.rho = rho
        self.gamma = gamma
        self.method = method
        self.initial_pool_factor = initial_pool_factor
        self.min_pool_size = min_pool_size
        self.topup_batch_size = topup_batch_size
        self.topup_max_rounds = topup_max_rounds
        self.output_file = output_file
        self.random_seed = random_seed

    # ---------------------------------------------------------------------
    # Data loading
    # ---------------------------------------------------------------------

    def load_data(self, survey_df, od_df):
        print("\n=== DITRAS Adapted: Loading Data ===")

        self.survey_df = survey_df.copy()
        self.od_df = od_df.copy()

        self.spatial_tessellation, self.subzone_to_id = \
            build_spatial_tessellation(survey_df, od_df)
        self.n_zones = len(self.spatial_tessellation)

        self.hourly_od = build_hourly_od_matrix(od_df, self.subzone_to_id, self.n_zones)
        self.diary_generator = SurveyDiaryGenerator(survey_df)
        self.traj_generator = HourlyDEPR(rho=self.rho, gamma=self.gamma)

        print("\n=== Data Loading Complete ===\n")

    # ---------------------------------------------------------------------
    # Single-agent generation
    # ---------------------------------------------------------------------

    def _generate_one_agent(self, agent_id):
        """
        Generate one synthetic agent unconditionally and return trip rows.
        """
        diary, sampled_trips, trip_endtimes = self.diary_generator.start_simulation(
            diary_length=self.diary_length
        )

        trips, _ = self.traj_generator.start_simulation(
            spatial_tessellation=self.spatial_tessellation,
            subzone_to_id=self.subzone_to_id,
            mobility_diary=diary,
            sampled_trips=sampled_trips,
            hourly_od=self.hourly_od,
            trip_endtimes=trip_endtimes,
        )

        syn_id = f"SYN_{agent_id:06d}"
        for trip in trips:
            trip["ID"] = syn_id

        return trips

    # ---------------------------------------------------------------------
    # Unconditional generation
    # ---------------------------------------------------------------------

    def _generate_population_none(self):
        """
        Unconditional DITRAS generation without tour-code stratification.
        """
        print(f"\n=== DITRAS Simulation (method=none) ===")
        print(f"  Agents       : {self.n_agents:,}")
        print(f"  Diary length : {self.diary_length} hours")
        print(f"  Zones        : {self.n_zones}")
        print(f"  rho={self.rho}, gamma={self.gamma}\n")

        all_rows = []
        bar_length = 20
        old_p = 0

        for agent_id in range(self.n_agents):
            trips = self._generate_one_agent(agent_id)
            all_rows.extend(trips)

            percentage = int((agent_id + 1) * 100 / self.n_agents)
            if percentage > old_p:
                hashes = "-" * int(round(percentage / 5))
                spaces = " " * (bar_length - len(hashes))
                sys.stdout.write(f"\rProgress: [{hashes + spaces}] {percentage}%")
                sys.stdout.flush()
                old_p = percentage

        print()
        return pd.DataFrame(all_rows)

    # ---------------------------------------------------------------------
    # Fast online stratified-pool helpers
    # ---------------------------------------------------------------------

    def _build_pool_by_tour_code(self, pool_size, start_id=0):
        """
        Generate an unconditional pool and bucket agents immediately by tour code.

        Returns
        -------
        pool_by_code : dict
            {tour_code: [agent_trip_list, ...]}
        next_id : int
            next available agent id
        """
        pool_by_code = defaultdict(list)
        next_id = start_id

        for _ in range(pool_size):
            agent_trips = self._generate_one_agent(next_id)
            code = compute_agent_tour_code_from_trips(agent_trips)
            pool_by_code[code].append(agent_trips)
            next_id += 1

        return pool_by_code, next_id

    def _top_up_pool_by_code(self, pool_by_code, target_counts, start_id=0):
        """
        Top up only the buckets that remain below the target counts.
        """
        next_id = start_id

        for round_idx in range(self.topup_max_rounds):
            shortfall = {
                c: max(target_counts.get(c, 0) - len(pool_by_code.get(c, [])), 0)
                for c in target_counts
            }
            total_short = sum(shortfall.values())

            if total_short == 0:
                print("  No shortfall remains after pool generation.")
                break

            active_short = sum(v > 0 for v in shortfall.values())
            print(f"  Top-up round {round_idx + 1}: total shortfall={total_short:,}, active codes={active_short:,}")

            for _ in range(self.topup_batch_size):
                agent_trips = self._generate_one_agent(next_id)
                code = compute_agent_tour_code_from_trips(agent_trips)

                # Only store the agent if the code is a target code and that bucket is still short.
                if code in target_counts and len(pool_by_code[code]) < target_counts[code]:
                    pool_by_code[code].append(agent_trips)

                next_id += 1

        return pool_by_code, next_id

    def _sample_from_pool_by_code(self, pool_by_code, target_counts):
        """
        Sample exactly N_c agents from each tour-code bucket.

        Uses replacement only when a bucket remains smaller than N_c.
        """
        rng = np.random.default_rng(self.random_seed)
        sampled_rows = []
        new_seq = 0

        target_counts_sorted = dict(sorted(target_counts.items(), key=lambda x: (-x[1], x[0])))

        for code, n_target in target_counts_sorted.items():
            if n_target <= 0:
                continue

            bucket = pool_by_code.get(code, [])

            if len(bucket) == 0:
                print(f"  WARNING: no agents available for code {code}; skipping.")
                continue

            replace = len(bucket) < n_target
            idx = rng.choice(len(bucket), size=n_target, replace=replace)

            if replace:
                print(f"  WARNING: bucket {code} has only {len(bucket):,} agents; "
                      f"sampling {n_target:,} with replacement.")

            for k in idx:
                agent_trips = bucket[k]
                new_id = f"SYN_{new_seq:06d}"

                for row in agent_trips:
                    new_row = dict(row)
                    new_row["ID"] = new_id
                    new_row["TOUR_CODE"] = code
                    sampled_rows.append(new_row)

                new_seq += 1

        return pd.DataFrame(sampled_rows)

    # ---------------------------------------------------------------------
    # Fast stratified-pool generation
    # ---------------------------------------------------------------------

    def _generate_population_stratified_pool(self):
        """
        Fast pool-based tour-code stratified generation.

        Algorithm
        ---------
        1. Build target tour-code counts from survey.
        2. Generate a large unconditional pool of agents.
        3. Compute each agent's code immediately and bucket directly.
        4. Top up only short buckets.
        5. Sample exactly N_c agents from each bucket.
        """
        target_dist = build_survey_tour_distribution(self.survey_df)
        target_counts = allocate_target_counts(target_dist, self.n_agents)

        print(f"\n=== DITRAS Simulation (method=stratified_pool) ===")
        print(f"  Agents              : {self.n_agents:,}")
        print(f"  Diary length        : {self.diary_length} hours")
        print(f"  Zones               : {self.n_zones}")
        print(f"  rho={self.rho}, gamma={self.gamma}")
        print(f"  Target tour codes   : {len(target_counts):,}")
        print(f"  Initial pool factor : {self.initial_pool_factor}\n")

        total_n = sum(target_counts.values())
        initial_pool_size = max(self.initial_pool_factor * total_n, self.min_pool_size)

        print(f"Building unconditional pool: {initial_pool_size:,} agents")
        pool_by_code, next_id = self._build_pool_by_tour_code(
            pool_size=initial_pool_size,
            start_id=0
        )

        target_codes = set(target_counts.keys())
        initial_cover = sum(len(pool_by_code.get(c, [])) > 0 for c in target_codes)
        print(f"  Initial bucket coverage: {initial_cover}/{len(target_codes)} target tour codes")

        pool_by_code, next_id = self._top_up_pool_by_code(
            pool_by_code=pool_by_code,
            target_counts=target_counts,
            start_id=next_id
        )

        output_df = self._sample_from_pool_by_code(
            pool_by_code=pool_by_code,
            target_counts=target_counts
        )

        return output_df

    # ---------------------------------------------------------------------
    # Main run
    # ---------------------------------------------------------------------

    @timeit
    def run(self, save_output=True):
        """
        Run the DITRAS simulation and optionally save results to CSV.

        Output columns
        --------------
        ID, TRIP_CNT, TRIP_MAX, TRIP_STARTTIME, TRIP_ENDTIME,
        ORIGIN_SUBZONE, ORIGIN_SUBZONE_X, ORIGIN_SUBZONE_Y,
        DESTINATION_SUBZONE, DESTINATION_SUBZONE_X, DESTINATION_SUBZONE_Y,
        TOUR_CODE
        """
        output_cols = [
            "ID", "TRIP_CNT", "TRIP_MAX",
            "TRIP_STARTTIME", "TRIP_ENDTIME",
            "ORIGIN_SUBZONE", "ORIGIN_SUBZONE_X", "ORIGIN_SUBZONE_Y",
            "DESTINATION_SUBZONE", "DESTINATION_SUBZONE_X", "DESTINATION_SUBZONE_Y",
            "TOUR_CODE",
        ]

        if self.method == "none":
            output_df = self._generate_population_none()
            output_df = attach_tour_code_to_output(output_df)

        elif self.method == "stratified_pool":
            output_df = self._generate_population_stratified_pool()

            # Safety step: if TOUR_CODE somehow missing, attach once at the end.
            if output_df is not None and len(output_df) > 0 and "TOUR_CODE" not in output_df.columns:
                output_df = attach_tour_code_to_output(output_df)

        else:
            raise ValueError("Unknown method. Choose 'none' or 'stratified_pool'.")

        if output_df is None or len(output_df) == 0:
            output_df = pd.DataFrame(columns=output_cols)
        else:
            for col in output_cols:
                if col not in output_df.columns:
                    output_df[col] = np.nan
            output_df = output_df[output_cols]

        if save_output:
            output_df.to_csv(self.output_file, index=False)
            statinfo = os.stat(self.output_file)

            print(f"\nDone. Output: {self.output_file}")
            print(f"  Trip rows    : {len(output_df):,}")
            print(f"  Agents       : {output_df['ID'].nunique() if 'ID' in output_df.columns else 0:,}")
            print(f"  File size    : {statinfo.st_size / 1e6:.2f} MB")
        else:
            print("\nDone. Output kept in memory only.")
            print(f"  Trip rows    : {len(output_df):,}")
            print(f"  Agents       : {output_df['ID'].nunique() if 'ID' in output_df.columns else 0:,}")

        return output_df


# ---------------------------------------------------------------------------
# Wide Conversion
# ---------------------------------------------------------------------------

def trip_diaries_to_wide(df_trip: pd.DataFrame) -> pd.DataFrame:
    """
    Convert long-format trip diaries to wide tour format.
    """
    trip_cols = [
        "ORIGIN_SUBZONE", "ORIGIN_SUBZONE_X", "ORIGIN_SUBZONE_Y",
        "DESTINATION_SUBZONE", "DESTINATION_SUBZONE_X", "DESTINATION_SUBZONE_Y",
        "TRIP_STARTTIME", "TRIP_ENDTIME",
    ]

    df = df_trip.copy()
    df["ID"] = df["ID"].astype(str)

    df_wide = df.pivot_table(
        index="ID",
        columns="TRIP_CNT",
        values=trip_cols,
        aggfunc="first",
    )

    df_wide.columns = [f"{col}_{int(k)}" for col, k in df_wide.columns]
    df_wide = df_wide.reset_index()

    trip_max_map = df.groupby("ID")["TRIP_MAX"].max()
    df_wide = df_wide.merge(trip_max_map.rename("TRIP_MAX"), on="ID")

    if "TOUR_CODE" in df.columns:
        code_map = df.groupby("ID")["TOUR_CODE"].first()
        df_wide = df_wide.merge(code_map.rename("TOUR_CODE"), on="ID")

    max_trips = int(df["TRIP_MAX"].max()) if not df.empty else 0
    ordered = ["ID", "TRIP_MAX"]

    if "TOUR_CODE" in df_wide.columns:
        ordered.append("TOUR_CODE")

    for k in range(1, max_trips + 1):
        for col in trip_cols:
            c = f"{col}_{k}"
            if c in df_wide.columns:
                ordered.append(c)

    return df_wide[[c for c in ordered if c in df_wide.columns]]


# ---------------------------------------------------------------------------
# Example Usage
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    ff = 0.5
    survey_df = pd.read_csv(path_data + f"val_data_hts_trip_{ff}.csv")
    od_df = pd.read_csv(path_data + "val_data_pcm_trip.csv")

    output_path = path_result + f"val_sim_tour_ditras_{ff}.csv"

    ditras = DITRASAdapted(
        n_agents=50_000,
        diary_length=24,
        rho=0.6,
        gamma=0.21,
        method="stratified_pool",   # "stratified_pool" | "none"
        initial_pool_factor=5,
        min_pool_size=10_000,
        topup_batch_size=5_000,
        topup_max_rounds=20,
        output_file=output_path,
        random_seed=42,
    )

    ditras.load_data(survey_df, od_df)

    # Keep in memory; save only final desired output
    ditras_df_long = ditras.run(save_output=False)

    ditras_df_wide = trip_diaries_to_wide(ditras_df_long)
    ditras_df_wide.to_csv(output_path, index=False)

    print(ditras_df_wide.info())
    print(ditras_df_wide["TRIP_MAX"].value_counts())


=== DITRAS Adapted: Loading Data ===
[Spatial Tessellation] 309 unique subzones found.
[Building Hourly OD Matrices]
  Hour 23/23 done
[Hourly OD Matrices] Built 24 matrices of shape (309 x 309).
[SurveyDiaryGenerator] 7705 persons loaded.

=== Data Loading Complete ===


=== DITRAS Simulation (method=stratified_pool) ===
  Agents              : 50,000
  Diary length        : 24 hours
  Zones               : 309
  rho=0.6, gamma=0.21
  Target tour codes   : 210
  Initial pool factor : 5

Building unconditional pool: 250,000 agents
  Initial bucket coverage: 143/210 target tour codes
  Top-up round 1: total shortfall=6,414, active codes=155
  Top-up round 2: total shortfall=5,742, active codes=154
  Top-up round 3: total shortfall=5,656, active codes=152
  Top-up round 4: total shortfall=5,566, active codes=152
  Top-up round 5: total shortfall=5,467, active codes=152
  Top-up round 6: total shortfall=5,373, active codes=152
  Top-up round 7: total shortfall=5,291, active codes=152
  T

# Imputing sociodemphraphics and activity semantics

In [ ]:
import os
import re
import numpy as np
import pandas as pd


# ============================================================
# (1) Utilities
# ============================================================
def _detect_max_k(wide_df: pd.DataFrame) -> int:
    """
    Detect the maximum trip index k from trip-indexed columns.
    """
    pat = re.compile(
        r"^(ORIGIN_SUBZONE|DESTINATION_SUBZONE|TRIP_PURPOSE|TRIP_STARTTIME|TRIP_ENDTIME)_(\d+)$"
    )
    ks = [int(m.group(2)) for c in wide_df.columns if (m := pat.match(c))]
    return max(ks) if ks else 0


def _to_str_or_none(x):
    """
    Convert a value to a stripped string, or return None if missing or empty.
    """
    if pd.isna(x):
        return None
    s = str(x).strip()
    return s if s != "" else None


def _norm_key_value(x):
    """
    Normalize a key value so dictionary lookup is stable across NaN / pandas scalars.
    """
    if pd.isna(x):
        return None
    return x


def _norm_key_tuple(values):
    """
    Normalize a tuple of key values for dictionary lookup.
    """
    return tuple(_norm_key_value(v) for v in values)


# ============================================================
# (2) Motif extraction
# ============================================================
def extract_location_sequence_from_wide(
    row: pd.Series,
    *,
    max_k: int,
    origin_prefix: str = "ORIGIN_SUBZONE_",
    dest_prefix: str = "DESTINATION_SUBZONE_",
) -> list[str]:
    """
    Extract the visited location sequence from one wide-format row.

    Example
    -------
    [origin_1, dest_1, dest_2, ..., dest_m]
    """
    o1 = _to_str_or_none(row.get(f"{origin_prefix}1"))
    if o1 is None:
        return []

    seq = [o1]

    for k in range(1, max_k + 1):
        dk = _to_str_or_none(row.get(f"{dest_prefix}{k}"))
        if dk is None:
            break
        seq.append(dk)

    return seq


def sequence_to_motif(seq: list[str]) -> str:
    """
    Convert a location sequence to a motif string.

    Example
    -------
    [A, B, A] -> "010"
    [A, B, C, A] -> "0120"
    """
    if not seq:
        return ""

    mapping = {}
    next_code = 0
    codes = []

    for loc in seq:
        if loc not in mapping:
            mapping[loc] = next_code
            next_code += 1
        codes.append(str(mapping[loc]))

    return "".join(codes)


def extract_motifs_from_wide(
    wide_df: pd.DataFrame,
    *,
    max_k: int | None = None,
    origin_prefix: str = "ORIGIN_SUBZONE_",
    dest_prefix: str = "DESTINATION_SUBZONE_",
    motif_col: str = "MOTIF",
) -> pd.DataFrame:
    """
    Keep all original columns and add a motif column.
    """
    if max_k is None:
        max_k = _detect_max_k(wide_df)
    if max_k == 0:
        raise ValueError("Cannot detect trip-indexed columns.")

    motifs = []

    for _, row in wide_df.iterrows():
        seq = extract_location_sequence_from_wide(
            row,
            max_k=max_k,
            origin_prefix=origin_prefix,
            dest_prefix=dest_prefix,
        )
        motifs.append(sequence_to_motif(seq))

    out = wide_df.copy()
    out[motif_col] = motifs
    return out


# ============================================================
# (3) Matching features
# ============================================================
def compute_home_features(df: pd.DataFrame, max_k: int | None = None) -> pd.DataFrame:
    """
    Compute matching features from wide trip-chain dataframe.

    Features
    --------
    - home_loc = ORIGIN_SUBZONE_1 + "_" + DESTINATION_SUBZONE_{TRIP_MAX}
    - dep_time = TRIP_STARTTIME_1
    - arr_time = TRIP_ENDTIME_{TRIP_MAX}
    - TRIP_MAX
    - MOTIF
    """
    if max_k is None:
        max_k = _detect_max_k(df)

    if "MOTIF" not in df.columns:
        raise ValueError("Column 'MOTIF' is required before computing home features.")

    home_origin = df["ORIGIN_SUBZONE_1"]

    last_dest = df.apply(
        lambda r: r.get(f"DESTINATION_SUBZONE_{int(r.TRIP_MAX)}"),
        axis=1,
    )

    arr_time = df.apply(
        lambda r: r.get(f"TRIP_ENDTIME_{int(r.TRIP_MAX)}"),
        axis=1,
    )

    home_loc = home_origin.astype(str) + "_" + last_dest.astype(str)
    dep_time = df["TRIP_STARTTIME_1"]

    return pd.DataFrame(
        {
            "home_loc": home_loc,
            "dep_time": dep_time,
            "arr_time": arr_time,
            "TRIP_MAX": df["TRIP_MAX"],
            "MOTIF": df["MOTIF"],
        },
        index=df.index,
    )


def attach_features(df: pd.DataFrame, max_k: int | None = None) -> pd.DataFrame:
    """
    Return a copy of df with matching features appended.
    """
    out = df.copy()
    feat = compute_home_features(out, max_k=max_k)
    for c in feat.columns:
        out[c] = feat[c]
    return out


# ============================================================
# (4) Diagnostics
# ============================================================
def infer_real_last_trip(
    df: pd.DataFrame,
    *,
    max_k: int | None = None,
    use: str = "both",
) -> pd.Series:
    """
    Infer the actual last existing trip index from non-null trip columns.
    """
    if max_k is None:
        max_k = _detect_max_k(df)

    if use not in {"origin", "destination", "both"}:
        raise ValueError("use must be one of {'origin', 'destination', 'both'}.")

    real_last = np.zeros(len(df), dtype=int)

    for k in range(1, max_k + 1):
        ocol = f"ORIGIN_SUBZONE_{k}"
        dcol = f"DESTINATION_SUBZONE_{k}"

        if use == "origin":
            exists = df[ocol].notna() if ocol in df.columns else pd.Series(False, index=df.index)
        elif use == "destination":
            exists = df[dcol].notna() if dcol in df.columns else pd.Series(False, index=df.index)
        else:
            o_exists = df[ocol].notna() if ocol in df.columns else pd.Series(False, index=df.index)
            d_exists = df[dcol].notna() if dcol in df.columns else pd.Series(False, index=df.index)
            exists = o_exists & d_exists

        real_last = np.where(exists.to_numpy(), k, real_last)

    return pd.Series(real_last, index=df.index, name="REAL_TRIP_MAX")


def check_tripmax_consistency(
    df: pd.DataFrame,
    *,
    max_k: int | None = None,
    use: str = "both",
    verbose: bool = True,
) -> tuple[pd.DataFrame, dict]:
    """
    Check whether declared TRIP_MAX matches the actual last non-null trip.
    """
    if max_k is None:
        max_k = _detect_max_k(df)

    real_tripmax = infer_real_last_trip(df, max_k=max_k, use=use)
    declared = pd.to_numeric(df["TRIP_MAX"], errors="coerce").fillna(0).astype(int)

    inconsistent_mask = declared != real_tripmax

    inconsistent_df = df.loc[inconsistent_mask].copy()
    inconsistent_df["REAL_TRIP_MAX"] = real_tripmax.loc[inconsistent_mask]

    summary = {
        "total_rows": int(len(df)),
        "inconsistent_rows": int(inconsistent_mask.sum()),
        "consistency_rate": float(1.0 - inconsistent_mask.mean()),
    }

    if verbose:
        print("Trip-chain consistency check")
        print("--------------------------------")
        print(f"Total rows        : {summary['total_rows']}")
        print(f"Inconsistent rows : {summary['inconsistent_rows']}")
        print(f"Consistency rate  : {summary['consistency_rate']:.6f}")

    return inconsistent_df, summary


def summarize_trip_presence_vs_purpose(
    df: pd.DataFrame,
    *,
    max_k: int | None = None,
) -> pd.DataFrame:
    """
    Summarize, for each trip k, the number of non-null trip structure fields
    versus the number of non-null TRIP_PURPOSE_k.
    """
    if max_k is None:
        max_k = _detect_max_k(df)

    rows = []
    for k in range(1, max_k + 1):
        ocol = f"ORIGIN_SUBZONE_{k}"
        dcol = f"DESTINATION_SUBZONE_{k}"
        pcol = f"TRIP_PURPOSE_{k}"

        origin_nonnull = int(df[ocol].notna().sum()) if ocol in df.columns else 0
        dest_nonnull = int(df[dcol].notna().sum()) if dcol in df.columns else 0
        purpose_nonnull = int(df[pcol].notna().sum()) if pcol in df.columns else 0

        rows.append(
            {
                "trip_k": k,
                "origin_nonnull": origin_nonnull,
                "dest_nonnull": dest_nonnull,
                "purpose_nonnull": purpose_nonnull,
                "purpose_minus_origin": purpose_nonnull - origin_nonnull,
                "purpose_minus_dest": purpose_nonnull - dest_nonnull,
            }
        )

    return pd.DataFrame(rows)


# ============================================================
# (5) Distribution building: fast indexed lookups
# ============================================================
def _xy_columns(max_k: int) -> list[str]:
    """
    Return X + Y column names.
    """
    return ["AGE", "GENDER", "INCOME"] + [f"TRIP_PURPOSE_{k}" for k in range(1, max_k + 1)]


def _build_distribution_table(
    gt: pd.DataFrame,
    key_cols: list[str],
    *,
    max_k: int,
) -> pd.DataFrame:
    """
    Build a conditional empirical distribution:
        P(X, Y1..Yk | key_cols)

    Output columns:
    - key columns
    - X,Y columns
    - count
    - prob
    """
    xy_cols = [c for c in _xy_columns(max_k) if c in gt.columns]
    use_cols = key_cols + xy_cols

    dist = (
        gt[use_cols]
        .groupby(use_cols, dropna=False, sort=False)
        .size()
        .reset_index(name="count")
    )

    if len(key_cols) > 0:
        dist["prob"] = dist["count"] / dist.groupby(key_cols, dropna=False)["count"].transform("sum")
    else:
        dist["prob"] = dist["count"] / dist["count"].sum()

    return dist


def _build_lookup_from_dist(
    dist: pd.DataFrame,
    key_cols: list[str],
    *,
    max_k: int,
) -> dict:
    """
    Build a fast lookup:
        key_tuple -> {vals: ndarray [n_candidates, n_xy], probs: ndarray [n_candidates]}
    """
    xy_cols = [c for c in _xy_columns(max_k) if c in dist.columns]

    if len(key_cols) == 0:
        vals = dist[xy_cols].to_numpy()
        probs = dist["prob"].to_numpy(dtype=float)
        probs = probs / probs.sum()
        return {(): {"vals": vals, "probs": probs, "xy_cols": xy_cols}}

    grouped = dist.groupby(key_cols, dropna=False, sort=False)
    lookup = {}

    for key, sub in grouped:
        if not isinstance(key, tuple):
            key = (key,)
        key = _norm_key_tuple(key)

        vals = sub[xy_cols].to_numpy()
        probs = sub["prob"].to_numpy(dtype=float)
        probs = probs / probs.sum()

        lookup[key] = {
            "vals": vals,
            "probs": probs,
            "xy_cols": xy_cols,
        }

    return lookup


def build_gt_distribution_lookups(
    gt_df: pd.DataFrame,
    *,
    max_k: int | None = None,
) -> dict:
    """
    Build fast lookup tables for the 4-level hierarchical conditional distributions
    and the global fallback P(X,Y).

    Returns
    -------
    {
        "lvl1": {"key_cols": [...], "lookup": {...}},
        "lvl2": {"key_cols": [...], "lookup": {...}},
        "lvl3": {"key_cols": [...], "lookup": {...}},
        "lvl4": {"key_cols": [...], "lookup": {...}},
        "global": {"key_cols": [], "lookup": {...}},
    }
    """
    if max_k is None:
        max_k = _detect_max_k(gt_df)

    gt = attach_features(gt_df, max_k=max_k)

    level_specs = {
        "lvl1": ["home_loc", "MOTIF", "TRIP_MAX", "dep_time", "arr_time"],
        "lvl2": ["home_loc", "MOTIF", "TRIP_MAX"],
        "lvl3": ["MOTIF", "TRIP_MAX"],
        "lvl4": ["TRIP_MAX"],
        "global": [],
    }

    out = {}
    for name, key_cols in level_specs.items():
        dist = _build_distribution_table(gt, key_cols, max_k=max_k)
        lookup = _build_lookup_from_dist(dist, key_cols, max_k=max_k)
        out[name] = {
            "key_cols": key_cols,
            "lookup": lookup,
        }

    return out


def _sample_xy_from_lookup(
    lookup_dict: dict,
    key_cols: list[str],
    row: pd.Series,
    *,
    rng: np.random.Generator,
) -> dict | None:
    """
    Sample one XY profile from a lookup table for one generated row.

    Returns
    -------
    {
        "xy_cols": [...],
        "sampled_vals": ndarray shape (n_xy,)
    }
    or None if no support.
    """
    key_vals = _norm_key_tuple([row[c] for c in key_cols])
    entry = lookup_dict.get(key_vals)

    if entry is None:
        return None

    probs = entry["probs"]
    idx = int(rng.choice(np.arange(len(probs)), p=probs))

    return {
        "xy_cols": entry["xy_cols"],
        "sampled_vals": entry["vals"][idx],
    }


# ============================================================
# (6) Fast distribution-based oracle imputation
# ============================================================
def impute_oracle_labels_from_distribution_fast(
    gen_df: pd.DataFrame,
    gt_df: pd.DataFrame,
    *,
    max_k: int | None = None,
    random_state: int | None = 42,
    strict_trip_existence: bool = True,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Impute X and Y by drawing from GT conditional distributions, not by directly
    copying a matched GT row.

    Hierarchical fallback:
      1) P(X,Y | home_loc, MOTIF, TRIP_MAX, dep_time, arr_time)
      2) P(X,Y | home_loc, MOTIF, TRIP_MAX)
      3) P(X,Y | MOTIF, TRIP_MAX)
      4) P(X,Y | TRIP_MAX)
      5) P(X,Y)  [global fallback]

    IMPORTANT
    ---------
    - X and Y are sampled jointly from the same conditional-distribution row.
    - TRIP_PURPOSE_k is written only up to the generated TRIP_MAX.
    - If strict_trip_existence=True, TRIP_PURPOSE_k is written only if both
      ORIGIN_SUBZONE_k and DESTINATION_SUBZONE_k exist in the generated row.
    """
    if max_k is None:
        max_k = max(_detect_max_k(gen_df), _detect_max_k(gt_df))

    rng = np.random.default_rng(random_state)

    gen = attach_features(gen_df, max_k=max_k)

    # Build fast lookup tables once
    lookups = build_gt_distribution_lookups(gt_df, max_k=max_k)

    n = len(gen)

    age = np.empty(n, dtype=np.int16)
    gender = np.empty(n, dtype=np.int8)
    income = np.empty(n, dtype=np.int16)
    purposes = np.full((n, max_k), np.nan, dtype=float)

    level_hit_counter = {"lvl1": 0, "lvl2": 0, "lvl3": 0, "lvl4": 0, "global": 0}

    for i, (_, row) in enumerate(gen.iterrows()):
        sampled = None
        used_level = None

        # Try levels in order
        for level in ["lvl1", "lvl2", "lvl3", "lvl4"]:
            sampled = _sample_xy_from_lookup(
                lookups[level]["lookup"],
                lookups[level]["key_cols"],
                row,
                rng=rng,
            )
            if sampled is not None:
                used_level = level
                break

        # Global fallback
        if sampled is None:
            sampled = _sample_xy_from_lookup(
                lookups["global"]["lookup"],
                lookups["global"]["key_cols"],
                row,
                rng=rng,
            )
            used_level = "global"

        level_hit_counter[used_level] += 1

        # Convert sampled XY back to named dict for convenience
        sampled_xy = dict(zip(sampled["xy_cols"], sampled["sampled_vals"]))

        # Copy X
        age[i] = int(sampled_xy["AGE"])
        gender[i] = int(sampled_xy["GENDER"])
        income[i] = int(sampled_xy["INCOME"])

        # Copy Y safely
        trip_max = int(row["TRIP_MAX"])

        for k in range(1, max_k + 1):
            if k > trip_max:
                continue

            if strict_trip_existence:
                ocol = f"ORIGIN_SUBZONE_{k}"
                dcol = f"DESTINATION_SUBZONE_{k}"
                if pd.isna(row.get(ocol)) or pd.isna(row.get(dcol)):
                    continue

            pcol = f"TRIP_PURPOSE_{k}"
            if pcol in sampled_xy:
                purposes[i, k - 1] = sampled_xy[pcol]

    out = gen.copy()
    out["AGE"] = age
    out["GENDER"] = gender
    out["INCOME"] = income

    for k in range(1, max_k + 1):
        out[f"TRIP_PURPOSE_{k}"] = purposes[:, k - 1]

    if verbose:
        print("\n[IMPUTATION LEVEL USAGE]")
        for k, v in level_hit_counter.items():
            print(f"{k:>6}: {v}")

    return out


# ============================================================
# (7) Example pipeline
# ============================================================
def run_oracle_imputation_example(
    *,
    path_result: str,
    path_data: str,
    ff: float = 0.1,
    cc: str = "true",   # "true" or "hts"
    random_state: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Example end-to-end pipeline:
    - load generated tours
    - extract motifs
    - load GT / HTS reference
    - impute X,Y from GT conditional distributions
    - save outputs

    Returns
    -------
    (df_tour_context_free, df_tour_ditras, df_tour_explore_return)
    """
    # --------------------------------------------------------
    # Load generated tours
    # --------------------------------------------------------
    df_tour_context_free = pd.read_csv(
        os.path.join(path_result, f"val_sim_tour_context_free_{ff}.csv")
    )
    df_tour_ditras = pd.read_csv(
        os.path.join(path_result, f"val_sim_tour_ditras_{ff}.csv")
    )
    df_tour_explore_return = pd.read_csv(
        os.path.join(path_result, f"val_sim_tour_xr_{ff}.csv")
    )

    # --------------------------------------------------------
    # Add motifs
    # --------------------------------------------------------
    df_tour_context_free = extract_motifs_from_wide(df_tour_context_free)
    df_tour_ditras = extract_motifs_from_wide(df_tour_ditras)
    df_tour_explore_return = extract_motifs_from_wide(df_tour_explore_return)

    # --------------------------------------------------------
    # Load GT / HTS reference
    # --------------------------------------------------------
    df_full = pd.read_csv(os.path.join(path_data, f"val_data_hts_tour_{ff}.csv"))
    if cc == "true":
        df_full = pd.read_csv(os.path.join(path_data, "val_data_true_tour.csv"))

    df_full = extract_motifs_from_wide(df_full)

    # --------------------------------------------------------
    # Optional diagnostics before imputation
    # --------------------------------------------------------
    print("\n[DIAGNOSTIC] Generated context-free")
    check_tripmax_consistency(df_tour_context_free, use="both", verbose=True)

    print("\n[DIAGNOSTIC] Generated DITRAS")
    check_tripmax_consistency(df_tour_ditras, use="both", verbose=True)

    print("\n[DIAGNOSTIC] Generated Explore-return")
    check_tripmax_consistency(df_tour_explore_return, use="both", verbose=True)

    # --------------------------------------------------------
    # Fast distribution-based oracle imputation
    # --------------------------------------------------------
    df_tour_context_free = impute_oracle_labels_from_distribution_fast(
        df_tour_context_free,
        df_full,
        random_state=random_state,
        strict_trip_existence=True,
        verbose=True,
    )
    df_tour_ditras = impute_oracle_labels_from_distribution_fast(
        df_tour_ditras,
        df_full,
        random_state=random_state,
        strict_trip_existence=True,
        verbose=True,
    )
    df_tour_explore_return = impute_oracle_labels_from_distribution_fast(
        df_tour_explore_return,
        df_full,
        random_state=random_state,
        strict_trip_existence=True,
        verbose=True,
    )

    # --------------------------------------------------------
    # Post-imputation diagnostics
    # --------------------------------------------------------
    print("\n[DIAGNOSTIC] Purpose vs trip presence: context-free")
    print(summarize_trip_presence_vs_purpose(df_tour_context_free))

    print("\n[DIAGNOSTIC] Purpose vs trip presence: DITRAS")
    print(summarize_trip_presence_vs_purpose(df_tour_ditras))

    print("\n[DIAGNOSTIC] Purpose vs trip presence: Explore-return")
    print(summarize_trip_presence_vs_purpose(df_tour_explore_return))

    # --------------------------------------------------------
    # Save outputs
    # --------------------------------------------------------
    df_tour_context_free.to_csv(
        os.path.join(path_result, f"val_sim_tour_context_free_{cc}_imputed_{ff}.csv"),
        index=False,
    )
    df_tour_ditras.to_csv(
        os.path.join(path_result, f"val_sim_tour_ditras_{cc}_imputed_{ff}.csv"),
        index=False,
    )
    df_tour_explore_return.to_csv(
        os.path.join(path_result, f"val_sim_tour_xr_{cc}_imputed_{ff}.csv"),
        index=False,
    )

    return df_tour_context_free, df_tour_ditras, df_tour_explore_return


# ============================================================
# (8) Minimal example usage
# ============================================================

# Direct run example
ff = 0.5
cc = "hts"   # "true" or "hts"

df_tour_context_free, df_tour_ditras, df_tour_explore_return = run_oracle_imputation_example(
    path_result=path_result,
    path_data=path_data,
    ff=ff,
    cc=cc,
    random_state=42,
)

print(df_tour_ditras.info())


[DIAGNOSTIC] Generated context-free
Trip-chain consistency check
--------------------------------
Total rows        : 50000
Inconsistent rows : 0
Consistency rate  : 1.000000

[DIAGNOSTIC] Generated DITRAS
Trip-chain consistency check
--------------------------------
Total rows        : 49444
Inconsistent rows : 0
Consistency rate  : 1.000000

[DIAGNOSTIC] Generated Explore-return
Trip-chain consistency check
--------------------------------
Total rows        : 49736
Inconsistent rows : 0
Consistency rate  : 1.000000

[IMPUTATION LEVEL USAGE]
  lvl1: 9334
  lvl2: 28898
  lvl3: 11377
  lvl4: 391
global: 0

[IMPUTATION LEVEL USAGE]
  lvl1: 24173
  lvl2: 19162
  lvl3: 5840
  lvl4: 191
global: 78

[IMPUTATION LEVEL USAGE]
  lvl1: 28876
  lvl2: 15266
  lvl3: 5325
  lvl4: 191
global: 78

[DIAGNOSTIC] Purpose vs trip presence: context-free
   trip_k  origin_nonnull  dest_nonnull  purpose_nonnull  \
0       1           50000         50000            50000   
1       2           50000         

In [ ]:
import os
import re
import numpy as np
import pandas as pd


# ============================================================
# (1) Utilities for motif extraction
# ============================================================
def _detect_max_k(wide_df: pd.DataFrame) -> int:
    """
    Detect the maximum trip index k from columns such as DESTINATION_SUBZONE_7.
    """
    pat = re.compile(
        r"^(ORIGIN_SUBZONE|DESTINATION_SUBZONE|TRIP_PURPOSE|TRIP_STARTTIME|TRIP_ENDTIME)_(\d+)$"
    )
    ks = [int(m.group(2)) for c in wide_df.columns if (m := pat.match(c))]
    return max(ks) if ks else 0


def _to_str_or_none(x):
    """
    Convert a value to a stripped string, or return None if missing or empty.
    """
    if pd.isna(x):
        return None
    s = str(x).strip()
    return s if s != "" else None


def extract_location_sequence_from_wide(
    row: pd.Series,
    *,
    max_k: int,
    origin_prefix: str = "ORIGIN_SUBZONE_",
    dest_prefix: str = "DESTINATION_SUBZONE_",
) -> list[str]:
    """
    Extract the visited location sequence from one wide-format row.

    Example
    -------
    [origin_1, dest_1, dest_2, ..., dest_m]
    """
    o1 = _to_str_or_none(row.get(f"{origin_prefix}1"))
    if o1 is None:
        return []

    seq = [o1]

    for k in range(1, max_k + 1):
        dk = _to_str_or_none(row.get(f"{dest_prefix}{k}"))
        if dk is None:
            break
        seq.append(dk)

    return seq


def sequence_to_motif(seq: list[str]) -> str:
    """
    Convert a location sequence to a motif string.

    Example
    -------
    [A, B, A] -> "010"
    [A, B, C, A] -> "0120"
    """
    if not seq:
        return ""

    mapping: dict[str, int] = {}
    next_code = 0
    codes = []

    for loc in seq:
        if loc not in mapping:
            mapping[loc] = next_code
            next_code += 1
        codes.append(str(mapping[loc]))

    return "".join(codes)


def extract_motifs_from_wide(
    wide_df: pd.DataFrame,
    *,
    max_k: int | None = None,
    origin_prefix: str = "ORIGIN_SUBZONE_",
    dest_prefix: str = "DESTINATION_SUBZONE_",
    motif_col: str = "MOTIF",
) -> pd.DataFrame:
    """
    Keep all original columns and add a motif column.
    """
    if max_k is None:
        max_k = _detect_max_k(wide_df)
    if max_k == 0:
        raise ValueError("Cannot detect trip-indexed columns.")

    motifs = []

    for _, row in wide_df.iterrows():
        seq = extract_location_sequence_from_wide(
            row,
            max_k=max_k,
            origin_prefix=origin_prefix,
            dest_prefix=dest_prefix,
        )
        motifs.append(sequence_to_motif(seq))

    out = wide_df.copy()
    out[motif_col] = motifs
    return out


# ============================================================
# (2) Home features for matching
# ============================================================
def compute_home_features(df: pd.DataFrame, max_k: int | None = None) -> pd.DataFrame:
    """
    Compute matching features from wide trip-chain dataframe.

    Features:
    - home_loc = ORIGIN_SUBZONE_1 + "_" + DESTINATION_SUBZONE_{TRIP_MAX}
    - dep_time = TRIP_STARTTIME_1
    - arr_time = TRIP_ENDTIME_{TRIP_MAX}
    - TRIP_MAX
    - MOTIF
    """
    if max_k is None:
        max_k = _detect_max_k(df)

    if "MOTIF" not in df.columns:
        raise ValueError("Column 'MOTIF' is required before computing home features.")

    home_origin = df["ORIGIN_SUBZONE_1"]

    last_dest = df.apply(
        lambda r: r.get(f"DESTINATION_SUBZONE_{int(r.TRIP_MAX)}"),
        axis=1
    )

    arr_time = df.apply(
        lambda r: r.get(f"TRIP_ENDTIME_{int(r.TRIP_MAX)}"),
        axis=1
    )

    home_loc = home_origin.astype(str) + "_" + last_dest.astype(str)
    dep_time = df["TRIP_STARTTIME_1"]

    return pd.DataFrame(
        {
            "home_loc": home_loc,
            "dep_time": dep_time,
            "arr_time": arr_time,
            "TRIP_MAX": df["TRIP_MAX"],
            "MOTIF": df["MOTIF"],
        },
        index=df.index,
    )


def attach_features(df: pd.DataFrame, max_k: int | None = None) -> pd.DataFrame:
    """
    Return a copy of df with matching features appended.
    """
    out = df.copy()
    feat = compute_home_features(out, max_k=max_k)
    for c in feat.columns:
        out[c] = feat[c]
    return out


# ============================================================
# (3) Diagnostics for TRIP_MAX consistency
# ============================================================
def infer_real_last_trip(
    df: pd.DataFrame,
    *,
    max_k: int | None = None,
    use: str = "origin",
) -> pd.Series:
    """
    Infer the actual last existing trip index from non-null trip columns.

    Parameters
    ----------
    use : {"origin", "destination", "both"}
        Which columns to use to decide whether trip k exists.
    """
    if max_k is None:
        max_k = _detect_max_k(df)

    if use not in {"origin", "destination", "both"}:
        raise ValueError("use must be one of {'origin', 'destination', 'both'}.")

    real_last = np.zeros(len(df), dtype=int)

    for k in range(1, max_k + 1):
        ocol = f"ORIGIN_SUBZONE_{k}"
        dcol = f"DESTINATION_SUBZONE_{k}"

        if use == "origin":
            exists = df[ocol].notna() if ocol in df.columns else pd.Series(False, index=df.index)
        elif use == "destination":
            exists = df[dcol].notna() if dcol in df.columns else pd.Series(False, index=df.index)
        else:
            o_exists = df[ocol].notna() if ocol in df.columns else pd.Series(False, index=df.index)
            d_exists = df[dcol].notna() if dcol in df.columns else pd.Series(False, index=df.index)
            exists = o_exists & d_exists

        real_last = np.where(exists.to_numpy(), k, real_last)

    return pd.Series(real_last, index=df.index, name="REAL_TRIP_MAX")


def check_tripmax_consistency(
    df: pd.DataFrame,
    *,
    max_k: int | None = None,
    use: str = "both",
    verbose: bool = True,
) -> tuple[pd.DataFrame, dict]:
    """
    Check whether declared TRIP_MAX matches the actual last non-null trip.

    Returns
    -------
    inconsistent_df : pd.DataFrame
        Rows where TRIP_MAX != inferred REAL_TRIP_MAX
    summary : dict
        Diagnostic summary
    """
    if max_k is None:
        max_k = _detect_max_k(df)

    real_tripmax = infer_real_last_trip(df, max_k=max_k, use=use)
    declared = pd.to_numeric(df["TRIP_MAX"], errors="coerce").fillna(0).astype(int)

    inconsistent_mask = declared != real_tripmax

    inconsistent_df = df.loc[inconsistent_mask].copy()
    inconsistent_df["REAL_TRIP_MAX"] = real_tripmax.loc[inconsistent_mask]

    summary = {
        "total_rows": int(len(df)),
        "inconsistent_rows": int(inconsistent_mask.sum()),
        "consistency_rate": float(1.0 - inconsistent_mask.mean()),
    }

    if verbose:
        print("Trip-chain consistency check")
        print("--------------------------------")
        print(f"Total rows        : {summary['total_rows']}")
        print(f"Inconsistent rows : {summary['inconsistent_rows']}")
        print(f"Consistency rate  : {summary['consistency_rate']:.6f}")

    return inconsistent_df, summary


def summarize_trip_presence_vs_purpose(
    df: pd.DataFrame,
    *,
    max_k: int | None = None,
) -> pd.DataFrame:
    """
    Summarize, for each trip k, the number of non-null trip structure fields
    versus the number of non-null TRIP_PURPOSE_k.

    Useful for diagnosing purpose-count mismatches.
    """
    if max_k is None:
        max_k = _detect_max_k(df)

    rows = []
    for k in range(1, max_k + 1):
        ocol = f"ORIGIN_SUBZONE_{k}"
        dcol = f"DESTINATION_SUBZONE_{k}"
        pcol = f"TRIP_PURPOSE_{k}"

        origin_nonnull = int(df[ocol].notna().sum()) if ocol in df.columns else 0
        dest_nonnull = int(df[dcol].notna().sum()) if dcol in df.columns else 0
        purpose_nonnull = int(df[pcol].notna().sum()) if pcol in df.columns else 0

        rows.append(
            {
                "trip_k": k,
                "origin_nonnull": origin_nonnull,
                "dest_nonnull": dest_nonnull,
                "purpose_nonnull": purpose_nonnull,
                "purpose_minus_origin": purpose_nonnull - origin_nonnull,
                "purpose_minus_dest": purpose_nonnull - dest_nonnull,
            }
        )

    return pd.DataFrame(rows)


# ============================================================
# (4) Oracle matching with safe purpose copying
# ============================================================
def match_oracle_labels(
    gen_df: pd.DataFrame,
    gt_df: pd.DataFrame,
    *,
    max_k: int | None = None,
    random_state: int | None = 42,
    strict_trip_existence: bool = True,
) -> pd.DataFrame:
    """
    Match generated chains to ground-truth chains and copy X, Y labels.

    Matching hierarchy:
      1) home_loc + MOTIF + TRIP_MAX + dep_time + arr_time
      2) home_loc + MOTIF + TRIP_MAX
      3) MOTIF + TRIP_MAX
      4) TRIP_MAX
      5) random fallback

    IMPORTANT:
    - AGE, GENDER, INCOME are copied from the matched GT row.
    - TRIP_PURPOSE_k is copied ONLY if trip k exists in the generated row.
      This avoids having non-null purposes for non-existent generated trips.
    """
    if max_k is None:
        max_k = max(_detect_max_k(gen_df), _detect_max_k(gt_df))

    rng = np.random.default_rng(random_state)

    # Add matching features on copies
    gen = attach_features(gen_df, max_k=max_k)
    gt = attach_features(gt_df, max_k=max_k)

    # Build hierarchical indices
    def build_index(df: pd.DataFrame, cols: list[str]) -> dict:
        return df.groupby(cols, sort=False).indices

    idx_lvl1 = build_index(gt, ["home_loc", "MOTIF", "TRIP_MAX", "dep_time", "arr_time"])
    idx_lvl2 = build_index(gt, ["home_loc", "MOTIF", "TRIP_MAX"])
    idx_lvl3 = build_index(gt, ["MOTIF", "TRIP_MAX"])
    idx_lvl4 = build_index(gt, ["TRIP_MAX"])

    # Prepare outputs
    n = len(gen)
    age = np.empty(n, dtype=np.int16)
    gender = np.empty(n, dtype=np.int8)
    income = np.empty(n, dtype=np.int16)

    purposes = np.full((n, max_k), np.nan, dtype=float)

    gt_idx_array = np.arange(len(gt))

    for i, (_, row) in enumerate(gen.iterrows()):
        trip_max = int(row["TRIP_MAX"])

        key1 = (row["home_loc"], row["MOTIF"], row["TRIP_MAX"], row["dep_time"], row["arr_time"])
        key2 = (row["home_loc"], row["MOTIF"], row["TRIP_MAX"])
        key3 = (row["MOTIF"], row["TRIP_MAX"])
        key4 = (row["TRIP_MAX"],)

        if key1 in idx_lvl1:
            candidates = idx_lvl1[key1]
        elif key2 in idx_lvl2:
            candidates = idx_lvl2[key2]
        elif key3 in idx_lvl3:
            candidates = idx_lvl3[key3]
        elif key4 in idx_lvl4:
            candidates = idx_lvl4[key4]
        else:
            candidates = gt_idx_array

        j = int(rng.choice(candidates))
        gt_row = gt.iloc[j]

        # Copy X
        age[i] = gt_row["AGE"]
        gender[i] = gt_row["GENDER"]
        income[i] = gt_row["INCOME"]

        # Copy Y safely
        for k in range(1, max_k + 1):
            # First condition: do not fill beyond generated TRIP_MAX
            if k > trip_max:
                continue

            # Optional stronger condition: only fill if generated trip structure exists
            if strict_trip_existence:
                ocol = f"ORIGIN_SUBZONE_{k}"
                dcol = f"DESTINATION_SUBZONE_{k}"
                if pd.isna(row.get(ocol)) or pd.isna(row.get(dcol)):
                    continue

            pcol = f"TRIP_PURPOSE_{k}"
            if pcol in gt.columns:
                purposes[i, k - 1] = gt_row[pcol]

    out = gen.copy()
    out["AGE"] = age
    out["GENDER"] = gender
    out["INCOME"] = income

    for k in range(1, max_k + 1):
        out[f"TRIP_PURPOSE_{k}"] = purposes[:, k - 1]

    return out


# ============================================================
# (5) Example pipeline
# ============================================================
def run_oracle_imputation_example(
    *,
    path_result: str,
    path_data: str,
    ff: float = 0.1,
    cc: str = "true",   # "true" or "hts"
    random_state: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Example end-to-end pipeline:
    - load generated tours
    - extract motifs
    - load GT / HTS reference
    - match oracle labels
    - save outputs

    Returns
    -------
    (df_tour_context_free, df_tour_ditras, df_tour_explore_return)
    """
    # --------------------------------------------------------
    # Load generated tours
    # --------------------------------------------------------
    df_tour_context_free = pd.read_csv(
        os.path.join(path_result, f"val_sim_tour_context_free_{ff}.csv")
    )
    df_tour_ditras = pd.read_csv(
        os.path.join(path_result, f"val_sim_tour_ditras_{ff}.csv")
    )
    df_tour_explore_return = pd.read_csv(
        os.path.join(path_result, f"val_sim_tour_xr_{ff}.csv")
    )

    # --------------------------------------------------------
    # Add motifs
    # --------------------------------------------------------
    df_tour_context_free = extract_motifs_from_wide(df_tour_context_free)
    df_tour_ditras = extract_motifs_from_wide(df_tour_ditras)
    df_tour_explore_return = extract_motifs_from_wide(df_tour_explore_return)

    # --------------------------------------------------------
    # Load GT / HTS reference
    # --------------------------------------------------------
    df_full = pd.read_csv(os.path.join(path_data, f"val_data_hts_tour_{ff}.csv"))
    if cc == "true":
        df_full = pd.read_csv(os.path.join(path_data, "val_data_true_tour.csv"))

    df_full = extract_motifs_from_wide(df_full)

    # --------------------------------------------------------
    # Optional diagnostics before matching
    # --------------------------------------------------------
    print("\n[DIAGNOSTIC] Generated context-free")
    check_tripmax_consistency(df_tour_context_free, use="both", verbose=True)

    print("\n[DIAGNOSTIC] Generated DITRAS")
    check_tripmax_consistency(df_tour_ditras, use="both", verbose=True)

    print("\n[DIAGNOSTIC] Generated Explore-return")
    check_tripmax_consistency(df_tour_explore_return, use="both", verbose=True)

    # --------------------------------------------------------
    # Oracle matching
    # --------------------------------------------------------
    df_tour_context_free = match_oracle_labels(
        df_tour_context_free,
        df_full,
        random_state=random_state,
        strict_trip_existence=True,
    )
    df_tour_ditras = match_oracle_labels(
        df_tour_ditras,
        df_full,
        random_state=random_state,
        strict_trip_existence=True,
    )
    df_tour_explore_return = match_oracle_labels(
        df_tour_explore_return,
        df_full,
        random_state=random_state,
        strict_trip_existence=True,
    )

    # --------------------------------------------------------
    # Post-imputation diagnostics
    # --------------------------------------------------------
    print("\n[DIAGNOSTIC] Purpose vs trip presence: context-free")
    print(summarize_trip_presence_vs_purpose(df_tour_context_free))

    print("\n[DIAGNOSTIC] Purpose vs trip presence: DITRAS")
    print(summarize_trip_presence_vs_purpose(df_tour_ditras))

    print("\n[DIAGNOSTIC] Purpose vs trip presence: Explore-return")
    print(summarize_trip_presence_vs_purpose(df_tour_explore_return))

    # --------------------------------------------------------
    # Save outputs
    # --------------------------------------------------------
    df_tour_context_free.to_csv(
        os.path.join(path_result, f"val_sim_tour_context_free_{cc}_imputed_{ff}.csv"),
        index=False,
    )
    df_tour_ditras.to_csv(
        os.path.join(path_result, f"val_sim_tour_ditras_{cc}_imputed_{ff}.csv"),
        index=False,
    )
    df_tour_explore_return.to_csv(
        os.path.join(path_result, f"val_sim_tour_xr_{cc}_imputed_{ff}.csv"),
        index=False,
    )

    return df_tour_context_free, df_tour_ditras, df_tour_explore_return


# ============================================================
# (6) Minimal example usage
# ============================================================
# Example:
#
ff = 0.1
cc = "hts"   # "true" or "hts"

df_tour_context_free, df_tour_ditras, df_tour_explore_return = run_oracle_imputation_example(
    path_result=path_result,
    path_data=path_data,
    ff=ff,
    cc=cc,
    random_state=42,
)

print(df_tour_ditras.info())

#bad_rows, stats = check_tripmax_consistency(df_tour_ditras, use="both")
#print(bad_rows.head())

#print(summarize_trip_presence_vs_purpose(df_tour_ditras))


[DIAGNOSTIC] Generated context-free
Trip-chain consistency check
--------------------------------
Total rows        : 50000
Inconsistent rows : 0
Consistency rate  : 1.000000

[DIAGNOSTIC] Generated DITRAS
Trip-chain consistency check
--------------------------------
Total rows        : 49385
Inconsistent rows : 0
Consistency rate  : 1.000000

[DIAGNOSTIC] Generated Explore-return
Trip-chain consistency check
--------------------------------
Total rows        : 49676
Inconsistent rows : 0
Consistency rate  : 1.000000

[DIAGNOSTIC] Purpose vs trip presence: context-free
   trip_k  origin_nonnull  dest_nonnull  purpose_nonnull  \
0       1           50000         50000            50000   
1       2           50000         50000            50000   
2       3           15185         15185            14411   
3       4            7917          7917             6976   
4       5            3439          3439             2403   
5       6            1395          1395              753   
6  

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from scipy.spatial.distance import jensenshannon
from sklearn.metrics import r2_score, mean_squared_error


def _detect_max_k(wide_df: pd.DataFrame) -> int:
    """
    Detect the maximum trip index k from columns such as DESTINATION_SUBZONE_7.
    """
    pat = re.compile(r"^(ORIGIN_SUBZONE|DESTINATION_SUBZONE|TRIP_PURPOSE|TRIP_STARTTIME|TRIP_ENDTIME)_(\d+)$")
    ks = [int(m.group(2)) for c in wide_df.columns if (m := pat.match(c))]
    return max(ks) if ks else 0


def _to_str_or_none(x):
    """
    Convert a value to a stripped string, or return None if missing or empty.
    """
    if pd.isna(x):
        return None
    s = str(x).strip()
    return s if s != "" else None


def extract_location_sequence_from_wide(
    row: pd.Series,
    *,
    max_k: int,
    origin_prefix: str = "ORIGIN_SUBZONE_",
    dest_prefix: str = "DESTINATION_SUBZONE_",
) -> list[str]:
    """
    Extract the visited location sequence from one wide-format row.

    Example
    -------
    [origin_1, dest_1, dest_2, ..., dest_m]
    """
    o1 = _to_str_or_none(row.get(f"{origin_prefix}1"))
    if o1 is None:
        return []

    seq = [o1]

    for k in range(1, max_k + 1):
        dk = _to_str_or_none(row.get(f"{dest_prefix}{k}"))
        if dk is None:
            break
        seq.append(dk)

    return seq


def sequence_to_motif(seq: list[str]) -> str:
    """
    Convert a location sequence to a motif string.

    Example
    -------
    [A, B, A] -> "010"
    [A, B, C, A] -> "0120"
    """
    if not seq:
        return ""

    mapping: dict[str, int] = {}
    next_code = 0
    codes = []

    for loc in seq:
        if loc not in mapping:
            mapping[loc] = next_code
            next_code += 1
        codes.append(str(mapping[loc]))

    return "".join(codes)


def extract_motifs_from_wide(
    wide_df: pd.DataFrame,
    *,
    max_k: int | None = None,
    origin_prefix: str = "ORIGIN_SUBZONE_",
    dest_prefix: str = "DESTINATION_SUBZONE_",
    motif_col: str = "MOTIF",
) -> pd.DataFrame:
    """
    Keep all original columns and add a motif column.

    Parameters
    ----------
    wide_df : pd.DataFrame
        Input wide-format tour table.

    max_k : int or None
        Maximum trip index. If None, detect automatically.

    origin_prefix : str
        Prefix for origin columns.

    dest_prefix : str
        Prefix for destination columns.

    motif_col : str
        Name of the motif column to add.

    Returns
    -------
    pd.DataFrame
        Copy of input dataframe with an added motif column.
    """
    if max_k is None:
        max_k = _detect_max_k(wide_df)
    if max_k == 0:
        raise ValueError("Cannot detect trip-indexed columns.")

    motifs = []

    for _, row in wide_df.iterrows():
        seq = extract_location_sequence_from_wide(
            row,
            max_k=max_k,
            origin_prefix=origin_prefix,
            dest_prefix=dest_prefix,
        )
        motifs.append(sequence_to_motif(seq))

    out = wide_df.copy()
    out[motif_col] = motifs
    return out


import pandas as pd
import numpy as np


def compute_home_features(df):
    """
    Compute matching features from wide trip-chain dataframe.
    """

    # home location
    home_origin = df["ORIGIN_SUBZONE_1"]

    # last destination depends on TRIP_MAX
    last_dest = df.apply(
        lambda r: r[f"DESTINATION_SUBZONE_{int(r.TRIP_MAX)}"], axis=1
    )

    home_loc = home_origin.astype(str) + "_" + last_dest.astype(str)

    dep_time = df["TRIP_STARTTIME_1"]

    arr_time = df.apply(
        lambda r: r[f"TRIP_ENDTIME_{int(r.TRIP_MAX)}"], axis=1
    )

    return pd.DataFrame({
        "home_loc": home_loc,
        "dep_time": dep_time,
        "arr_time": arr_time,
        "TRIP_MAX": df["TRIP_MAX"],
        "MOTIF": df["MOTIF"]
    })


def attach_features(df):
    """
    Append matching features to dataframe.
    """
    feat = compute_home_features(df)
    for c in feat.columns:
        df[c] = feat[c]
    return df


def match_oracle_labels(gen_df, gt_df):
    """
    Match generated chains to ground-truth chains and copy X,Y labels.
    """

    # attach matching features
    gen_df = attach_features(gen_df)
    gt_df = attach_features(gt_df)

    # build hierarchical index dictionaries
    def build_index(df, cols):
        groups = df.groupby(cols).indices
        return groups

    idx_lvl1 = build_index(gt_df, ["home_loc","MOTIF","TRIP_MAX","dep_time","arr_time"])
    idx_lvl2 = build_index(gt_df, ["home_loc","MOTIF","TRIP_MAX"])
    idx_lvl3 = build_index(gt_df, ["MOTIF","TRIP_MAX"])
    idx_lvl4 = build_index(gt_df, ["TRIP_MAX"])

    # output arrays
    AGE = np.empty(len(gen_df), dtype=np.int16)
    GENDER = np.empty(len(gen_df), dtype=np.int8)
    INCOME = np.empty(len(gen_df), dtype=np.int16)

    trip_purpose_cols = [f"TRIP_PURPOSE_{i}" for i in range(1,8)]
    purposes = np.full((len(gen_df),7), np.nan)

    for i, row in gen_df.iterrows():

        key1 = (row.home_loc, row.MOTIF, row.TRIP_MAX, row.dep_time, row.arr_time)
        key2 = (row.home_loc, row.MOTIF, row.TRIP_MAX)
        key3 = (row.MOTIF, row.TRIP_MAX)
        key4 = (row.TRIP_MAX,)

        if key1 in idx_lvl1:
            candidates = idx_lvl1[key1]
        elif key2 in idx_lvl2:
            candidates = idx_lvl2[key2]
        elif key3 in idx_lvl3:
            candidates = idx_lvl3[key3]
        elif key4 in idx_lvl4:
            candidates = idx_lvl4[key4]
        else:
            candidates = np.arange(len(gt_df))

        j = np.random.choice(candidates)

        AGE[i] = gt_df.iloc[j]["AGE"]
        GENDER[i] = gt_df.iloc[j]["GENDER"]
        INCOME[i] = gt_df.iloc[j]["INCOME"]

        trip_max = int(row.TRIP_MAX)

        for k in range(trip_max):
            col = f"TRIP_PURPOSE_{k+1}"
            if col in gt_df.columns:
                purposes[i, k] = gt_df.iloc[j][col]

    gen_df["AGE"] = AGE
    gen_df["GENDER"] = GENDER
    gen_df["INCOME"] = INCOME

    for k in range(7):
        gen_df[f"TRIP_PURPOSE_{k+1}"] = purposes[:,k]

    return gen_df

# ============================================================
# Example usage
# ============================================================
ff = 0.1
cc = 'hts' # 'true' or 'hts'

df_tour_context_free = pd.read_csv(os.path.join(path_result, f"val_sim_tour_context_free_{ff}.csv"))
df_tour_ditras = pd.read_csv(os.path.join(path_result, f"val_sim_tour_ditras_{ff}.csv"))
df_tour_explore_return = pd.read_csv(os.path.join(path_result, f"val_sim_tour_xr_{ff}.csv"))


df_tour_context_free = extract_motifs_from_wide(df_tour_context_free)
df_tour_ditras = extract_motifs_from_wide(df_tour_ditras)
df_tour_explore_return = extract_motifs_from_wide(df_tour_explore_return)

df_full = pd.read_csv(path_data + f'val_data_hts_tour_{ff}.csv')
if cc == 'true':
    df_full = pd.read_csv(os.path.join(path_data, "val_data_true_tour.csv"))

df_full = extract_motifs_from_wide(df_full)

df_tour_context_free = match_oracle_labels(df_tour_context_free, df_full)
df_tour_ditras = match_oracle_labels(df_tour_ditras, df_full)
df_tour_explore_return = match_oracle_labels(df_tour_explore_return, df_full)

df_tour_context_free.to_csv(path_result + f'val_sim_tour_context_free_{cc}_imputed_{ff}.csv', index=False)
df_tour_ditras.to_csv(path_result + f'val_sim_tour_ditras_{cc}_imputed_{ff}.csv', index=False)
df_tour_explore_return.to_csv(path_result + f'val_sim_tour_xr_{cc}_imputed_{ff}.csv', index=False)

print(df_tour_ditras.info())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49385 entries, 0 to 49384
Data columns (total 73 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID                       49385 non-null  object 
 1   TRIP_MAX                 49385 non-null  int64  
 2   TOUR_CODE                49385 non-null  int64  
 3   ORIGIN_SUBZONE_1         49385 non-null  object 
 4   ORIGIN_SUBZONE_X_1       49385 non-null  float64
 5   ORIGIN_SUBZONE_Y_1       49385 non-null  float64
 6   DESTINATION_SUBZONE_1    49385 non-null  object 
 7   DESTINATION_SUBZONE_X_1  49385 non-null  float64
 8   DESTINATION_SUBZONE_Y_1  49385 non-null  float64
 9   TRIP_STARTTIME_1         49385 non-null  float64
 10  TRIP_ENDTIME_1           49385 non-null  float64
 11  ORIGIN_SUBZONE_2         49222 non-null  object 
 12  ORIGIN_SUBZONE_X_2       49222 non-null  float64
 13  ORIGIN_SUBZONE_Y_2       49222 non-null  float64
 14  DESTINATION_SUBZONE_2 

In [ ]:
import pandas as pd
import numpy as np


def check_tripmax_consistency(df, max_k=7, verbose=True):
    """
    Check whether TRIP_MAX matches the actual last non-null trip column.

    Parameters
    ----------
    df : pandas.DataFrame
        Wide-format trip chain dataframe.

    max_k : int
        Maximum number of trip columns.

    verbose : bool
        If True, print summary statistics.

    Returns
    -------
    inconsistent_df : pandas.DataFrame
        Subset of rows where TRIP_MAX is inconsistent with real trips.

    summary : dict
        Diagnostic statistics.
    """

    origin_cols = [f"ORIGIN_SUBZONE_{k}" for k in range(1, max_k + 1)]

    # Boolean matrix: whether each trip column exists
    trip_exists = df[origin_cols].notna().values

    # Compute real last trip index
    real_last_trip = trip_exists[:, ::-1].argmax(axis=1)
    real_last_trip = max_k - real_last_trip

    # If no trip exists, set to 0
    no_trip_mask = ~trip_exists.any(axis=1)
    real_last_trip[no_trip_mask] = 0

    declared_tripmax = df["TRIP_MAX"].values

    inconsistent_mask = declared_tripmax != real_last_trip

    inconsistent_df = df.loc[inconsistent_mask].copy()

    summary = {
        "total_rows": len(df),
        "inconsistent_rows": int(inconsistent_mask.sum()),
        "consistency_rate": float(1 - inconsistent_mask.mean()),
    }

    if verbose:
        print("Trip-chain consistency check")
        print("--------------------------------")
        print(f"Total rows: {summary['total_rows']}")
        print(f"Inconsistent rows: {summary['inconsistent_rows']}")
        print(f"Consistency rate: {summary['consistency_rate']:.4f}")

    return inconsistent_df, summary

df_full = pd.read_csv(os.path.join(path_result, f"val_sim_tour_ditras_{ff}.csv"))

bad_rows, stats = check_tripmax_consistency(df_full)

print(bad_rows.head())
print(df_full.info())

Trip-chain consistency check
--------------------------------
Total rows: 49385
Inconsistent rows: 0
Consistency rate: 1.0000
Empty DataFrame
Columns: [ID, TRIP_MAX, TOUR_CODE, ORIGIN_SUBZONE_1, ORIGIN_SUBZONE_X_1, ORIGIN_SUBZONE_Y_1, DESTINATION_SUBZONE_1, DESTINATION_SUBZONE_X_1, DESTINATION_SUBZONE_Y_1, TRIP_STARTTIME_1, TRIP_ENDTIME_1, ORIGIN_SUBZONE_2, ORIGIN_SUBZONE_X_2, ORIGIN_SUBZONE_Y_2, DESTINATION_SUBZONE_2, DESTINATION_SUBZONE_X_2, DESTINATION_SUBZONE_Y_2, TRIP_STARTTIME_2, TRIP_ENDTIME_2, ORIGIN_SUBZONE_3, ORIGIN_SUBZONE_X_3, ORIGIN_SUBZONE_Y_3, DESTINATION_SUBZONE_3, DESTINATION_SUBZONE_X_3, DESTINATION_SUBZONE_Y_3, TRIP_STARTTIME_3, TRIP_ENDTIME_3, ORIGIN_SUBZONE_4, ORIGIN_SUBZONE_X_4, ORIGIN_SUBZONE_Y_4, DESTINATION_SUBZONE_4, DESTINATION_SUBZONE_X_4, DESTINATION_SUBZONE_Y_4, TRIP_STARTTIME_4, TRIP_ENDTIME_4, ORIGIN_SUBZONE_5, ORIGIN_SUBZONE_X_5, ORIGIN_SUBZONE_Y_5, DESTINATION_SUBZONE_5, DESTINATION_SUBZONE_X_5, DESTINATION_SUBZONE_Y_5, TRIP_STARTTIME_5, TRIP_ENDTIME_5

#=====================================